# Transformer Decoder

## 1. Decoder란 무엇인가

Transformer에서 **decoder**는 지금까지 주어진 토큰들을 바탕으로 **다음 토큰을 예측하여 문장을 생성하는 모듈**이다.  
BERT를 공부했다면 encoder는 "문장을 읽고 이해하는 구조"에 가깝고, decoder는 "문장을 한 토큰씩 만들어 가는 구조"에 가깝다고 보면 된다.

가장 중요한 관점은 다음과 같다.

- **encoder**: 입력 전체를 보고 의미를 파악
- **decoder**: 앞에 나온 것만 보고 다음 내용을 생성

decoder는 생성형 언어모델의 핵심.

## 2. 왜 decoder가 중요한가

GPT 계열, LLaMA 계열, 대부분의 현대적인 생성형 LLM은 기본적으로 **decoder-only Transformer**를 사용.  
그 이유는 decoder 구조가 다음 작업에 매우 잘 맞기 때문.

- 문장 생성
- 대화 응답 생성
- 요약문 생성
- 번역문 생성
- 코드 생성

이 모든 작업은 결국 다음 문제로 볼 수 있다.

> 지금까지 나온 토큰들을 보고, 다음 토큰을 예측.

---

## 3. Decoder가 해결하는 핵심 문제

예를 들어 다음과 같은 입력이 있다고 하자.

> 나는 오늘 학교에

decoder는 이 뒤에 올 다음 토큰이 무엇일지 예측한다.

가능한 후보는 예를 들면 다음과 같다.

- 갔다
- 간다
- 가고
- 있었다

decoder는 확률적으로 가장 적절한 토큰을 선택.  
그다음에는 새로 생성된 토큰까지 포함해서 다시 다음 토큰을 예측.

예를 들어 생성 과정은 다음과 같다.

1. 입력: `나는 오늘 학교에`
2. 다음 토큰 예측: `갔다`
3. 입력: `나는 오늘 학교에 갔다`
4. 다음 토큰 예측: `.`
5. 생성 종료

decoder는 **한 번에 문장 전체를 만드는 것이 아니라, 한 토큰씩 순차적으로 생성**.

## 4. Decoder-only 모델이란

Transformer 전체 구조는 크게 다음 세 가지로 나눌 수 있다.

1. **Encoder-only**
   - 대표: BERT
   - 문장 이해 중심

2. **Encoder-Decoder**
   - 대표: T5, BART
   - 입력을 읽고 새로운 출력을 생성

3. **Decoder-only**
   - 대표: GPT, LLaMA
   - 이전 토큰들만 보고 다음 토큰 생성

이 중 GPT 계열은 **decoder-only** 구조를 사용.  

encoder 없이 decoder만 여러 층 쌓아서 모델을 만든다.

## 5. Decoder의 입력과 출력

decoder의 입력은 보통 다음과 같다.

- 현재까지의 토큰 시퀀스
- 각 토큰의 위치 정보
- 학습 과정에서는 정답 문장을 한 칸씩 밀어 넣은 형태

예를 들어 문장이 다음과 같다고 하자.

> 나는 밥을 먹었다

학습할 때는 보통 다음처럼 구성.

- 입력: `<BOS> 나는 밥을` , **Beginning Of Sequence(BOS)**
- 목표: `나는 밥을 먹었다`

좀 더 자세히 쓰면 다음과 같다.

- 입력 토큰: `<BOS>, 나는, 밥을, 먹었다`
- 목표 토큰: `나는, 밥을, 먹었다, <EOS>`

decoder는 각 위치에서 **다음 위치의 토큰**을 맞히도록 학습된다.

## 6. Decoder의 핵심 아이디어: Autoregressive 생성

decoder는 보통 **autoregressive** 방식으로 작동한다.(auto: 자기자신의 출력이, regressivce: 다음 예측의 조건이 됨.)

문장 전체의 확률은 다음과 같이 분해할 수 있다.

$$
P(x_1, x_2, \dots, x_T)
= \prod_{t=1}^{T} P(x_t \mid x_1, x_2, \dots, x_{t-1})
$$

이 식은 문장 전체의 확률을  "이전 토큰들이 주어졌을 때 다음 토큰이 나올 조건부 확률"의 곱으로 분해한 것이다.

decoder는 매 단계에서 다음을 학습한다.

$$
P(x_t \mid x_1, \dots, x_{t-1})
$$
이것이 생성형 언어모델의 수학적 핵심.

$$
x_1 -> x_2 -> x_3 -> \cdots -> x_n
$$

앞이 생성된 결과가 뒤에 들어가는 구조

$$
\prod_{t=1}^{T} P(x_t \mid x_1, x_2, \dots, x_{t-1}) = P(x_1) P(x_2\mid x_1) P(x_3\mid x_2) \cdots P(x_n\mid x_1,x_2,\cdots,x_n)  
$$

앞 토큰이 주어졌을 때, 다음 토큰을 예측

> The cat sat on the mat.

1. `The`
2. `cat` given `The`
3. `sat` given `The cat`
4. `on` given `The cat sat`
5. `the` given `The cat sat on`
6. `mat` given `The cat sat mat`
7. `.` given ``The cat sat on mat`

**decoder는 항상 왼쪽에서 오른쪽으로 진행**

## 7. Decoder가 미래 토큰을 보면 안 되는 이유

문장 생성에서는 현재 단어를 예측할 때 **미래 단어를 미리 보면 안 된다**.

예를 들어 다음 문장에서

> 오늘 날씨가 좋아서 산책을 갔다

`산책을`이라는 단어를 예측할 때 이미 뒤에 있는 `갔다`를 보고 있다면, 생성 문제의 의미가 사라짐.

실제 생성 상황에서는 미래 단어가 아직 존재하지 않기 때문.

그래서 decoder에는 반드시 다음 제약이 필요.

- 현재 위치는 **이전 위치들만 볼 수 있다**
- 현재 위치 이후의 토큰은 볼 수 없다

이 제약을 구현하는 것이 바로 **masked self-attention** 또는 **causal self-attention**이다.


## 8. Decoder의 핵심 구성 요소

Transformer decoder block은 대체로 다음 요소들로 이루어진다.

1. **입력 임베딩**
2. **위치 임베딩**
3. **masked self-attention**
4. **feed-forward network**
5. **residual connection**
6. **layer normalization**
7. 여러 block의 반복
8. 최종 linear projection과 softmax

decoder-only LLM에서는 이 블록이 여러 층 쌓임.

## 9. 입력 임베딩과 위치 정보

### 9.1 입력 임베딩

텍스트는 먼저 토큰으로 분해된다.

예:
`나는`, `오늘`, `학교에`, `갔다`

각 토큰은 정수 id로 바뀌고, 각 id는 임베딩 벡터로 변환된다.

$$
x_i \in \mathbb{R}^{d_{\text{model}}}
$$

즉 각 토큰은 $d_{\text{model}}$ 차원의 벡터로 표현된다.

### 9.2 위치 임베딩

self-attention만 사용하면 토큰의 순서를 직접 알 수 없다.  
따라서 각 토큰의 위치 정보도 함께 더해주어야 한다.

$$
h_i^{(0)} = \text{TokenEmbedding}(x_i) + \text{PositionalEncoding}(i)
$$

이렇게 하면 모델은

- 어떤 토큰이 들어왔는지
- 그것이 몇 번째 위치에 있는지

를 동시에 알 수 있다.


## 10. Decoder에서의 Masked Self-Attention

일반 self-attention은 모든 위치가 서로를 볼 수 있다.  
하지만 decoder에서는 미래 토큰을 볼 수 없어야 한다.

그래서 attention score 행렬에 **mask**를 씌운다.

예를 들어 길이 4인 시퀀스에 대해, attention 가능한 구조는 다음과 같다.

$$
\begin{bmatrix}
1 & 0 & 0 & 0 \\
1 & 1 & 0 & 0 \\
1 & 1 & 1 & 0 \\
1 & 1 & 1 & 1
\end{bmatrix}
$$

의미는 다음과 같다.

- 1번째 토큰은 자기 자신만 볼 수 있다.
- 2번째 토큰은 1, 2번째만 볼 수 있다.
- 3번째 토큰은 1, 2, 3번째만 볼 수 있다.
- 4번째 토큰은 1, 2, 3, 4번째만 볼 수 있다.

**상삼각 영역은 가려진다**. 실제로는 미래 위치에 매우 작은 값 $-\infty$에 가까운 값을 넣는다.

$$
\text{MaskedAttention}(Q,K,V)
=
\text{softmax}\left(\frac{QK^T + M}{\sqrt{d_k}}\right)V
$$

여기서 $M$은 mask 행렬이다.

- 허용된 위치: 0
- 금지된 위치: 매우 큰 음수

이렇게 하면 softmax 후 미래 위치의 확률은 거의 0이 됨.

## 11. 왜 이 mask가 생성과 정확히 맞아떨어지는가?

문장을 생성할 때는 실제로 다음과 같은 상황.

- 첫 번째 토큰을 생성할 때는 아무것도 모른다.
- 두 번째 토큰을 생성할 때는 첫 번째 토큰만 안다.
- 세 번째 토큰을 생성할 때는 앞의 두 토큰만 안다.

decoder의 causal mask는 이 상황을 학습 단계에서 그대로 반영한다.  

**훈련과 생성의 조건을 일치시키는 장치**다.

---

## 12. Multi-Head Attention

실제 decoder에서는 attention을 한 번만 하지 않고, 여러 개의 head로 나누어 계산.

$$
\text{head}_i = \text{Attention}(Q_i, K_i, V_i)
$$

그 후 이들을 이어 붙인다.

$$
\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W_O
$$

### 왜 여러 head가 필요한가

각 head가 서로 다른 관계를 볼 수 있기 때문.

예를 들어 어떤 head는

- 주어와 서술어 관계

를 볼 수 있고, 또 다른 head는

- 멀리 떨어진 단어 간의 문맥 관계

를 볼 수 있다.

**multi-head attention은 문장의 다양한 패턴을 병렬적으로 포착하게 해줌.**


## 13. Feed-Forward Network

attention 뒤에는 각 위치마다 독립적으로 적용되는 작은 신경망이 온다.

보통 다음처럼 쓴다.

$$
\text{FFN}(x) = W_2 \sigma(W_1 x + b_1) + b_2
$$

여기서 $\sigma$는 ReLU, GELU 같은 비선형 함수다.

attention이 토큰 간 상호작용을 담당한다면, FFN은 각 위치의 표현을 더 풍부하게 바꾸는 역할을 함.

다음처럼 이해할 수 있다.

- **attention**: 다른 토큰들과의 관계 반영
- **FFN**: 각 토큰 표현 자체를 비선형적으로 변환

---

## 14. Residual Connection과 LayerNorm

Transformer decoder block은 학습 안정성을 위해 다음 구조를 사용.

### 14.1 Residual Connection

입력을 출력에 더한다.

$$
y = x + \text{Sublayer}(x)
$$

이렇게 하면 깊은 네트워크에서도 정보 전달이 잘 되고 학습이 안정된다.

### 14.2 Layer Normalization

각 층의 출력을 정규화하여 학습을 안정화한다.

Transformer에서는 residual connection과 layer normalization이 매우 중요.

---

## 15. Decoder Block의 전체 흐름

decoder-only block 하나를 개념적으로 쓰면 다음과 같다.

1. 입력 임베딩 + 위치 임베딩
2. masked multi-head self-attention
3. residual connection + layer norm
4. feed-forward network
5. residual connection + layer norm

이 과정을 여러 층 반복.

간단히 쓰면 다음과 같다.

$$
H^{(l+1)} = \text{FFN}\big(\text{MaskedSelfAttention}(H^{(l)})\big)
$$

실제로는 중간에 residual과 normalization이 포함.

---

## 16. Decoder의 마지막 출력

마지막 decoder layer의 출력은 각 위치마다 하나의 hidden vector를 만든다.

이 벡터를 vocabulary 크기로 선형변환한다.

$$
z_t = h_t W_{\text{vocab}} + b
$$

그러면 각 위치에서 vocabulary 전체에 대한 점수(logit)가 나온다.

그다음 softmax를 적용한다.

$$
P(x_{t+1} \mid x_{\le t}) = \text{softmax}(z_t)
$$

**다음 토큰의 확률분포를 얻는다.**

---

## 17. 학습 시 decoder는 어떻게 배우는가

학습에서는 정답 문장을 알고 있으므로, 각 위치에서 다음 토큰 정답을 비교할 수 있다.

예를 들어 정답 문장이

> 나는 밥을 먹었다

라면 입력과 목표는 다음처럼 정렬.

| 입력 | 목표 |
|---|---|
| `<BOS>` | 나는 |
| `<BOS> 나는` | 밥을 |
| `<BOS> 나는 밥을` | 먹었다 |
| `<BOS> 나는 밥을 먹었다` | `<EOS>` |

각 위치에서 모델이 예측한 확률분포와 정답 토큰을 비교하여 **cross-entropy loss**를 계산.

$$
\mathcal{L} = - \sum_{t=1}^{T} \log P(x_t \mid x_1, \dots, x_{t-1})
$$

이 loss를 최소화하도록 학습하면, decoder는 점점 더 자연스러운 다음 토큰을 예측하게 된다.

---

## 18. 추론 시 decoder는 어떻게 작동하는가

학습과 추론은 약간 다르다.

### 학습
정답 문장이 있으므로 모든 위치를 한 번에 계산할 수 있다.

### 추론
정답을 모르므로 한 토큰씩 생성.

예를 들어:

1. 입력: `<BOS>`
2. 다음 토큰 예측: `나는`
3. 입력: `<BOS> 나는`
4. 다음 토큰 예측: `오늘`
5. 입력: `<BOS> 나는 오늘`
6. 다음 토큰 예측: `학교에`
7. ...

이 과정을 `<EOS>`가 나올 때까지 반복.

## 19. Decoder와 BERT의 구조 차이


### BERT
- encoder-only
- 양방향 문맥 사용
- 미래 토큰도 볼 수 있음
- 주로 이해 작업에 강함

### Decoder
- decoder-only
- 왼쪽 문맥만 사용
- 미래 토큰을 볼 수 없음
- 생성 작업에 강함

 BERT는

> 문장 전체를 보고 이 단어가 무슨 역할인지 이해

하는 데 강하고,


decoder는

> 지금까지 나온 토큰을 보고 다음 토큰 생성

에 강함.

---

## 20. Decoder에서 "자기 자신을 본다"는 말의 의미

self-attention에서 각 토큰은 자기 자신도 포함해서 attention을 계산.

예를 들어 3번째 위치는

- 1번째 토큰
- 2번째 토큰
- 3번째 토큰 자기 자신

을 볼 수 있다.

자기 자신을 보는 이유는 현재 위치의 정보 자체도 중요하기 때문이다.  
완전히 주변 정보만 보는 것이 아니라, 자기 표현을 중심으로 문맥을 재구성.

---

## 21. Decoder의 장점

### 21.1 생성에 자연스럽다
다음 토큰 예측이라는 목표와 구조가 정확히 일치.

### 21.2 범용성이 높다
대화, 글쓰기, 코드 생성, 요약, 번역 등을 모두 "다음 토큰 생성" 문제로 통일할 수 있다.

### 21.3 대규모 사전학습에 적합하다
웹 문서, 책, 코드 등 대량 텍스트에 대해 별도의 정답 라벨 없이 학습할 수 있다.

---

## 22. Decoder의 한계

### 22.1 순차 생성이라 느릴 수 있다
추론 시 토큰을 한 개씩 생성하므로 병렬성이 제한된다.

### 22.2 긴 문맥에서 계산량이 커진다
self-attention은 시퀀스 길이에 따라 비용이 커진다.

### 22.3 양방향 이해에는 encoder보다 불리할 수 있다
BERT처럼 문장 전체를 동시에 보는 구조가 아니기 때문에, 특정 이해 과제에서는 encoder가 더 직접적일 수 있다.

---

## 23. 왜 GPT는 decoder-only를 선택했는가

GPT 계열의 핵심 목표는 **자연스러운 텍스트 생성**.  
이를 위해서는 "앞부분을 보고 뒷부분을 생성하는 구조"가 가장 적절.

decoder-only 구조는 이 목적에 매우 잘 맞는다.

- causal mask로 미래 차단
- next token prediction으로 학습
- autoregressive generation으로 추론

구조, 학습 목표, 추론 방식이 모두 일관된다.


- 아주 작은 GPT를 만들어보자!

In [ ]:
# 라이브러리 호출
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
#device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("사용 장치:", device)

사용 장치: cuda


In [ ]:
# 아주 작은 학습 문장 준비
texts = [
    "i like math",
    "i like deep learning",
    "i like transformers",
    "you like math",
    "you like coding",
    "we like deep learning",
    "we like math",
    "transformers are powerful",
    "deep learning is fun",
    "math is fun"
]

print("문장 개수:", len(texts))
for t in texts:
    print(t)

문장 개수: 10
i like math
i like deep learning
i like transformers
you like math
you like coding
we like deep learning
we like math
transformers are powerful
deep learning is fun
math is fun


- 이 문장들은 반복 패턴이 있어서 작은 모델도 어느정도 할 수 있음.

In [ ]:
# 특수 토큰과 단어 사전 만들기
special_tokens = ["<PAD>", "<BOS>", "<EOS>"]

word_set = set()
for text in texts:
    for word in text.split():
        word_set.add(word)

vocab = special_tokens + sorted(list(word_set))

stoi = {word: i for i, word in enumerate(vocab)}
itos = {i: word for word, i in stoi.items()}

vocab_size = len(vocab)

print("vocab_size:", vocab_size)
print("vocab:", vocab)

vocab_size: 16
vocab: ['<PAD>', '<BOS>', '<EOS>', 'are', 'coding', 'deep', 'fun', 'i', 'is', 'learning', 'like', 'math', 'powerful', 'transformers', 'we', 'you']


- `<PAD>`: 길이를 맞출 때 넣는 패딩 토큰
- `<BOS>`: 문장의 시작을 나타내는 토큰
- `<EOS>`: 문장의 끝을 나타내는 토큰

또 다음 두 개의 사전을 만듭니다.

- stoi: string to index
- itos: index to string

- 예를 들어 "math"가 8번이면 모델은 "math" 대신 8을 보게 됩니다.

In [ ]:
#자주 쓰는 토큰 ID저장
PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
EOS_ID = stoi["<EOS>"]

print("PAD_ID:", PAD_ID)
print("BOS_ID:", BOS_ID)
print("EOS_ID:", EOS_ID)

PAD_ID: 0
BOS_ID: 1
EOS_ID: 2


- 코드에서 반복적으로 사용할 특수 토큰의 번호를 따로 저장-> 이렇게 해두면 코드가 더 읽기 쉬워짐.

In [ ]:
# 문장을 숫자 시퀀스로 바꾸는 함수

def encode(text):
    return [stoi[word] for word in text.split()]

def decode(token_ids):
    words = []
    for idx in token_ids:
        word = itos[idx]
        if word == "<EOS>":
            break
        if word not in ["<BOS>", "<PAD>"]:
            words.append(word)
    return " ".join(words)

- encode

    - 문장을 단어 ID 리스트로 바꿉니다.

    - 예:

    - "i like math"
    - → [3, 7, 8] 같은 형태

- decode

    - 숫자 리스트를 다시 문장으로 바꿉니다.

    - 이때 <BOS>, <PAD>는 출력에서 제외하고 <EOS>를 만나면 문장을 끝냅니다.

In [ ]:
#각 문장 앞 뒤에 BOS, EOS 붙이기
encoded_texts = []
for text in texts:
    ids = [BOS_ID] + encode(text) + [EOS_ID]
    encoded_texts.append(ids)

for text, ids in zip(texts, encoded_texts):
    print(text, "->", ids)

i like math -> [1, 7, 10, 11, 2]
i like deep learning -> [1, 7, 10, 5, 9, 2]
i like transformers -> [1, 7, 10, 13, 2]
you like math -> [1, 15, 10, 11, 2]
you like coding -> [1, 15, 10, 4, 2]
we like deep learning -> [1, 14, 10, 5, 9, 2]
we like math -> [1, 14, 10, 11, 2]
transformers are powerful -> [1, 13, 3, 12, 2]
deep learning is fun -> [1, 5, 9, 8, 6, 2]
math is fun -> [1, 11, 8, 6, 2]


- GPT는 문장을 생성할 때 시작점과 종료점을 알아야 합니다.

- 그래서 각 문장을 다음처럼 바꿉니다.

    - 예:

    - 원래 문장: i like math
    - 변환 후: `<BOS>` i like math `<EOS>`

이렇게 해야 모델이

어디서 시작해야 하는지

어디서 멈춰야 하는지를 배울 수 있음

In [ ]:
# 최대 길이 구하고 패딩 준비
max_len = max(len(ids) for ids in encoded_texts)
print("최대 길이:", max_len)

def pad_sequence(seq, max_len, pad_value=PAD_ID):
    return seq + [pad_value] * (max_len - len(seq))

padded_texts = [pad_sequence(ids, max_len) for ids in encoded_texts]

for seq in padded_texts:
    print(seq)

최대 길이: 6
[1, 7, 10, 11, 2, 0]
[1, 7, 10, 5, 9, 2]
[1, 7, 10, 13, 2, 0]
[1, 15, 10, 11, 2, 0]
[1, 15, 10, 4, 2, 0]
[1, 14, 10, 5, 9, 2]
[1, 14, 10, 11, 2, 0]
[1, 13, 3, 12, 2, 0]
[1, 5, 9, 8, 6, 2]
[1, 11, 8, 6, 2, 0]


- 문장 길이는 서로 다르기 때문에 한 번에 텐서로 만들려면 길이를 맞춰야 함

In [ ]:
# 입력 x와 정답 y 만들기

x_data = []
y_data = []

for seq in padded_texts:
    x_data.append(seq[:-1])
    y_data.append(seq[1:])

x_data = torch.tensor(x_data, dtype=torch.long).to(device)
y_data = torch.tensor(y_data, dtype=torch.long).to(device)

print("x_data shape:", x_data.shape)
print("y_data shape:", y_data.shape)

print("\n첫 번째 입력 x:")
print(x_data[0])

print("\n첫 번째 정답 y:")
print(y_data[0])

x_data shape: torch.Size([10, 5])
y_data shape: torch.Size([10, 5])

첫 번째 입력 x:
tensor([ 1,  7, 10, 11,  2], device='cuda:0')

첫 번째 정답 y:
tensor([ 7, 10, 11,  2,  0], device='cuda:0')


- GPT는 현재까지의 토큰을 보고 다음 토큰을 맞히는 모델입니다.

- 그래서 한 문장이

    - `<BOS>` i like math `<EOS>` 라면,

    - 입력 x: `<BOS>` i like math
    - 정답 y: i like math `<EOS>`

로 만듭니다.

**한 칸 밀어서 정답을 만듭니다.**

이렇게 하면 모델은 각 위치에서 항상 다음 토큰을 예측하게 됨.

In [ ]:
# Positional Embedding 정의
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        batch_size, seq_len = x.shape
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        pos_embed = self.pos_embedding(positions)
        return pos_embed

- Transformer는 순환신경망이 아니기 때문에,토큰의 순서를 자동으로 알지 못함.

    - 예를 들어

    - i like math
    - math like i

는 같은 단어를 쓰더라도 순서가 다름.

그래서 각 위치에 대해 위치 정보(position information) 를 추가해 줌.

사인/코사인 positional Encoding 대신 학습 가능한 positional Embedding을 사용함.

In [ ]:
# Causal Self-Attention 정의

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model은 num_heads로 나누어 떨어져야 합니다."

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).bool()
        scores = scores.masked_fill(~mask, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)

        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        out = self.out_proj(context)
        return out

1. Q, K, V 만들기
2. Score 계산
3. Causal mark 적용


```
mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device)).bool()
scores = scores.masked_fill(~mask, float("-inf"))
```

이 코드는 미래 토큰을 보지 못하게 막는 역할을 함.



현재 토큰은 자기 자신과 이전 토큰만 볼 수 있음

뒤에 있는 미래 토큰은 볼 수 없음

**이것이 decoder-only GPT의 핵심 차이**



In [ ]:
## Feed Forward Network 정의

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)

attention이 토큰들 사이의 관계를 반영한다면,
FFN은 각 토큰 표현을 더 복잡하게 변환해 주는 역할을 합니다.


- attention: 토큰 간 관계 학습
- FFN: 각 토큰 벡터를 더 풍부하게 변환

In [ ]:
# Decoder Block 정의

class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

In [ ]:
# Mini GPT 모델 정의
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, max_len, d_model=64, num_heads=4, d_ff=128, num_layers=2):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = PositionalEmbedding(max_len, d_model)

        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(x)
        h = tok_emb + pos_emb

        for block in self.blocks:
            h = block(h)

        h = self.ln_f(h)
        logits = self.head(h)
        return logits
# 최종 logits의 shape; (batch_size, seq_len, vocab_size)
# 각 위치마다 다음 토큰 후보들에 대한 점수를 출력

In [ ]:
# 모델 생성
model = MiniGPT(
    vocab_size=vocab_size,
    max_len=max_len - 1,
    d_model=64,   # 토큰 벡터 차원
    num_heads=4,  # attension head 개수
    d_ff=128,     # FFN 내부 차원
    num_layers=2  # decoder block수
).to(device)

print(model)

MiniGPT(
  (token_embedding): Embedding(16, 64)
  (pos_embedding): PositionalEmbedding(
    (pos_embedding): Embedding(5, 64)
  )
  (blocks): ModuleList(
    (0-1): 2 x DecoderBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (q_proj): Linear(in_features=64, out_features=64, bias=True)
        (k_proj): Linear(in_features=64, out_features=64, bias=True)
        (v_proj): Linear(in_features=64, out_features=64, bias=True)
        (out_proj): Linear(in_features=64, out_features=64, bias=True)
      )
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=64, out_features=128, bias=True)
          (1): ReLU()
          (2): Linear(in_features=128, out_features=64, bias=True)
        )
      )
    )
  )
  (ln_f): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=64, out_features=16, bias=True)
)


In [ ]:
# 손실 함수와 옵티마이저 설정
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# 학습 전 출력 모양 확인
model.eval()
with torch.no_grad():
    logits = model(x_data)

print("logits shape:", logits.shape)
#(batch_size, seq_len, vocab_size)

logits shape: torch.Size([10, 5, 16])


In [ ]:
#학습 루프

epochs = 500

for epoch in range(epochs):
    model.train()

    logits = model(x_data)

    loss = criterion(
        logits.view(-1, vocab_size),
        y_data.view(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 50/500, Loss: 0.6071
Epoch 100/500, Loss: 0.5559
Epoch 150/500, Loss: 0.5470
Epoch 200/500, Loss: 0.5430
Epoch 250/500, Loss: 0.5409
Epoch 300/500, Loss: 0.5397
Epoch 350/500, Loss: 0.5387
Epoch 400/500, Loss: 0.5381
Epoch 450/500, Loss: 0.5376
Epoch 500/500, Loss: 0.5373


- 왜 view(-1, vocab_size)를 하나?

    - 현재 logits shape은 (batch_size, seq_len, vocab_size)
    - 인데, CrossEntropyLoss는 보통 (전체 샘플 수, 클래스 수)형태를 기대함.

- 그래서 배치와 시퀀스 축을 합쳐서
모든 위치를 하나의 큰 분류 문제처럼 바꿈

In [ ]:
#생성 함수 정의
def generate(model, prompt="", max_new_tokens=10):
    model.eval()
    # 평가 모드로 전환

    if prompt.strip() == "":
        #프롬프트가 비어있으면 BOS 토큰만 시작
        input_ids = [BOS_ID]
    else:
        # 프롬프트가 있으면 BOS 뒤에 프롬프트 토큰 ID로 인코딩해서 붙임
        input_ids = [BOS_ID] + encode(prompt)

    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    # 현재 입력 토큰들을 텐서로 변환하고 device로 이동 / shape: (1, seq_len)

    for _ in range(max_new_tokens):
        # 최대 max_new_token까지 새 토큰 생성
        if input_tensor.size(1) > max_len - 1:
            # 현재 입력 길이가 모델이 허용하는 최대 길이보다 길어지면
            input_tensor = input_tensor[:, -(max_len - 1):]
            # 최근 max_len-1개 토큰만 남기고 잘라냄

        with torch.no_grad():
            # gradient 계산 없이 추론만 수행
            logits = model(input_tensor)
            # 모델에 현재 입력을 넣어 전체 위치의 logitis을 얻음 # shape: (1, seq_len, vocab_size)
            next_token_logits = logits[:, -1, :]
            # 마지막 위치의 logits만 사용 , "다음 토큰"에 대한 예측 점수 / shape: (1, vocab_size)
            next_token_id = torch.argmax(next_token_logits, dim=-1).item()
            # 가장 점수가 큰 토큰을 선택, greedy decoing 방식

        input_ids.append(next_token_id)
        # 선택된 다음 토큰을 input_ids에 추가
        input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
        # 새롭게 늘어난 input_ids를 다시 텐서로 변환

        if next_token_id == EOS_ID:
            break
        # EOS 토큰이 생성되면 문장 종료

    return decode(input_ids)
    # 최종적으로 생성된 토큰 ID들을 다시 문자열로 디코팅해서 반환.

In [ ]:
# 생성 테스트
print("\n생성 결과")
prompts = ["i like", "you like", "math", "deep learning", "transformers"]

for p in prompts:
    result = generate(model, prompt=p, max_new_tokens=5)
    print(f"입력: {p}")
    print(f"출력: {result}")
    print("-" * 40)


생성 결과
입력: i like
출력: i like deep learning
----------------------------------------
입력: you like
출력: you like coding
----------------------------------------
입력: math
출력: math is fun
----------------------------------------
입력: deep learning
출력: deep learning is fun
----------------------------------------
입력: transformers
출력: transformers are powerful
----------------------------------------


In [ ]:
# 빈 프롬프트로도 생성해보기
result = generate(model, prompt="", max_new_tokens=6)
print("빈 프롬프트 생성 결과:", result)

빈 프롬프트 생성 결과: i like deep learning


- 한국어로도 진행해보자.

In [ ]:
korean_texts = [
    "나는 수학을 좋아한다",
    "나는 과학을 좋아한다",
    "나는 코딩을 좋아한다",
    "너는 수학을 좋아한다",
    "너는 과학을 좋아한다",
    "너는 코딩을 좋아한다",
    "우리는 수학을 공부한다",
    "우리는 과학을 공부한다",
    "우리는 코딩을 공부한다",
    "수학은 재미있다",
    "과학은 재미있다",
    "코딩은 재미있다"
]

print("한국어 문장 개수:", len(korean_texts))
for t in korean_texts:
    print(t)

한국어 문장 개수: 12
나는 수학을 좋아한다
나는 과학을 좋아한다
나는 코딩을 좋아한다
너는 수학을 좋아한다
너는 과학을 좋아한다
너는 코딩을 좋아한다
우리는 수학을 공부한다
우리는 과학을 공부한다
우리는 코딩을 공부한다
수학은 재미있다
과학은 재미있다
코딩은 재미있다


In [ ]:
# 한국어 사전 만들기
korean_special_tokens = ["<PAD>", "<BOS>", "<EOS>"]

korean_word_set = set()
for text in korean_texts:
    for word in text.split():
        korean_word_set.add(word)

korean_vocab = korean_special_tokens + sorted(list(korean_word_set))

korean_stoi = {word: i for i, word in enumerate(korean_vocab)}
korean_itos = {i: word for word, i in korean_stoi.items()}

korean_vocab_size = len(korean_vocab)

print("korean_vocab_size:", korean_vocab_size)
print("korean_vocab:", korean_vocab)

korean_vocab_size: 15
korean_vocab: ['<PAD>', '<BOS>', '<EOS>', '공부한다', '과학은', '과학을', '나는', '너는', '수학은', '수학을', '우리는', '재미있다', '좋아한다', '코딩은', '코딩을']


In [ ]:
# 한국어용 토큰 함수 만들기
K_PAD_ID = korean_stoi["<PAD>"]
K_BOS_ID = korean_stoi["<BOS>"]
K_EOS_ID = korean_stoi["<EOS>"]

def korean_encode(text):
    return [korean_stoi[word] for word in text.split()]

def korean_decode(token_ids):
    words = []
    for idx in token_ids:
        word = korean_itos[idx]
        if word == "<EOS>":
            break
        if word not in ["<BOS>", "<PAD>"]:
            words.append(word)
    return " ".join(words)

print("K_PAD_ID:", K_PAD_ID)
print("K_BOS_ID:", K_BOS_ID)
print("K_EOS_ID:", K_EOS_ID)

K_PAD_ID: 0
K_BOS_ID: 1
K_EOS_ID: 2


In [ ]:
# 한국어 문장을 숫자로 바꾸고 BOS, EOS 붙이기
korean_encoded_texts = []
for text in korean_texts:
    ids = [K_BOS_ID] + korean_encode(text) + [K_EOS_ID]
    korean_encoded_texts.append(ids)

for text, ids in zip(korean_texts, korean_encoded_texts):
    print(text, "->", ids)

나는 수학을 좋아한다 -> [1, 6, 9, 12, 2]
나는 과학을 좋아한다 -> [1, 6, 5, 12, 2]
나는 코딩을 좋아한다 -> [1, 6, 14, 12, 2]
너는 수학을 좋아한다 -> [1, 7, 9, 12, 2]
너는 과학을 좋아한다 -> [1, 7, 5, 12, 2]
너는 코딩을 좋아한다 -> [1, 7, 14, 12, 2]
우리는 수학을 공부한다 -> [1, 10, 9, 3, 2]
우리는 과학을 공부한다 -> [1, 10, 5, 3, 2]
우리는 코딩을 공부한다 -> [1, 10, 14, 3, 2]
수학은 재미있다 -> [1, 8, 11, 2]
과학은 재미있다 -> [1, 4, 11, 2]
코딩은 재미있다 -> [1, 13, 11, 2]


In [ ]:
# 한국어 패딩 처리
korean_max_len = max(len(ids) for ids in korean_encoded_texts)
print("korean_max_len:", korean_max_len)

def korean_pad_sequence(seq, max_len, pad_value=K_PAD_ID):
    return seq + [pad_value] * (max_len - len(seq))

korean_padded_texts = [korean_pad_sequence(ids, korean_max_len) for ids in korean_encoded_texts]

for seq in korean_padded_texts:
    print(seq)

korean_max_len: 5
[1, 6, 9, 12, 2]
[1, 6, 5, 12, 2]
[1, 6, 14, 12, 2]
[1, 7, 9, 12, 2]
[1, 7, 5, 12, 2]
[1, 7, 14, 12, 2]
[1, 10, 9, 3, 2]
[1, 10, 5, 3, 2]
[1, 10, 14, 3, 2]
[1, 8, 11, 2, 0]
[1, 4, 11, 2, 0]
[1, 13, 11, 2, 0]


In [ ]:
# 한국어 입력 x, 정답 y 만들기
korean_x_data = []
korean_y_data = []

for seq in korean_padded_texts:
    korean_x_data.append(seq[:-1])
    korean_y_data.append(seq[1:])

korean_x_data = torch.tensor(korean_x_data, dtype=torch.long).to(device)
korean_y_data = torch.tensor(korean_y_data, dtype=torch.long).to(device)

print("korean_x_data shape:", korean_x_data.shape)
print("korean_y_data shape:", korean_y_data.shape)

print("\n첫 번째 한국어 입력 x:")
print(korean_x_data[0])

print("\n첫 번째 한국어 정답 y:")
print(korean_y_data[0])

korean_x_data shape: torch.Size([12, 4])
korean_y_data shape: torch.Size([12, 4])

첫 번째 한국어 입력 x:
tensor([ 1,  6,  9, 12], device='cuda:0')

첫 번째 한국어 정답 y:
tensor([ 6,  9, 12,  2], device='cuda:0')


In [ ]:
# 한국어 Mini GPT 모델 만들기
korean_model = MiniGPT(
    vocab_size=korean_vocab_size,
    max_len=korean_max_len - 1,
    d_model=64,
    num_heads=4,
    d_ff=128,
    num_layers=2
).to(device)

print(korean_model)

MiniGPT(
  (token_embedding): Embedding(15, 64)
  (pos_embedding): PositionalEmbedding(
    (pos_embedding): Embedding(4, 64)
  )
  (blocks): ModuleList(
    (0-1): 2 x DecoderBlock(
      (ln1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (q_proj): Linear(in_features=64, out_features=64, bias=True)
        (k_proj): Linear(in_features=64, out_features=64, bias=True)
        (v_proj): Linear(in_features=64, out_features=64, bias=True)
        (out_proj): Linear(in_features=64, out_features=64, bias=True)
      )
      (ln2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=64, out_features=128, bias=True)
          (1): ReLU()
          (2): Linear(in_features=128, out_features=64, bias=True)
        )
      )
    )
  )
  (ln_f): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=64, out_features=15, bias=True)
)


In [ ]:
# 한국어 손실 함수와 옵티마이저
korean_criterion = nn.CrossEntropyLoss(ignore_index=K_PAD_ID)
korean_optimizer = torch.optim.Adam(korean_model.parameters(), lr=0.001)

In [ ]:
# 한국어 모델 학습
korean_epochs = 500

for epoch in range(korean_epochs):
    korean_model.train()

    korean_logits = korean_model(korean_x_data)

    korean_loss = korean_criterion(
        korean_logits.view(-1, korean_vocab_size),
        korean_y_data.view(-1)
    )

    korean_optimizer.zero_grad()
    korean_loss.backward()
    korean_optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"[한국어] Epoch {epoch+1}/{korean_epochs}, Loss: {korean_loss.item():.4f}")

[한국어] Epoch 50/500, Loss: 0.7044
[한국어] Epoch 100/500, Loss: 0.6773
[한국어] Epoch 150/500, Loss: 0.6710
[한국어] Epoch 200/500, Loss: 0.6682
[한국어] Epoch 250/500, Loss: 0.6666
[한국어] Epoch 300/500, Loss: 0.6656
[한국어] Epoch 350/500, Loss: 0.6649
[한국어] Epoch 400/500, Loss: 0.6645
[한국어] Epoch 450/500, Loss: 0.6642
[한국어] Epoch 500/500, Loss: 0.6639


In [ ]:
# 한국어 생성 함수
def korean_generate(model, prompt="", max_new_tokens=10):
    model.eval()

    if prompt.strip() == "":
        input_ids = [K_BOS_ID]
    else:
        input_ids = [K_BOS_ID] + korean_encode(prompt)

    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    for _ in range(max_new_tokens):
        if input_tensor.size(1) > korean_max_len - 1:
            input_tensor = input_tensor[:, -(korean_max_len - 1):]

        with torch.no_grad():
            logits = model(input_tensor)
            next_token_logits = logits[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1).item()

        input_ids.append(next_token_id)
        input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

        if next_token_id == K_EOS_ID:
            break

    return korean_decode(input_ids)

In [ ]:
# 한국어 생성 테스트

print("\n한국어 생성 결과")
korean_prompts = ["나는", "너는", "우리는", "수학은", "과학은", "코딩은"]

for p in korean_prompts:
    result = korean_generate(korean_model, prompt=p, max_new_tokens=5)
    print(f"입력: {p}")
    print(f"출력: {result}")
    print("-" * 40)


한국어 생성 결과
입력: 나는
출력: 나는 코딩을 좋아한다
----------------------------------------
입력: 너는
출력: 너는 수학을 좋아한다
----------------------------------------
입력: 우리는
출력: 우리는 수학을 공부한다
----------------------------------------
입력: 수학은
출력: 수학은 재미있다
----------------------------------------
입력: 과학은
출력: 과학은 재미있다
----------------------------------------
입력: 코딩은
출력: 코딩은 재미있다
----------------------------------------


In [ ]:
# 빈 프롬프트 생성
result = korean_generate(korean_model, prompt="", max_new_tokens=6)
print("빈 프롬프트 생성 결과:", result)

빈 프롬프트 생성 결과: 너는 수학을 좋아한다


# GPT-2 / DistilGPT2

---

## 1. Pretrained GPT란?

Pretrained GPT는 대규모 텍스트 데이터로 **미리 학습된 언어모델**이다.  
사용자는 이 모델을 처음부터 학습하지 않고, 이미 학습된 가중치를 불러와 바로 사용할 수 있다.

GPT 계열 모델은 기본적으로 다음과 같은 방식으로 학습.

- 입력된 토큰들을 보고
- **다음 토큰(next token)** 을 예측한다.
- 이 과정을 매우 많은 텍스트에 대해 반복하면서
- 문장 구조, 단어 패턴, 문맥 정보를 학습.

예를 들어,

> Deep learning is

라는 입력이 주어졌을 때, 모델은 다음에 올 단어로 `a`, `important`, `useful`, `powerful` 같은 후보를 예측할 수 있다.

이 예측을 한 번만 하는 것이 아니라, 예측된 토큰을 다시 입력에 붙여서 다음 토큰을 계속 생성한다.  
이렇게 해서 문장이 길게 이어짐.

---

# 2. GPT-2와 DistilGPT2

## 2.1 GPT-2

GPT-2는 대표적인 **decoder-only Transformer 기반 언어모델**이다.  
왼쪽의 문맥만 보고 다음 토큰을 예측하는 방식으로 학습.

특징은 다음과 같다.

- 생성형 언어모델
- autoregressive 방식
- next-token prediction 기반 학습
- 문장 이어쓰기, 요약, 질의응답 등에 활용 가능

---

## 2.2 DistilGPT2

DistilGPT2는 GPT-2를 더 가볍게 만든 모델.

특징은 다음과 같다.

- GPT-2보다 크기가 작다
- 속도가 빠르다
- 메모리 사용량이 적다



- **GPT-2**: 표준적인 pretrained GPT 실습 모델
- **DistilGPT2**: 가볍게 실습하기 좋은 모델

In [1]:
!pip install transformers torch

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
# GPT 불러오기
model_name = "gpt2"

tokenizer_gpt2 = AutoTokenizer.from_pretrained(model_name)
model_gpt2 = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:
distil_model_name = "distilgpt2"

tokenizer_distil = AutoTokenizer.from_pretrained(distil_model_name)
model_distil = AutoModelForCausalLM.from_pretrained(distil_model_name)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

model_gpt2 = model_gpt2.to(device)
model_distil = model_distil.to(device)

print(device)

mps


In [6]:
#tokenizer 이해하기
##언어모델은 문자열을 그대로 이해하지 못함 -> 입력 문장은 먼저 토큰(token) 으로 나뉘고, 각 토큰은 정수 ID로 변환

text = "I love natural language processing."

tokens = tokenizer_gpt2.tokenize(text)
print(tokens)

['I', 'Ġlove', 'Ġnatural', 'Ġlanguage', 'Ġprocessing', '.']


In [7]:
#encode
ids = tokenizer_gpt2.encode(text)
print(ids)

[40, 1842, 3288, 3303, 7587, 13]


In [8]:
#모델 출력 직접 보기
text = "Deep learning is"
inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model_gpt2(**inputs)

print(outputs.logits.shape)
#(batch_size, seq_len, vocab_size)

torch.Size([1, 3, 50257])


In [9]:
#마지막 토큰 위치의 예측
last_token_logits = outputs.logits[:, -1, :]
print(last_token_logits.shape)

torch.Size([1, 50257])


In [10]:
# 가장 확률이 높은 것
pred_id = torch.argmax(last_token_logits, dim=-1)
print(pred_id)

tensor([257], device='mps:0')


In [11]:
# 토큰을 문장으로
pred_token = tokenizer_gpt2.decode(pred_id)
print(pred_token)
#현재 문장 뒤에 가장 올 가능성이 높은 다음 토큰

 a


In [12]:
#직접 한 토큰씩 예측할 수도 있지만, 보통은 generate()를 사용.
text = "Deep learning is"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

output = model_gpt2.generate(**inputs, max_length=30)

generated_text = tokenizer_gpt2.decode(output[0], skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Deep learning is a very powerful tool for learning about the world around us. It's a tool that can help us understand the world around us, and


In [13]:
output = model_gpt2.generate(**inputs, max_length=30)
#max_length=30은 입력 토큰 + 생성 토큰을 합친 전체 길이의 최대값

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


- Greedy decoding

    - Greedy decoding은 매 단계마다 가장 점수가 높은 토큰 하나만 선택하는 방식

In [14]:
text = "The future of AI is"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

output = model_gpt2.generate(
    **inputs,
    max_length=40,
    do_sample=False  # 확률적인 샘플링을 사용할 것인가? or 가장 확률이 높은 단어만 선택할 것인가?(greed-decoing)
)

print(tokenizer_gpt2.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The future of AI is uncertain. The future of AI is uncertain.

The future of AI is uncertain. The future of AI is uncertain.

The future of AI is uncertain. The


- do_sample=False
- 매 시점에서 argmax 선택
- 가장 결정적이지만, 반복적이고 단조로운 출력이 나올 수 있음

- do_sample=False(Greedhy Search)
    - 특징 : 결과가 항상 일정함(결론론적). 똑같은 입력을 주면 똑같은 출력을 나옴.
    - 단점 : 모델이 반복적인 문구를 생성하거나, 문맥상 단조로운 답변을 내놓을 확률이 높아짐.

- Greedy decoding 대신 확률분포에서 샘플링

In [15]:
text = "The future of AI is"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

output = model_gpt2.generate(
    **inputs,
    max_length=40,
    do_sample=True
)

print(tokenizer_gpt2.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The future of AI is uncertain, with different ways to achieve it; one proposal is to rely on the AI's natural intelligence to help us avoid error, while another is to enable it to work best


- Top-k sampling은 확률이 높은 상위 k개 후보만 남기고, 그 안에서 샘플링하는 방식

In [16]:
text = "The future of AI is"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

output = model_gpt2.generate(
    **inputs,
    max_length=40,
    do_sample=True,
    top_k=50
)

print(tokenizer_gpt2.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The future of AI is in the control algorithms, but that's probably not the end of the story: the human being has the option of making an intelligent choice about its future (assuming, of course


- Top-p sampling은 누적 확률이 p가 될 때까지 상위 후보를 남기고, 그 안에서 샘플링하는 방식 ->nucleus sampling 이라고 함

In [17]:
text = "The future of AI is"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

output = model_gpt2.generate(
    **inputs,
    max_length=40,
    do_sample=True,
    top_p=0.9
)

print(tokenizer_gpt2.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The future of AI is now quite bright and I'm excited about our journey. We have been doing this for a couple of years now.

I'm always going to say the same thing with


- Temperature는 확률분포를 얼마나 날카롭게 또는 평평하게 만들 것인가를 조절하는 값
    - temperature < 1: 높은 확률 토큰을 더 강하게 선택
    - temperature = 1: 원래 분포 유지
    - temperature > 1: 다양한 토큰이 더 쉽게 선택됨

In [18]:
text = "The future of AI is"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

output = model_gpt2.generate(
    **inputs,
    max_length=40,
    do_sample=True,
    top_p=0.9,
    temperature=0.7
)

print(tokenizer_gpt2.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The future of AI is not in the future, it is in the future," said Mr. Rocha. "The future is going to be AI, not computers. We're not talking about


In [19]:
#GPT-2와 DistilGPT2 비교 실험
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

text = "Machine learning can help us"

# GPT-2
tokenizer_gpt2 = AutoTokenizer.from_pretrained("gpt2")
model_gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

inputs_gpt2 = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs_gpt2 = {k: v.to(device) for k, v in inputs_gpt2.items()}

start = time.time()
output_gpt2 = model_gpt2.generate(
    **inputs_gpt2,
    max_length=40,
    do_sample=True,
    top_p=0.9,
    temperature=0.8
)
end = time.time()

print("=== GPT-2 ===")
print(tokenizer_gpt2.decode(output_gpt2[0], skip_special_tokens=True))
print("생성 시간:", end - start)

# DistilGPT2
tokenizer_distil = AutoTokenizer.from_pretrained("distilgpt2")
model_distil = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)

inputs_distil = tokenizer_distil(text, return_tensors="pt").to(device)
inputs_distil = {k: v.to(device) for k, v in inputs_distil.items()}

start = time.time()
output_distil = model_distil.generate(
    **inputs_distil,
    max_length=40,
    do_sample=True,
    top_p=0.9,
    temperature=0.8
)
end = time.time()

print("\n=== DistilGPT2 ===")
print(tokenizer_distil.decode(output_distil[0], skip_special_tokens=True))
print("생성 시간:", end - start)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== GPT-2 ===
Machine learning can help us understand the structure of the network and how it works, but it's not the only way to learn how to do this. If you're interested in learning about network architectures,
생성 시간: 0.32439279556274414


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



=== DistilGPT2 ===
Machine learning can help us to understand and understand the world.




























생성 시간: 0.2998666763305664


In [20]:
# 같은 문장으로 여러번 생성해보면 sampling의 효과를 얻을 수 있다.
text = "Science will change"

inputs = tokenizer_distil(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

for i in range(5):
    output = model_distil.generate(
        **inputs,
        max_length=35,
        do_sample=True,
        top_p=0.9,
        temperature=1.0
    )
    print(f"결과 {i+1}:")
    print(tokenizer_distil.decode(output[0], skip_special_tokens=True))
    print()

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


결과 1:
Science will change the way the world is used by those living on Earth, according to a new report in the journal Science.













Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


결과 2:
Science will change. ‒
































Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


결과 3:
Science will change that, by the end of the next century.





The last few years have seen big increases in the number of college students and adults

결과 4:
Science will change that.‌

결과 5:
Science will change what we think about when they're at risk of death.›



## 과제)DistilGPT2로 직접 생성 함수 만들기

DistilGPT2을 사용하여 **한 토큰씩 직접 생성**하는 함수를 구현

```
generate_manual(model, tokenizer, prompt, max_new_tokens=30)
```

- 함수 요구사항
    - 입력 프롬프트를 토큰화할 것
    - 현재까지의 입력을 모델에 넣고, 마지막 위치의 logits를 사용하여 다음 토큰을 예측 할것
    - greedy decoding을 사용할 것. -> 가장 큰 logits값을 갖는 토큰을 다음 토큰을 선택 할 것
    - 생성된 토큰을 기존 입력 뒤에 붙이고, 이 과정을 반복할 것
    - max_new_token만큼 생성하거나, eos_token_id가 나오면 종료 할것
    - 최종 결과를 문자열로 복원하여 변환할 것


- 프롬프트

> "Artificial intelligence is"

> "The future of education will"

> "Once upon a time"

In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

def generate_manual(model, tokenizer, prompt, max_new_tokens=30):
    model.eval()

    # 프롬프트를 토큰 ID로 변환
    if prompt.strip() == "":
        input_ids = [tokenizer.eos_token_id]
    else:
        input_ids = tokenizer.encode(prompt, add_special_tokens=False)

    # 텐서로 변환
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            # 현재 입력 전체를 모델에 넣음
            outputs = model(input_tensor)

            # 마지막 위치의 logits만 사용 / shape: (1, vocab_size)
            next_token_logits = outputs.logits[:, -1, :]

            # greedy decoding: 가장 큰 값을 선택
            next_token_id = torch.argmax(next_token_logits, dim=-1).item()

        # 선택된 토큰을 기준 입력 뒤에 추가
        input_ids.append(next_token_id)

        # EOS 토큰이 나오면 종료
        if next_token_id == tokenizer.eos_token_id:
            break

        # 모델 최대 길이를 넘지 않도록 최근 토큰만 유지
        max_positions = model.config.n_positions
        if len(input_ids) > max_positions:
            input_ids = input_ids[-max_positions:]

        # 업데이트된 입력을 다시 텐서로 변환
        input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    # 생성된 전체 토큰을 문자열로 복원
    return tokenizer.decode(input_ids, skip_special_tokens=True)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [22]:
prompt = "Artificial intelligence is"
result = generate_manual(model, tokenizer, prompt, max_new_tokens=30)
print(result)

Artificial intelligence is a new technology that is being developed by researchers at the University of California, Berkeley.















## Beam Search Decoding

지금까지는 다음과 같은 생성 방식을 실습했다.

- greedy decoding
- top-k sampling
- top-p sampling
- temperature 조절

이제 여기에 **beam search decoding**을 추가해볼 수 있다.

---

## 1. Beam Search란?

Greedy decoding은 매 단계마다 **가장 점수가 높은 토큰 하나만 선택**.
이 방식은 단순하고 빠르지만, 초반에 한 번 잘못 선택하면 뒤의 문장 전체가 제한될 수 있다.

Beam search는 한 시점에서 후보를 하나만 남기지 않고,**점수가 높은 여러 개의 후보 경로를 동시에 유지**하면서 문장을 생성하는 방식.


- greedy: 매 단계마다 1개 후보만 유지
- beam search: 매 단계마다 여러 개 후보를 유지

이렇게 하면 더 좋은 전체 문장을 찾을 가능성이 높아짐.

---

## 2. Beam Search의 직관

예를 들어 입력이 다음과 같다고 하자.

> The future of AI

Greedy decoding은 첫 단계에서 가장 점수가 높은 단어 하나만 선택한다.

예를 들어:

- `"is"` 선택

그러면 이후 모든 생성은 `"The future of AI is ..."` 경로만 따라간다.

반면 beam search에서 `num_beams=3`이면, 처음 단계에서 점수가 높은 3개의 후보 경로를 남길 수 있다.

예를 들어:

1. The future of AI **is**
2. The future of AI **will**
3. The future of AI **can**

그리고 다음 단계에서도 각 경로를 계속 확장하면서 전체 점수가 좋은 문장을 고른다.

beam search는 **부분적으로 좋은 선택 몇 개를 끝까지 비교하는 방식**.


In [23]:
text = "The future of AI is"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

output_beam = model_gpt2.generate(
    **inputs,
    max_length=40,
    num_beams=5,
    early_stopping=True  #문장이 충분히 끝났다고 판단되면 조기에 생성을 멈춘다.
)

print(tokenizer_gpt2.decode(output_beam[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The future of AI is in the hands of the next generation of scientists and engineers.

The future of AI is in the hands of the next generation of scientists and engineers.

The future


In [24]:
text = "Machine learning can"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Greedy
output_greedy = model_gpt2.generate(
    **inputs,
    max_length=40,
    do_sample=False
)

# Beam Search
output_beam = model_gpt2.generate(
    **inputs,
    max_length=40,
    num_beams=5,
    early_stopping=True
)

print("=== Greedy ===")
print(tokenizer_gpt2.decode(output_greedy[0], skip_special_tokens=True))

print("\n=== Beam Search ===")
print(tokenizer_gpt2.decode(output_beam[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== Greedy ===
Machine learning can be used to predict the future.

The first step is to understand the neural network that is responsible for the prediction. The neural network is a set of neurons that are connected to

=== Beam Search ===
Machine learning can also be used to predict the future.

In this post, I'll show you how you can use this technique to predict the future. I'll also show you how you can


In [25]:
##Beam Search + 반복 방지
text = "Machine learning can"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

output_beam = model_gpt2.generate(
    **inputs,
    max_length=40,
    num_beams=5,
    early_stopping=True,
    no_repeat_ngram_size=2 #비슷한 표현을 반복 방지
    )

print(tokenizer_gpt2.decode(output_beam[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Machine learning can also be used to predict the future.

In this post, I'll show you how you can use this technique to create a predictive model that predicts future events based on the current


## Contrastive Search Decoding

지금까지는 다음과 같은 생성 방식을 살펴보았다.

- greedy decoding
- beam search
- top-k sampling
- top-p sampling
- temperature 조절

---

## Contrastive Search란?

Contrastive search는 **확률이 높은 토큰을 선택하되, 이미 생성된 문장과 너무 비슷하여 반복을 만들 가능성이 큰 토큰은 피하는 방식**.

다음 토큰을 고를 때 단순히

- 확률이 높은가?

만 보는 것이 아니라,

- 확률이 높으면서
- 반복적인 문장을 만들 가능성은 낮은가?

를 함께 고려.

이 방식은 다음 두 문제를 줄이기 위해 제안됨.

### Greedy / Beam Search의 문제
- 확률이 높은 토큰만 계속 선택하면
- 문장이 반복적이고 단조로워질 수 있다.

### Sampling의 문제
- 다양성은 생기지만
- 너무 랜덤해서 문맥과 어색한 문장이 나올 수 있다.

Contrastive search는 이 두 방식의 중간에서, **자연스러우면서도 반복이 적은 문장**을 만들기 위한 방법.

---

## 핵심 아이디어

Contrastive search는 상위 후보들 중에서 다음을 함께 고려.

1. **모델 확률이 높은 후보**
2. **이미 생성된 문장과 지나치게 비슷하지 않은 후보**



> 확률은 높되, 이미 나온 표현을 반복할 가능성이 큰 후보는 덜 선호.

이런 방식으로 다음 토큰을 선택.

---

## 왜 필요한가?

일반적인 생성에서는 자주 다음과 같은 문제가 생긴다.

### 예시: 반복 문제
입력이 다음과 같다고 하자.

> Deep learning is

Greedy decoding에서는 높은 확률의 안전한 표현을 계속 고르게 되어,

> Deep learning is very important and very important and very important ...

처럼 반복적인 문장이 만들어질 수 있다.

Contrastive search는 이런 반복을 줄이려 한다.

---

## 29.4 Greedy / Beam / Sampling / Contrastive Search 비교

| 방식 | 선택 기준 | 장점 | 단점 |
|------|------|------|------|
| Greedy | 매 단계 최고 확률 1개 선택 | 빠르고 단순함 | 반복적일 수 있음 |
| Beam Search | 여러 후보 경로 유지 | 더 나은 전체 문장 가능 | 다양성 부족 |
| Sampling | 확률적으로 토큰 선택 | 다양성 높음 | 불안정할 수 있음 |
| Contrastive Search | 높은 확률 + 낮은 반복성 | 자연스럽고 반복 적음 | 설정 이해가 필요함 |

In [27]:
text = "The future of AI is"

inputs = tokenizer_gpt2(text, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

output_contrastive = model_gpt2.generate(
    **inputs,
    custom_generate="transformers-community/contrastive-search",
    trust_remote_code=True,
    penalty_alpha=0.6,
    top_k=4,
    max_new_tokens=30
)

print(tokenizer_gpt2.decode(output_contrastive[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
An assistant model is provided, using a dynamic cache instead of a cache of type='dynamic_full'.


The future of AI is in the hands of the next generation of AI.

The future of AI is in the hands of the next generation of AI.

The


In [29]:
# Greedy와 Contrastive Search 비교

text = "Machine learning can"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Greedy
output_greedy = model_gpt2.generate(
    **inputs,
    max_length=40,
    do_sample=False
)

# Contrastive Search
output_contrastive = model_gpt2.generate(
    **inputs,
    custom_generate="transformers-community/contrastive-search",
    trust_remote_code=True,
    max_length=40,
    penalty_alpha=0.6,
    top_k=4
)

print("=== Greedy ===")
print(tokenizer_gpt2.decode(output_greedy[0], skip_special_tokens=True))

print("\n=== Contrastive Search ===")
print(tokenizer_gpt2.decode(output_contrastive[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== Greedy ===
Machine learning can be used to predict the future.

The first step is to understand the neural network that is responsible for the prediction. The neural network is a set of neurons that are connected to

=== Contrastive Search ===
Machine learning can be used to predict the future.

The goal is to develop a system that can predict the future.

The system is designed to be able to predict the future.



In [30]:
# Beam Search와 contrastive search 비교
text = "Machine learning can"

inputs = tokenizer_gpt2(text, return_tensors="pt").to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Beam Search
output_beam = model_gpt2.generate(
    **inputs,
    max_length=40,
    num_beams=5,
    early_stopping=True,
    no_repeat_ngram_size=2
)

# Contrastive Search
output_contrastive = model_gpt2.generate(
    **inputs,
    custom_generate="transformers-community/contrastive-search",
    trust_remote_code=True,
    max_length=40,
    penalty_alpha=0.6,
    top_k=4
)

print("=== Beam Search ===")
print(tokenizer_gpt2.decode(output_beam[0], skip_special_tokens=True))

print("\n=== Contrastive Search ===")
print(tokenizer_gpt2.decode(output_contrastive[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


=== Beam Search ===
Machine learning can also be used to predict the future.

In this post, I'll show you how you can use this technique to create a predictive model that predicts future events based on the current

=== Contrastive Search ===
Machine learning can be used to predict the future.

The goal is to develop a system that can predict the future.

The system is designed to be able to predict the future.



## 동적 패딩(dynamic padding)

현재 batch에 들어온 샘플 중 가장 긴 길이를 맞춰서만 패딩하는 방식

-> 데이터셋 전체에서 가장 긴 문장 길이에 맞춰 패딩하는 것이 아니라, 매 배치마다 필요한 만큼 배치함.

1. 계산량 감소
 - 패딩이 많을 수록 불필요한 attention, matrix multiplication이 늘어남

2. 메모리 절약
 - 짧은 문장이 많은 데이터셋에서 특히 효과가 큼

3. 실제 학습 파이프라인과 잘 맞음

In [31]:
# 단순한 패딩 패딩 함수
import torch
from torch.nn.utils.rnn import pad_sequence

# 길이가 서로 다른 시퀀스
seq1 = torch.tensor([10, 11, 12, 13], dtype=torch.long)
seq2 = torch.tensor([20, 21], dtype=torch.long)
seq3 = torch.tensor([30, 31, 32], dtype=torch.long)

batch = [seq1, seq2, seq3]

# 현재 배치의 가장 긴 길이에 맞춰 패딩
padded = pad_sequence(batch, batch_first=True, padding_value=0)

print("padded")
print(padded)
print("shape:", padded.shape)

attention_mask = (padded != 0).long()
print("\nattention_mask")
print(attention_mask)

padded
tensor([[10, 11, 12, 13],
        [20, 21,  0,  0],
        [30, 31, 32,  0]])
shape: torch.Size([3, 4])

attention_mask
tensor([[1, 1, 1, 1],
        [1, 1, 0, 0],
        [1, 1, 1, 0]])


In [32]:
# DataCollatorWithPadding

import torch
from transformers import AutoTokenizer, DataCollatorWithPadding

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT2 계열은 기본적으로 pad_token이 없을 수 있음
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [33]:
texts = [
    "I like math.",
    "Deep learning is powerful.",
    "AI.",
    "This is a much longer sentence than the others."
]

features = [tokenizer(t, truncation=True) for t in texts]

for i, f in enumerate(features):
    print(f"sample {i+1} length:", len(f["input_ids"]))

# padding=True를 주지 않는 것이 중요. 미리 고정 패딩 하지 않고, 배치를 만들 때 collator가 동적으로 패딩하게 둠

sample 1 length: 4
sample 2 length: 5
sample 3 length: 2
sample 4 length: 10


In [34]:
collator = DataCollatorWithPadding(tokenizer=tokenizer)

batch = collator(features)

print("input_ids shape:", batch["input_ids"].shape)
print(batch["input_ids"])
print("\nattention_mask")
print(batch["attention_mask"])

input_ids shape: torch.Size([4, 10])
tensor([[   40,   588, 10688,    13, 50256, 50256, 50256, 50256, 50256, 50256],
        [29744,  4673,   318,  3665,    13, 50256, 50256, 50256, 50256, 50256],
        [20185,    13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256],
        [ 1212,   318,   257,   881,  2392,  6827,   621,   262,  1854,    13]])

attention_mask
tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [35]:
# DataLoader와 함께 사용

from torch.utils.data import Dataset, DataLoader

class MyTextDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.features = [tokenizer(t, truncation=True) for t in texts]

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]

texts = [
    "I like math.",
    "Deep learning is powerful.",
    "AI changes everything.",
    "Hello.",
    "This sentence is a little bit longer than the others."
]

dataset = MyTextDataset(texts, tokenizer)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=collator)

for batch_idx, batch in enumerate(loader):
    print("=" * 60)
    print("batch", batch_idx + 1)
    print("input_ids shape:", batch["input_ids"].shape)
    print(batch["input_ids"])
    print("attention_mask")
    print(batch["attention_mask"])

batch 1
input_ids shape: torch.Size([2, 5])
tensor([[   40,   588, 10688,    13, 50256],
        [29744,  4673,   318,  3665,    13]])
attention_mask
tensor([[1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]])
batch 2
input_ids shape: torch.Size([2, 4])
tensor([[20185,  2458,  2279,    13],
        [15496,    13, 50256, 50256]])
attention_mask
tensor([[1, 1, 1, 1],
        [1, 1, 0, 0]])
batch 3
input_ids shape: torch.Size([1, 11])
tensor([[1212, 6827,  318,  257, 1310, 1643, 2392,  621,  262, 1854,   13]])
attention_mask
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [36]:
# DistilGPT2에 넣어보기

from transformers import AutoModelForCausalLM

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [37]:
# 한 배치 forward

batch = next(iter(loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)

print("logits shape:", outputs.logits.shape)

logits shape: torch.Size([2, 5, 50257])


In [ ]:
# cusotm collator로 casula LM용 동적 패딩 만들기

from transformers import AutoTokenizer
from torch.utils.data import DataLoader, Dataset

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

texts = [
    "I like math.",
    "Deep learning is powerful.",
    "AI changes the world.",
    "Hello."
]

class CausalLMDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.features = [tokenizer(t, truncation=True) for t in texts]

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]

def causal_lm_collator(features):
    batch = tokenizer.pad(features, padding=True, return_tensors="pt")

    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100 #pytorch -100은  학습 안함.
    batch["labels"] = labels

    return batch

dataset = CausalLMDataset(texts, tokenizer)
loader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=causal_lm_collator)

for batch in loader:
    print("=" * 60)
    print("input_ids")
    print(batch["input_ids"])
    print("attention_mask")
    print(batch["attention_mask"])
    print("labels")
    print(batch["labels"])

input_ids
tensor([[   40,   588, 10688,    13, 50256],
        [29744,  4673,   318,  3665,    13]])
attention_mask
tensor([[1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1]])
labels
tensor([[   40,   588, 10688,    13,  -100],
        [29744,  4673,   318,  3665,    13]])
input_ids
tensor([[20185,  2458,   262,   995,    13],
        [15496,    13, 50256, 50256, 50256]])
attention_mask
tensor([[1, 1, 1, 1, 1],
        [1, 1, 0, 0, 0]])
labels
tensor([[20185,  2458,   262,   995,    13],
        [15496,    13,  -100,  -100,  -100]])


In [ ]:
# Loss까지 보기
from transformers import AutoModelForCausalLM

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
model.eval()

batch = next(iter(loader))
batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

print("loss:", outputs.loss.item())
print("logits shape:", outputs.logits.shape)

#version 2.0

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


loss: 6.44114351272583
logits shape: torch.Size([2, 5, 50257])


## Causal LM(Language model, 인과적 언어모델) fine-tuning 이란?

GPT 계열은 **다음 토큰 예측(next token preidciton)**으로 학습됨.

예를 들어

> `<BOS>` I like math <EOS>

이면 개념적으로는

- 입력: `<BOS>` I like math
- 출력:  I like math `<EOS>`

처럼 **한 칸 밀린 목표**를 두고 학습

하지만 Hugging Face의 AutoModelForCausalLM에서는 보통 우리가 input_ids와 labels를 같게 넣어도, 모델 내부에서 causal LM loss 계산을 위해 자동으로 shift 함.



In [ ]:
!pip install -U transformers datasets accelerate
# accelerate: Huggingface에서 제공하는 패키지, Pytorch 코드를 크게 바꾸지 않고도 단일 GPU, 다중 GPU,TPU 등 과 같은 학습과 추론을 쉽게 돌릴 수 있게 해줌.
## 몇줄만 추가해서 다양한 분산 설정에서 같은 Pytorch코드 실행

In [41]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW

In [42]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

Using device: mps


In [43]:
model_id = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

print("모델 로드 완료")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

모델 로드 완료


In [44]:
# 아주 작은 QA 데이터 만들기

train_data = [
    {
        "question": "What is LoRA?",
        "answer": "LoRA is a method for fine-tuning a large model by training only small low-rank adapter matrices."
    },
    {
        "question": "What is QLoRA?",
        "answer": "QLoRA is a memory-efficient fine-tuning method that combines quantization with LoRA adapters."
    },
    {
        "question": "What is attention in transformers?",
        "answer": "Attention is a mechanism that lets each token focus on the most relevant tokens in the sequence."
    },
    {
        "question": "What is fine-tuning?",
        "answer": "Fine-tuning is the process of adapting a pretrained model to a specific task using additional training data."
    },
    {
        "question": "What is quantization?",
        "answer": "Quantization is a technique that reduces memory and computation by representing model weights with lower precision."
    }
]

valid_data = [
    {
        "question": "What is a transformer?",
        "answer": "A transformer is a neural network architecture based on attention mechanisms for modeling relationships in sequences."
    }
]

In [45]:
# 학습용 텍스트 포맷 만들기
## Casual LM에서는 보통 입력과 출력이 한 문자열 안에 함께 들어감.

def format_example(example):
    return f"Question: {example['question']}\nAnswer: {example['answer']}"

In [46]:
for ex in train_data[:2]:
    print(format_example(ex))
    print("-" * 60)

Question: What is LoRA?
Answer: LoRA is a method for fine-tuning a large model by training only small low-rank adapter matrices.
------------------------------------------------------------
Question: What is QLoRA?
Answer: QLoRA is a memory-efficient fine-tuning method that combines quantization with LoRA adapters.
------------------------------------------------------------


In [47]:
# Dataset 만들기
class QADataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = format_example(self.data[idx])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        labels = input_ids.clone()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [48]:
train_dataset = QADataset(train_data, tokenizer, max_length=128)
valid_dataset = QADataset(valid_data, tokenizer, max_length=128)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=1)

In [49]:
# input_ids와 label 확인
sample = train_dataset[0]

print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("labels shape:", sample["labels"].shape)

print("\nDecoded text:")
print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True))
# labels = input_ids.clone()로 둔 이유는 causal LM에서 다음 토큰 예측을 위해 모델 내부가 자동으로 shift 처리하기 때문

input_ids shape: torch.Size([128])
attention_mask shape: torch.Size([128])
labels shape: torch.Size([128])

Decoded text:
Question: What is LoRA?
Answer: LoRA is a method for fine-tuning a large model by training only small low-rank adapter matrices.


In [50]:
# loss 한 번 계산해보기

batch = next(iter(train_loader))
batch = {k: v.to(device) for k, v in batch.items()}

model.eval()
with torch.no_grad():
    outputs = model(**batch)

print("loss:", outputs.loss.item())
print("logits shape:", outputs.logits.shape)

loss: 9.938663482666016
logits shape: torch.Size([2, 128, 50257])


- loss: 현재 모델이 이 배치의 다음 토큰 예측을 얼마나 못 맞히는지
- logits shape = (batch, seq_len, vocab_size)
- 각 위치마다 vocabulary 전체에 대한 점수를 냄

In [51]:
# 학습 전 생성 결과 확인

def generate_answer(question, max_new_tokens=50):
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [52]:
print(generate_answer("What is LoRA?"))
print()
print(generate_answer("What is quantization?"))

# 이 시점에서는 base_model이므로, 형색이 어색할 수 있고 답이 길거나 엉뚱할 수도 있음

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Question: What is LoRA?
Answer: LoRA is a simple, simple, and simple, and simple, and simple, and simple, and simple, and simple, and simple, and simple, and simple, and simple, and simple, and simple, and simple, and simple,

Question: What is quantization?
Answer: Quantization is a term that is used to describe the concept of quantization. It is a term that is used to describe the concept of quantization. It is a term that is used to describe the concept of quantization. It is a term


In [53]:
# 학습 루프 만들기
optimizer = AdamW(model.parameters(), lr=5e-5)


In [54]:
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item()

    return total_loss / len(loader)

In [55]:
epochs = 100

for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0

    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    avg_valid_loss = evaluate(model, valid_loader)

    print(f"Epoch {epoch+1:02d} | train_loss={avg_train_loss:.4f} | valid_loss={avg_valid_loss:.4f}")

# 과적합을 체크할 것.

Epoch 01 | train_loss=6.4973 | valid_loss=1.7565
Epoch 02 | train_loss=1.3314 | valid_loss=1.0307
Epoch 03 | train_loss=1.1256 | valid_loss=1.0598
Epoch 04 | train_loss=1.0892 | valid_loss=0.9877
Epoch 05 | train_loss=1.0095 | valid_loss=0.9226
Epoch 06 | train_loss=0.8788 | valid_loss=0.8794
Epoch 07 | train_loss=0.7981 | valid_loss=0.8470
Epoch 08 | train_loss=0.7306 | valid_loss=0.8214
Epoch 09 | train_loss=0.6631 | valid_loss=0.8011
Epoch 10 | train_loss=0.6146 | valid_loss=0.7829
Epoch 11 | train_loss=0.5537 | valid_loss=0.7681
Epoch 12 | train_loss=0.4941 | valid_loss=0.7586
Epoch 13 | train_loss=0.4281 | valid_loss=0.7533
Epoch 14 | train_loss=0.4067 | valid_loss=0.7516
Epoch 15 | train_loss=0.3567 | valid_loss=0.7545
Epoch 16 | train_loss=0.3022 | valid_loss=0.7605
Epoch 17 | train_loss=0.2991 | valid_loss=0.7694
Epoch 18 | train_loss=0.2440 | valid_loss=0.7806
Epoch 19 | train_loss=0.2265 | valid_loss=0.7946
Epoch 20 | train_loss=0.2102 | valid_loss=0.8106
Epoch 21 | train_los

In [56]:
#학습 후 생성 비교
print(generate_answer("What is LoRA?"))
print()
print(generate_answer("What is quantization?"))
print()
print(generate_answer("What is a transformer?"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Question: What is LoRA?
Answer: LoRA is a method for fine-tuning a large model by training only small low-rank adapter matrices.



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Question: What is quantization?
Answer: Quantization is a technique that reduces memory and computation by representing model weights with lower precision.

Question: What is a transformer?
Answer: A transformer is a method for representing transformers with lower precision.


In [57]:
# overfitting 확인
test_questions = [
    "What is LoRA?",
    "What is QLoRA?",
    "What is attention in transformers?",
    "What is optimization?",
    "What is a neural network?"
]

for q in test_questions:
    print("=" * 80)
    print("QUESTION:", q)
    print(generate_answer(q))
    print()

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


QUESTION: What is LoRA?
Question: What is LoRA?
Answer: LoRA is a method for fine-tuning a large model by training only small low-rank adapter matrices.

QUESTION: What is QLoRA?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Question: What is QLoRA?
Answer: QLoRA is a memory-efficient fine-tuning method that combines quantization with LoRA adapters.

QUESTION: What is attention in transformers?
Question: What is attention in transformers?
Answer: Attention is a mechanism that lets each token focus on the most relevant tokens in the sequence.

QUESTION: What is optimization?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Question: What is optimization?
Answer: Optimization is a technique that reduces memory and computation by representing model weights with lower precision.

QUESTION: What is a neural network?
Question: What is a neural network?
Answer: Neural networks are a mechanism that lets each token focus on the most relevant tokens in the sequence.



- label masking이란?
  - 지금까지는 labels = input_ids.clone()으로 하였음.
  - 그런데 이 방식은 질문 부분과 답변 부분 모두에 대해 loss를 계산.




모델은

> Question: What is LoRA?
> 
> Answer: ...

전체 문자열을 맞히려고 학습.

하지만 실제로는 보통 질문 부분은 맞힐 필요가 없고, 답변 부분만 잘 생성하면 되는 경우가 많습니다.

이럴 때 질문 부분 label을 -100으로 마스킹합니다. PyTorch의 cross-entropy loss에서 -100은 무시됩니다.

In [58]:
# answer 부분만 학습하는 dataset
## 좀 더 정확한 instruction-style fine-tuning

class MaskedQADataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        q = self.data[idx]["question"]
        a = self.data[idx]["answer"]

        prompt = f"Question: {q}\nAnswer:"
        full_text = f"{prompt} {a}"

        enc_full = self.tokenizer(
            full_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        enc_prompt = self.tokenizer(
            prompt,
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = enc_full["input_ids"].squeeze(0)
        attention_mask = enc_full["attention_mask"].squeeze(0)
        labels = input_ids.clone()

        prompt_len = enc_prompt["input_ids"].shape[1]

        labels[:prompt_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [59]:
masked_train_dataset = MaskedQADataset(train_data, tokenizer, max_length=128)
masked_train_loader = DataLoader(masked_train_dataset, batch_size=2, shuffle=True)

sample = masked_train_dataset[0]

print("Decoded input:")
print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True))

print("\nFirst 40 labels:")
print(sample["labels"][:40])
# 여기서 앞부분이 -100이면, 그 부분은 loss에서 제외

Decoded input:
Question: What is LoRA?
Answer: LoRA is a method for fine-tuning a large model by training only small low-rank adapter matrices.

First 40 labels:
tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         6706,  3861,   318,   257,  2446,   329,  3734,    12, 28286,   278,
          257,  1588,  2746,   416,  3047,   691,  1402,  1877,    12, 43027,
        21302,  2603, 45977,    13, 50256, 50256, 50256, 50256, 50256, 50256])


## 1. LLaMA란 무엇인가

**LLaMA**는 Meta가 공개한 대규모 언어모델 계열.  

이름은 **Large Language Model Meta AI**의 약자이며, 대표적으로 **LLaMA 1**, **Llama 2**, **Llama 3**, **Llama 4** 계열로 이어진다.

LLaMA 계열은 기본적으로 **decoder-only Transformer** 구조를 사용하는 **생성형 언어모델(Generative Language Model)** .

앞에 주어진 토큰들을 보고 **다음 토큰을 예측하는 방식**으로 학습

LLaMA가 중요하게 평가되는 이유는 단순히 모델 크기 때문이 아니다.

초기 LLaMA 논문은, **효율적인 학습 데이터 구성과 학습 전략**을 통해 상대적으로 적은 파라미터 수로도 강한 성능을 낼 수 있음을 보여주었다. 특히 LLaMA 1 논문은 13B 모델이 일부 벤치마크에서 GPT-3 175B보다 우수한 결과를 보였다고 보고했다.

## 2. LLaMA를 이해하기 전에 알아야 할 배경

LLaMA를 이해하려면 먼저 **GPT 계열 모델**의 기본 틀을 알아야 한다.

Transformer에는 크게 세 가지 형태가 있다.

1. **Encoder-only**
   - 대표 예: BERT
   - 문장 이해, 분류, 문장쌍 비교 등에 강함

2. **Encoder-Decoder**
   - 대표 예: T5, BART
   - 입력을 이해한 뒤 출력 문장을 생성
   - 번역, 요약 등에 적합

3. **Decoder-only**
   - 대표 예: GPT, LLaMA
   - 왼쪽 문맥만 보고 다음 토큰을 생성
   - 자연스러운 텍스트 생성, 대화, 코드 생성에 강함

LLaMA는 이 중에서 **decoder-only 구조**를 채택한 모델.  


<학습 목표>

> "현재까지의 토큰이 주어졌을 때, 다음 토큰의 확률분포를 예측하라."

이것을 **causal language modeling**이라고 한다.

## 3. LLaMA의 기본 구조

LLaMA의 핵심 구조는 GPT와 매우 유사한 **decoder-only Transformer stack**.

전체 흐름은 보통 다음과 같다.

**입력 토큰 → 임베딩 → 여러 개의 Transformer decoder block → 최종 선형층 → 다음 토큰 확률**

조금 더 자세히 쓰면 다음과 같다.

### 3.1 Tokenization
문장은 먼저 tokenizer에 의해 작은 단위의 **토큰(token)** 으로 나뉜다.  
이 토큰은 단어 하나일 수도 있고, 단어의 일부(subword)일 수도 있다.

예를 들어,

- `"I like mathematics."`

라는 문장이 있다고 하면, 실제 모델 내부에서는 문자열 그대로 처리하지 않고  
정수 인덱스의 시퀀스로 변환하여 처리한다.

### 3.2 Embedding
토큰 인덱스는 곧바로 모델에 들어가지 않고, 각 토큰을 고차원 벡터로 바꾸는 **embedding layer**를 통과한다.

- 토큰 ID → 벡터 표현

으로 바뀜.

### 3.3 Transformer Decoder Blocks
LLaMA의 본체는 여러 개의 decoder block이 쌓인 구조.  
각 block 안에는 보통 다음 요소가 들어 있다.

- **Masked Self-Attention**
- **Feed-Forward Network(MLP)**
- **Residual Connection**
- **Normalization**

여기서 가장 중요한 것은 **masked self-attention**.

### 3.4 Output Projection
마지막 hidden state는 vocabulary 크기만큼의 점수(logits)로 변환.  
그 후 softmax를 취하면 다음 토큰에 대한 확률분포가 된다.

예를 들어 현재 문맥이

- `"The capital of France is"`

라면 모델은 다음 후보 토큰들에 대해 확률을 계산하고,  
그 중 하나를 선택하여 다음 토큰을 생성.

## 4. 왜 “decoder-only”가 중요한가

LLaMA는 입력 전체를 한 번에 이해하고 끝내는 모델이 아니라,  
**한 토큰씩 순차적으로 생성**하는 모델이다.

즉, 다음과 같은 식으로 동작한다.

1. 첫 문맥 입력
2. 다음 토큰 예측
3. 예측한 토큰을 다시 입력 뒤에 붙임
4. 다시 다음 토큰 예측
5. 반복

이 구조 때문에 LLaMA는 다음과 같은 특징을 가진다.

### 장점
- 자연스러운 텍스트 생성에 강함
- 대화형 시스템에 적합
- 코드 생성, 설명 생성, 요약, 질의응답 등 다양한 생성 작업 가능

### 한계
- 본질적으로는 “다음 토큰 예측기”이므로, 항상 깊은 이해를 보장하지는 않음
- 긴 문맥에서 계산량과 메모리 사용량이 커질 수 있음
- 사실과 다른 내용을 그럴듯하게 생성하는 **hallucination** 문제가 있을 수 있음


## 5. Self-Attention과 causal mask

LLaMA에서 핵심은 **self-attention**이다.

각 토큰은 현재 문장 안의 다른 토큰들과의 관련성을 계산하여, 어떤 토큰을 더 많이 참고할지 결정.

그런데 decoder-only 모델에서는 중요한 제약이 있다.

**미래 토큰을 보면 안 된다.**

예를 들어 아래 문장을 생성 중이라면,

- `The cat sat on the ____`

빈칸에 들어갈 다음 단어를 예측할 때  
모델은 아직 생성되지 않은 뒤쪽 정답을 보면 안 된다.

그래서 attention score를 계산할 때 **causal mask**를 씌운다.


- 현재 위치는 자기 자신과 왼쪽 토큰만 볼 수 있음
- 오른쪽(미래) 토큰은 가려짐

이것이 decoder-only GPT 계열과 encoder 계열의 가장 큰 차이 중 하나.


## 6. LLaMA의 학습 방식

LLaMA의 사전학습(pretraining)은 기본적으로 **대량의 텍스트 코퍼스**에 대해 다음 토큰 예측을 수행하는 방식.

학습 데이터가 다음과 같다고 하자.

- `A B C D`

모델은 다음과 같은 예측을 반복적으로 학습한다.

- `A -> B`
- `A B -> C`
- `A B C -> D`

정답 문장을 한 칸씩 오른쪽으로 밀어놓고 학습하는 형태.

수식으로 쓰면 보통 다음과 같다.

$$
\max_\theta \sum_t \log p_\theta(x_t \mid x_{<t})
$$

- $x_t$ : 현재 시점의 토큰
- $x_{<t}$ : 현재보다 앞에 있는 모든 토큰
- $\theta$ : 모델 파라미터

를 의미.

-> “앞 문맥이 주어졌을 때 현재 토큰이 나올 확률을 최대화하라”는 뜻.

## 7. LLaMA의 주요 특징

## 7.1 효율적인 공개 계열 기반 모델
초기 LLaMA는 **공개적으로 접근 가능한 데이터만으로도 경쟁력 있는 foundation model을 만들 수 있다**는 점을 강조. 이는 연구용 모델 생태계에 큰 영향을 줌.

## 7.2 Base 모델과 Chat 모델의 구분
Llama 2에서는 **pretrained base model**과 **fine-tuned chat model**이 함께 제시되.

- **Base model**: 다음 토큰 예측 중심의 일반 모델
- **Chat model**: 대화형 응답에 맞도록 추가 튜닝된 모델

같은 뿌리의 모델이라도 그냥 언어모델”과 “대화형 어시스턴트 모델”은 다름.

## 7.3 후속 파생 모델이 많음
LLaMA 계열은 코드 생성 특화 모델인 **Code Llama** 같은 파생 모델을 낳았다. Code Llama는 Llama 2 기반으로 만들어졌으며, 코드 완성, infilling, 프로그래밍 지시 수행 등에 특화되어 있다.

## 7.4 실습 친화성
LLaMA 계열은 연구와 교육에서 널리 사용됨.  
실제로 Meta의 공식 문서는 모델 접근, 배포, 프롬프트 형식, 모델 카드 등을 비교적 체계적으로 제공한다.

## 8. LLaMA 1, Llama 2, Llama 3를 어떻게 이해하면 좋은가

버전별 세부 수치보다 **흐름**을 이해하는 것이 중요.

### 8.1 LLaMA 1
- Meta가 공개한 초기 계열
- 7B ~ 65B 규모
- 효율적인 학습과 공개 데이터 기반 foundation model이라는 점에서 큰 반향을 일으킴 :

### 8.2 Llama 2
- pretrained 모델과 chat 모델을 함께 제시
- 7B ~ 70B 규모
- 대화형 사용을 위한 post-training이 더 강조됨

### 8.3 Llama 3 계열
- Meta의 공식 문서에서는 Llama 3 계열을 현대적인 배포 및 활용 흐름 속에서 안내하고 있으며, 모델 카드와 프롬프트 포맷 문서도 제공한다. 강의에서는 “LLaMA 계열이 단순한 연구 모델을 넘어 실제 활용 가능한 생태계로 확장되었다”는 흐름으로 이해하면 좋다.

## 9. GPT와 LLaMA의 관계

학생들이 자주 헷갈리는 부분은 **“LLaMA와 GPT는 완전히 다른 모델인가?”** 라는 점.


**기본 철학은 매우 유사하다.**

둘 다 보통 다음 특징을 공유한다.

- decoder-only Transformer
- causal mask 사용
- next-token prediction 학습
- autoregressive generation

큰 틀에서는 **GPT 계열 구조를 따르는 현대적 LLM**이라고 볼 수 있다.

다만 차이는 다음과 같은 부분에서 발생한다.

- 학습 데이터 구성
- tokenizer
- 파라미터 수
- 학습 안정화 기법
- fine-tuning 방식
- 프롬프트 포맷
- 공개 범위와 라이선스 정책

LLaMA를 이해하는 것은 사실상 “현대 decoder-only LLM을 이해하는 것”과 매우 가까움.

## 10. Base model과 Instruction-tuned model의 차이

같은 LLaMA 계열이라도 모델은 크게 두 종류로 나눌 수 있다.

### 10.1 Base model
사전학습만 된 모델이다. 기본적으로는 다음 토큰을 이어 쓰는 능력에 집중되어 있다.

예를 들어,

- `"Explain attention mechanism."`

이라고 넣으면 설명을 해줄 수도 있지만, 항상 사용자의 의도에 잘 맞는 답을 주는 것은 아니다.

### 10.2 Instruction-tuned / Chat model
사전학습 후에 추가로  
**instruction-following 데이터**, **대화 데이터**, **선호 정렬 과정** 등을 거쳐 사용자의 질문에 더 잘 반응하도록 만든 모델이다.  
Llama 2에서는 이런 대화형 모델을 **Llama 2-Chat**으로 제시했다.

- Base model: 언어를 잘 이어 쓰는 모델
- Chat model: 사람의 지시에 더 잘 따르도록 다듬어진 모델



## 11. LLaMA의 한계

LLaMA가 강력한 모델이라고 해서 완전한 것은 아니다.

### 11.1 Hallucination
그럴듯하지만 틀린 내용을 생성할 수 있다.

### 11.2 긴 문맥 비용
Transformer attention 구조상 문맥 길이가 길어질수록 계산량과 메모리 사용량이 커진다.

### 11.3 추론 능력의 불안정성
복잡한 수학 문제나 긴 논리 추론에서는 정답률이 안정적이지 않을 수 있다.

### 11.4 모델과 시스템은 다르다
LLaMA는 “기본 모델”이고, 실제 서비스 품질은 여기에  프롬프트 설계, 검색(RAG), 툴 사용, 안전장치 등을 추가한 **시스템 설계**에 크게 좌우됨.

# LLaMA 수식 정리

## 1. 전체 개요

LLaMA는 기본적으로 **decoder-only Transformer** 구조를 따른다.  
즉, 입력 토큰 시퀀스가 들어오면 각 층에서 다음 과정을 반복한다.

1. 토큰 임베딩
2. 위치 정보 반영
3. masked self-attention
4. feed-forward network
5. residual connection
6. 최종적으로 다음 토큰 확률 계산

수식으로 보면 LLaMA의 핵심은 다음 네 부분으로 나눌 수 있다.

- 입력 표현
- self-attention
- feed-forward network
- autoregressive language modeling objective

---

## 2. 입력 토큰과 임베딩

입력 시퀀스를

$$
x = (x_1, x_2, \dots, x_T)
$$

라고 하자.여기서 $x_t$는 $t$번째 토큰이다.

각 토큰은 임베딩 행렬 $E \in \mathbb{R}^{V \times d}$를 통해 $d$차원 벡터로 변환된다.

$$
h_t^{(0)} = E[x_t]
$$

- $V$: vocabulary 크기
- $d$: hidden dimension
- $h_t^{(0)}$: 첫 입력 임베딩

전체 입력은

$$
H^{(0)} = (h_1^{(0)}, h_2^{(0)}, \dots, h_T^{(0)})
$$
---

## 3. 위치 정보: RoPE (Rotary Positional Embedding)

LLaMA는 전통적인 absolute positional embedding 대신 **RoPE**를 사용.

RoPE는 query와 key에 회전 변환을 주어 위치 정보를 반영.

2차원 성분 쌍 $(u_{2i}, u_{2i+1})$에 대해, 위치 $m$에서의 회전은 다음과 같이 쓸 수 있다.

$$
\mathrm{RoPE}(u, m) =
\begin{pmatrix}
u_{2i}\cos\theta_{m,i} - u_{2i+1}\sin\theta_{m,i} \\
u_{2i}\sin\theta_{m,i} + u_{2i+1}\cos\theta_{m,i}
\end{pmatrix}
$$

여기서 각 차원에 대한 각도는 보통

$$
\theta_{m,i} = m \cdot \omega_i
$$

$$
\omega_i = 10000^{-2i/d}
$$

형태로 둠.

위치 $m$에 따라 query와 key를 회전시켜  attention score에 상대적 위치 정보가 자연스럽게 들어가게 만든다.

---

## 4. Self-Attention

각 layer에서 입력 $H \in \mathbb{R}^{T \times d}$가 주어졌을 때,  
query, key, value는 다음과 같이 계산됨.

$$
Q = HW_Q,\quad K = HW_K,\quad V = HW_V
$$

여기서

- $W_Q \in \mathbb{R}^{d \times d_k}$
- $W_K \in \mathbb{R}^{d \times d_k}$
- $W_V \in \mathbb{R}^{d \times d_v}$

LLaMA에서는 RoPE를 query와 key에 적용한다.

$$
\widetilde{Q} = \mathrm{RoPE}(Q), \quad \widetilde{K} = \mathrm{RoPE}(K)
$$

그 다음 attention score는

$$
S = \frac{\widetilde{Q}\widetilde{K}^\top}{\sqrt{d_k}}
$$

로 계산.

하지만 decoder-only 모델이므로 미래 토큰을 보면 안 된다.  

그래서 **causal mask** $M$를 적용.

$$
M_{ij} =
\begin{cases}
0, & j \le i \\
-\infty, & j > i
\end{cases}
$$

따라서 masked attention은

$$
A = \mathrm{softmax}(S + M)
$$

이고, 최종 attention 출력은

$$
\mathrm{Attention}(Q,K,V) = AV
$$

---

## 5. Multi-Head Attention

attention head가 $h$개라면, 각 head에서

$$
\text{head}_r = \mathrm{Attention}(Q_r, K_r, V_r), \quad r=1,\dots,h
$$

를 계산한 뒤 이어붙임.

$$
\mathrm{MHA}(H) = \mathrm{Concat}(\text{head}_1, \dots, \text{head}_h)W_O
$$

여기서 $W_O$는 출력 projection 행렬이다.

---

## 6. Residual Connection과 Pre-Norm 구조

LLaMA는 **pre-normalization** 구조를 사용.

attention과 FFN 앞에 normalization이 들어간다.

입력 $H^{(\ell)}$에 대해 attention block 출력은 다음과 같다.

$$
\hat{H}^{(\ell)} = H^{(\ell)} + \mathrm{MHA}(\mathrm{Norm}(H^{(\ell)}))
$$

그 다음 FFN block은

$$
H^{(\ell+1)} = \hat{H}^{(\ell)} + \mathrm{FFN}(\mathrm{Norm}(\hat{H}^{(\ell)}))
$$
---

## 7. RMSNorm

LLaMA는 LayerNorm 대신 **RMSNorm**을 사용한다.

벡터 $x \in \mathbb{R}^d$에 대해 RMSNorm은

$$
\mathrm{RMSNorm}(x) = \gamma \odot \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d}x_i^2 + \epsilon}}
$$

로 정의.

- $\gamma$: 학습 가능한 scale 파라미터
- $\epsilon$: 수치적 안정성을 위한 작은 값
- $\odot$: 원소별 곱



LayerNorm과 달리 평균을 빼지 않고, 크기만 정규화한다는 점이 특징.

---

## 8. Feed-Forward Network: SwiGLU

LLaMA의 FFN은 일반적인 ReLU FFN 대신 **SwiGLU** 계열을 사용.

입력 $x$에 대해 보통 다음과 같이 쓴다.

$$
\mathrm{FFN}(x) = W_2 \Big( \mathrm{SiLU}(W_1 x) \odot (W_3 x) \Big)
$$


- $W_1, W_3$: 입력을 확장하는 두 선형 변환
- $W_2$: 다시 hidden dimension으로 축소하는 선형 변환
- $\mathrm{SiLU}(z) = z \sigma(z)$


SiLU 함수는 다음과 같다.

$$
\mathrm{SiLU}(z) = z \cdot \frac{1}{1+e^{-z}}
$$

SwiGLU는 한쪽 가지는 gate처럼 작동하고, 다른 한쪽은 실제 정보 경로 역할을 하여 표현력을 높임.

---

## 9. 한 개 Transformer block의 수식 요약

LLaMA의 $\ell$번째 block을 하나로 정리하면 다음과 같다.

### Step 1. Attention sublayer
$$
Z^{(\ell)} = H^{(\ell)} + \mathrm{MHA}(\mathrm{RMSNorm}(H^{(\ell)}))
$$

### Step 2. Feed-forward sublayer
$$
H^{(\ell+1)} = Z^{(\ell)} + \mathrm{FFN}(\mathrm{RMSNorm}(Z^{(\ell)}))
$$

이 과정을 여러 층 반복.

---

## 10. 최종 출력과 logits

마지막 layer의 출력을 $H^{(L)}$라고 하자.  

보통 마지막에도 normalization을 거친다.

$$
\bar{H} = \mathrm{RMSNorm}(H^{(L)})
$$

그 다음 각 위치 $t$의 hidden state $\bar{h}_t$를 vocabulary 차원으로 사상하여 logits를 계산.

$$
o_t = W_{\text{lm}} \bar{h}_t
$$


- $W_{\text{lm}} \in \mathbb{R}^{V \times d}$

이후 softmax를 취하면 다음 토큰의 확률분포가 된다.

$$
p(x_{t+1}\mid x_{\le t}) = \mathrm{softmax}(o_t)
$$

---

## 11. 학습 목표: Autoregressive Language Modeling

LLaMA는 **다음 토큰 예측**으로 학습.  
전체 시퀀스 $x_1, x_2, \dots, x_T$에 대한 확률은 다음처럼 factorization 된다.

$$
p(x_1, x_2, \dots, x_T) = \prod_{t=1}^{T} p(x_t \mid x_{<t})
$$

따라서 로그우도 최대화는

$$
\max_\theta \sum_{t=1}^{T} \log p_\theta(x_t \mid x_{<t})
$$

가 된다.

실제로는 이를 **cross-entropy loss** 최소화 형태로 쓴다.

$$
\mathcal{L} = - \sum_{t=1}^{T} \log p_\theta(x_t \mid x_{<t})
$$

배치 단위로 평균을 내면 보통

$$
\mathcal{L}_{\text{CE}} = -\frac{1}{T}\sum_{t=1}^{T} \log p_\theta(x_t \mid x_{<t})
$$

형태로 사용.

---

## 12. 추론 시 토큰 생성

학습이 끝난 뒤에는 다음 토큰을 반복적으로 생성.

현재까지의 문맥이 $x_{1:t}$일 때,

$$
p(x_{t+1} \mid x_{1:t})
$$

를 계산하고, 그 분포에서 다음 토큰을 선택.

가장 단순한 방식은 greedy decoding.

$$
x_{t+1} = \arg\max_{v \in \mathcal{V}} p(v \mid x_{1:t})
$$

샘플링 방식에서는

$$
x_{t+1} \sim p(\cdot \mid x_{1:t})
$$

처럼 확률적으로 뽑는다.

temperature를 사용하면 분포를 다음처럼 조정한다.

$$
p_i = \frac{\exp(o_i/\tau)}{\sum_j \exp(o_j/\tau)}
$$

여기서 $\tau$는 temperature이다.

- $\tau < 1$: 더 날카로운 분포
- $\tau > 1$: 더 부드러운 분포

---

## 13. LLaMA 핵심 수식 한 번에 보기

LLaMA를 가장 압축해서 쓰면 다음 순서로 정리할 수 있다.

### 입력 임베딩
$$
h_t^{(0)} = E[x_t]
$$

### Query, Key, Value
$$
Q = HW_Q,\quad K = HW_K,\quad V = HW_V
$$

### RoPE 적용
$$
\widetilde{Q} = \mathrm{RoPE}(Q), \quad \widetilde{K} = \mathrm{RoPE}(K)
$$

### Masked Self-Attention
$$
A = \mathrm{softmax}\left(\frac{\widetilde{Q}\widetilde{K}^\top}{\sqrt{d_k}} + M\right)
$$

$$
\mathrm{Attention}(Q,K,V)=AV
$$

### Residual + RMSNorm + FFN
$$
Z^{(\ell)} = H^{(\ell)} + \mathrm{MHA}(\mathrm{RMSNorm}(H^{(\ell)}))
$$

$$
H^{(\ell+1)} = Z^{(\ell)} + W_2\Big(\mathrm{SiLU}(W_1 \mathrm{RMSNorm}(Z^{(\ell)})) \odot (W_3 \mathrm{RMSNorm}(Z^{(\ell)}))\Big)
$$

### 다음 토큰 확률
$$
p(x_{t+1}\mid x_{\le t}) = \mathrm{softmax}(W_{\text{lm}}\bar{h}_t)
$$

### 학습 loss
$$
\mathcal{L} = - \sum_{t=1}^{T} \log p_\theta(x_t \mid x_{<t})
$$

---

In [ ]:
#디바이스 설정 / LLaMA 계열은 모델 크기가 있으므로 GPU가 있으면 훨씬 편함
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("사용 디바이스:", device)

사용 디바이스: cuda


In [ ]:
from huggingface_hub import login
login(token= "HF_TOKEN")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 모델 이름 정하기
model_id = "meta-llama/Llama-3.2-1B-Instruct"
print(model_id)

# 일부 LLaMA 모델은 Hugging Face에서 접근 권한 승인이 필요할 수 있다.
# 접근이 되지 않으면 Hugging Face 로그인 또는 권한 승인 상태를 확인해야 한다.

meta-llama/Llama-3.2-1B-Instruct


In [ ]:
import torch
from transformers import pipeline

model_id = "meta-llama/Llama-3.2-1B-Instruct"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto"
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain what a large language model is in simple terms."}
]

output = pipe(
    messages,
    max_new_tokens=150
)

print(output[0]["generated_text"][-1]["content"])

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct.
403 Client Error. (Request ID: Root=1-69d1f0f5-3770d64443b4a4e251bcd8a1;8caed22f-af77-4605-8152-039c656ea816)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-1B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct to ask for access.

- 접근 권한이 없어서 접속이 안됨.
    - Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct.
    - exapnd to review and access
    - 시간이 좀 걸림

In [ ]:
from huggingface_hub import logout, notebook_login

try:
    logout()
except:
    pass

notebook_login()

In [ ]:
# 로그인 상태에서 다시 접근
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="meta-llama/Llama-3.2-1B-Instruct",
    filename="config.json"
)
print("접근 성공")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

접근 성공


In [ ]:
import torch
from transformers import pipeline

model_id = "meta-llama/Llama-3.2-1B-Instruct"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype="auto",
    device_map="auto"
)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain what a large language model is in simple terms."}
]

output = pipe(
    messages,
    max_new_tokens=150
)

print(output[0]["generated_text"][-1]["content"])

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A large language model is a type of computer program that can understand, generate, and even learn human language. It's like a super-smart, virtual assistant that can read, write, and talk to humans in a way that's similar to how a human would.

Imagine you have a super-powerful, language- learning robot that can understand and respond to any question or topic you ask it. That's basically what a large language model is.

These models are trained on vast amounts of text data, which allows them to learn patterns, relationships, and structures of language. They can then use this knowledge to generate text, answer questions, and even create new content.

Large language models are typically trained on massive datasets, such as books, articles,


In [ ]:
!pip -q install -U transformers accelerate huggingface_hub sentencepiece

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

messages = [
    {"role": "system", "content": "You are a helpful tutor."},
    {"role": "user", "content": "선형대수에서 고유값을 쉬운 예시와 함께 설명해줘."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=True,
        temperature=0.6,
        top_p=0.9
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 77.3 MB/s eta 0:00:00


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


선형대수에서 고유값을 쉬운 예시를 제공해 드리겠습니다.

고유값은 선형대수에서, 선형方程式의 일원으로서의 특정한 점에 대한 해를 나타내는 것입니다. 선형대수에서 고유값은 다음과 같은 예시로 제공됩니다.

1.  2x + 3 = 0: x = -3/2
2.  x + 2 = 0: x = -2
3.  2x - 3 = 0: x = 3/2
4.  x - 2 = 0: x = 2
5.  x + 1 = 0: x = -1

위의 예시에서, x = -3/2, x = -2, x = 3/2, x = 2, x = -1은 선형대수에서 고유값입니다.


- 기준 코드

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

def chat_with_llama(messages, max_new_tokens=180, temperature=0.7, top_p=0.9):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
# 실습 1: 가장 기본적인 한턴 실습
messages = [
    {"role": "system", "content": "You are a clear and friendly tutor."},
    {"role": "user", "content": "Explain optimization with one simple example."}
]

response = chat_with_llama(messages)
print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Optimization is a fundamental concept in mathematics and computer science that involves finding the best possible solution among a set of possible solutions. Think of it like trying to find the most efficient way to get from point A to point B.

Imagine you're planning a road trip from New York to Los Angeles. You have a few options to consider:

1. Take the most direct route, which is primarily along Interstate 80.
2. Take a route that goes through the Midwest, which might be longer but might also be more scenic.
3. Take a route that goes through the Rocky Mountains, which might be more fun, but could also be more expensive.

In this example, optimization is about finding the best possible solution among these three options. You want to minimize the total time and cost of your trip.

In the context of optimization, you can think of it as:

* The "optimal" solution


- messages가 단순 문자열이 아니라, role과 content를 가진 대화 기록

In [ ]:
# 실습 2: 실습 2: system prompt의 힘
messages = [
    {"role": "system", "content": "You are a patient tutor for beginners."},
    {"role": "user", "content": "Explain gradient descent."}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Gradient Descent is a fundamental concept in machine learning and optimization. I'd be happy to explain it in detail.

**What is Gradient Descent?**

Gradient Descent is an optimization algorithm used to find the minimum value of a function. It's a popular method for training neural networks, linear regression, and other types of machine learning models.

**How does Gradient Descent work?**

Imagine you're trying to find the shortest path between two points in a 2D or 3D space. You have a grid of points, and you want to find the point that is closest to the starting point. Gradient Descent is like a guide that helps you find the shortest path by iteratively moving in the direction of the steepest gradient.

Here's a step-by-step explanation:

1. **Initialize the parameters**: We start with an initial set of parameters, such as the weights and


In [ ]:
# 버전 C: 아주 짧게
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "Explain gradient descent in 3 sentences."}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Gradient descent is an optimization algorithm used to minimize the loss function of a model by iteratively adjusting its parameters in the direction of the negative gradient of the loss function. This process is repeated for multiple iterations, with the model's parameters being updated based on the average rate of convergence, which helps to reduce overfitting and improve model generalization. By minimizing the loss function, gradient descent aims to find the optimal set of parameters that best fit the training data and maximize the model's predictive accuracy.


In [ ]:
# Llama 3.2 1B Instruct는 multilingual dialogue 용도로 소개되어 있으므로, 한국어도 시험해볼 가치가 있다.

messages = [
    {"role": "system", "content": "You are a helpful tutor."},
    {"role": "user", "content": "경사하강법을 아주 쉬운 예시로 설명해줘."}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


경사하강법은 기차가 보행하는 경로를 설명하는 데 사용되는 전시법입니다. 이 법은 기차가 보행하는 경로를 여러 개의 개별 경로로 나눔으로써, 기차가 보행하는 경로를 쉽게 이해할 수 있도록 도와줍니다.

경사하강법의 예시로, 보통의 보행 경로를 보라색으로 표시하고, 기차가 보행하는 경로를 파란색으로 표시합니다. 보통의 보행 경로를 보라색으로 표시하면, 보통의 보행 경로의 길이가 보통의 보행 경로의 길이와 마찬가지로 100m가 되며, 보통의 보행 경로의 길이가 보통의 보행 경로의 길이


In [ ]:
## 영어 질문
messages = [
    {"role": "system", "content": "You are a helpful tutor."},
    {"role": "user", "content": "Explain gradient descent with a very simple example."}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Gradient Descent is a fundamental concept in machine learning that helps minimize the error between the model's predictions and the actual data. I'd be happy to explain it with a simple example.

**What is Gradient Descent?**

Imagine you're trying to learn the relationship between two variables, say "height" and "weight". You have a set of data points, like (5, 10), (6, 12), and (7, 14). You want to find the best "weight" for a person given their height.

**The Goal**

The goal is to find the optimal "weight" that minimizes the error between the predicted height and the actual height. In other words, you want to find the "weight" that best fits the data.

**The Method**

Gradient Descent is a step-by-step process that works as follows:

1. Start with an initial guess


In [ ]:
# 생성 옵션 바꾸기
## 낮은 temperature
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain what overfitting is."}
]

print(chat_with_llama(messages, temperature=0.2, top_p=0.9))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Overfitting is a common problem in machine learning, where a model is too complex and fits the training data too closely, but fails to generalize well to new, unseen data.

In simple terms, overfitting occurs when a model is trained on a large dataset that contains a lot of noise, but the model is so complex that it starts to memorize the noise rather than the underlying patterns in the data. This means that the model becomes too specialized to the training data and fails to make accurate predictions on new, unseen data.

Overfitting can happen in several ways:

1. **Too many parameters**: If a model has too many parameters, it may be too complex and start to fit the noise in the training data rather than the underlying patterns.
2. **Insufficient training data**: If the training data is too small or too noisy, the model may not have enough information to learn


In [ ]:
# 생성 옵션 바꾸기
## 높은 temperature
print(chat_with_llama(messages, temperature=1.0, top_p=0.9))

#temperature가 낮으면 더 보수적
# 높으면 더 다양하지만 산만해질 수 있음

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Overfitting is a common problem in machine learning that occurs when a model is too complex and fits the training data too closely, but fails to generalize well to new, unseen data.

In simpler terms, overfitting happens when a model is trained on a large dataset that contains a lot of noise and irrelevant features, and it starts to learn the patterns and structures in the training data rather than the underlying relationship between the input and output variables.

Here's an example to illustrate overfitting:

Let's say you have a model that's designed to predict house prices based on features like the number of bedrooms, square footage, and location. During training, the model is trained on a dataset of houses that includes a lot of noise and irrelevant features, such as the color of the roof or the type of flooring.

As a result, the model becomes overly specialized to the training data and starts to produce


In [ ]:
# 더 길게 생성
print(chat_with_llama(messages, max_new_tokens=250, temperature=0.6))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Overfitting is a common problem in machine learning models, particularly when the model is trained on a specific dataset or has a complex architecture.

In simple terms, overfitting occurs when a model is too closely fit to the training data, but fails to generalize well to new, unseen data. This means that the model is overly specialized to the specific patterns and relationships in the training data, and is unable to make accurate predictions on new, similar data.

There are several reasons why overfitting can occur:

1. **Insufficient training data**: If the model is trained on a small or low-quality dataset, it may learn to fit the noise in the data rather than the underlying patterns.
2. **Complex model architecture**: Models with many layers or complex relationships between inputs and outputs can be prone to overfitting.
3. **Large model size**: Larger models can be more prone to overfitting, as they have more parameters to fit.
4. **Overfitting to noise**: If the training data c

In [ ]:
# min-turn 대화
messages = [
    {"role": "system", "content": "You are a patient math tutor."},
    {"role": "user", "content": "What is overfitting?"},
    {"role": "assistant", "content": "Overfitting happens when a model learns the training data too closely and fails to generalize well."},
    {"role": "user", "content": "Then how is it different from underfitting?"}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Underfitting occurs when a model learns the training data too loosely and fails to capture the underlying patterns. This can result in the model not being able to make accurate predictions on new, unseen data.

In contrast, overfitting is when a model is too complex and learns the noise in the training data rather than the underlying structure. This can lead to poor performance on new data.

Think of it like a car that's too small to handle a big road. It may be able to handle the small road, but it will struggle to handle the big road. That's similar to overfitting - the model is too complex and can't generalize well to new data.

To illustrate the difference, consider a simple example. Let's say you have a model that's designed to predict the number of hours a person works per week. A simple model might learn that people who work 40 hours per


In [ ]:
# 원하는 형식 받기
## bullet points

messages = [
    {"role": "system", "content": "You are a clear tutor."},
    {"role": "user", "content": "Explain eigenvalues in exactly 4 bullet points."}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Here are four key points explaining eigenvalues in simple terms:

• **Eigenvalues are a measure of how much a linear transformation stretches or shrinks a vector**: In other words, they tell you how much a transformation will change the size of a vector. For example, if you have a vector and you apply a transformation that scales it by a certain factor, the resulting vector will be scaled by that same factor.

• **The eigenvectors are the directions in which the transformation stretches or shrinks the vector**: Think of eigenvectors like the directions in which a stretch or compression happens. The eigenvectors are the directions that the transformation will keep unchanged, while the eigenvectors of zero will be changed by the transformation.

• **Eigenvalues are scalar values that represent how much the transformation stretches or shrinks the vector**: Think of eigenvalues as the "stretchiness" or "squ


In [ ]:
# 표 스타일 지정
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Compare overfitting and underfitting in a 3-row table."}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In a 3-row table, we can compare overfitting and underfitting using a simple example. Let's say we have a model that is trying to predict the number of people in a household based on their income.

| Income (rows) | Number of People (columns) | Predicted Number of People |
| --- | --- | --- |
| 10000 | 1 | 10000 |
| 8000 | 2 | 8000 |
| 12000 | 3 | 12000 |

In this example, we have a 3-row table with 1, 2, and 3 columns, representing the income, number of people, and predicted number of people, respectively.

**Overfitting:**

Imagine that the model is too complex and fits the noise in the data perfectly. In this case, the model is overfitting.




In [ ]:
# 단계별 설정
messages = [
    {"role": "system", "content": "You are a programming tutor."},
    {"role": "user", "content": "Explain how gradient descent works step by step for a beginner."}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


I'd be happy to explain how gradient descent works step by step in a way that's easy to understand.

**What is Gradient Descent?**

Gradient descent is an algorithm used to minimize the loss function in machine learning models. It's a simple yet powerful technique that's widely used in many applications, including linear regression, logistic regression, and neural networks.

**Step 1: Define the Loss Function**

The loss function is a measure of how well the model fits the data. It's usually a quadratic function that calculates the difference between the predicted output and the actual output. The loss function is typically defined as:

L(y, y_pred) = (y - y_pred)^2

where y is the actual output, y_pred is the predicted output, and L is the loss.

**Step 2: Initialize the Model Parameters**

We need to initialize the model parameters, also known as


In [ ]:
# 수학/코딩 조교처럼 사용하기
## 수학 조교
messages = [
    {"role": "system", "content": "You are a mathematics teaching assistant. Explain clearly with intuition and formulas."},
    {"role": "user", "content": "What is an eigenvector?"}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


An eigenvector is a non-zero vector that, when a linear transformation is applied to it, results in a scaled version of the same vector. In other words, it's a vector that changes its direction when it's transformed by a linear operator.

Mathematically, if we have a linear transformation T, a vector v, and a scalar k, then the eigenvector v is said to be an eigenvector of T with eigenvalue k if:

T(v) = kv

In other words, when we apply the linear transformation T to the vector v, it scales v by a factor of k, and v itself becomes a multiple of k.

The eigenvalue k is a scalar that represents the amount of scaling. The eigenvector v is a non-zero vector that corresponds to this eigenvalue.

Here's a simple example to illustrate this concept. Let's say we have


In [ ]:
# 코딩 조교
messages = [
    {"role": "system", "content": "You are a Python tutor. Explain code line by line."},
    {"role": "user", "content": "Write Python code for gradient descent on f(x)=x^2 and explain each line."}
]

print(chat_with_llama(messages, max_new_tokens=300))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Here's an example of gradient descent implemented in Python on the function `f(x) = x^2`. We'll use the `numpy` library for numerical operations and `matplotlib` for visualization.

```python
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt

# Define the function to be minimized (in this case, f(x) = x^2)
def f(x):
    return x**2

# Define the derivative of the function (first derivative)
def f_prime(x):
    return 2*x

# Set the learning rate (value of alpha in gradient descent)
alpha = 0.1

# Set the maximum number of iterations (number of epochs)
max_iterations = 1000

# Initialize the initial guess for x (randomly choose a value between -100 and 100)
x0 = np.random.uniform(-100, 100)

# Initialize the lists to store the x values and the corresponding y values
x_values = []
y_values = []

# Perform gradient descent
for _ in range(max_iterations):
    # Compute the predicted y value using the current x value
    predicted_y = f(x0)

    # Compute the e

In [ ]:
# 같은 질문 여러번 생성하기
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Give me a simple analogy for neural networks."}
]

for i in range(3):
    print(f"\n--- Response {i+1} ---")
    print(chat_with_llama(messages, temperature=0.8, top_p=0.9))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



--- Response 1 ---


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


A simple analogy for neural networks is like a brain.

Imagine a brain has thousands of neurons, each with thousands of synapses. Each neuron has a unique "input" (e.g., seeing a red apple), and each synapse has a unique "weight" (e.g., how strongly it connects the neuron to another neuron).

When a neuron receives input, it can either fire (or "fire" or "activate") or not fire. The strength of the signal it receives determines how strongly it will fire. The synapses between neurons can adjust their strength based on the signals they receive, allowing the neurons to learn and change over time.

Just like how the brain's neurons and synapses interact and adapt to learn and remember, a neural network is a computer system that uses these principles to learn and improve its performance on a task, much like how the brain learns and remembers things.

This

--- Response 2 ---


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


A simple analogy for neural networks is comparing them to a group of highly skilled musicians.

Imagine a neural network as a team of highly skilled musicians working together to create a beautiful piece of music. Each musician represents a layer of the neural network, and they all work together to create a cohesive and harmonious sound.

* The musicians start with a simple melody (input layer), and as they play, they add more notes and layers to create a richer sound (hidden layers).
* As the musicians practice and improve, they develop a deeper understanding of the music and can play more complex harmonies (feature extraction).
* The musicians also learn to work together more effectively, creating a sense of unity and cohesion (fusion).
* Finally, the musicians refine their performance, adjusting the volume, tone, and rhythm to create a polished and engaging sound (modeling).

In this analogy, the input layer is like the

--- Response 3 ---
Imagine a computer as a skilled musician. T

In [ ]:
# Hugging Face의 pipeline은 채팅 입력을 받으면 모델의 chat template를 사용해 처리

from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

messages = [
    {"role": "system", "content": "You are a helpful tutor."},
    {"role": "user", "content": "Explain optimization with one simple example."}
]

out = pipe(messages, max_new_tokens=150)
print(out[0]["generated_text"][-1]["content"])

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Optimization is the process of finding the best possible solution among a set of possible solutions, given certain constraints or requirements. Here's a simple example to illustrate this concept:

**Example: Finding the shortest path in a city**

Imagine you're a taxi driver, and you need to transport a passenger from the airport to their destination in the city. You have a set of roads and traffic lights that you can use to get there. The goal is to find the shortest possible path from the airport to the passenger's destination.

**Constraints:**

* The airport is located at the starting point.
* The passenger's destination is located at the end point.
* You can only travel on certain roads and at certain times.
* There are traffic lights at certain


- pipeline은 더 간단함
- tokenizer + model 직접 사용은 더 세밀하게 제어 가능함


In [ ]:
# 나만의 함수 확장하기
chat_history = [
    {"role": "system", "content": "You are a patient tutor."}
]

while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit"]:
        break

    chat_history.append({"role": "user", "content": user_input})
    answer = chat_with_llama(chat_history, max_new_tokens=200, temperature=0.7)
    print("Llama:", answer)

    chat_history.append({"role": "assistant", "content": answer})

You: 경사하강법이 뭐야?


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Llama: 경사하강법은 도로의 경사 방향을 계산하는 방법 중 하나입니다. 이 방법은 도로의 기울imi를 계산하여 경사하강을 할 수 있습니다.

경사하강법은 도로의 기울imi를 계산하기 위해 도로의 지면과 지하수eways 등에 대한 정보를 आवश्यकत에 따라 조합하여 계산합니다. 이 방법은 도로를 정상적으로 운전할 때 사용할 수 있는 방법이며, 도로의 경사 방향을 정확하게 determining करन에 도움이 됩니다.

경사하강법은 도로의 지면과 지하수eways 등에 대한 정보를 사용하여 계산하는 방식은 다음과 같습니다.

1. 도로의 지면: 도로의 지면을 사용하여 계산합니다. 도로의 지면은 도로의 기울imi와 지하수eways 등에 대한 정보를
You: 


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Llama: 1. 도로의 지면: 도로의 지면을 사용하여 계산합니다. 도로의 지면은 도로의 기울imi와 지하수eways 등에 대한 정보를 사용하여 계산합니다.

2. 도로의 지하수eways: 도로의 지하수eways에 대한 정보를 사용하여 계산합니다. 도로의 지하수eways는 도로의 기울imi와 지하수eways 등에 대한 정보를 사용하여 계산합니다.

3. 도로의 지상 수준: 도로의 지상 수준을 사용하여 계산합니다. 도로의 지상 수준은 도로의 기울imi와 지하수eways 등에 대한 정보를 사용하여 계산합니다.

4. 도로의 기울imi: 도로의 기울imi를 사용하여 계산합니다. 도로의 기울imi는 도로의 기울imi와 지하수eways 등


KeyboardInterrupt: Interrupted by user

In [ ]:
messages = [
    {"role": "system", "content": "너는 친절하고 명확한 수학 튜터야."},
    {"role": "user", "content": "최적화를 아주 쉬운 예시 하나로 설명해줘."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.6,
        top_p=0.9
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


최적화를 설명해 드릴게요. 최적화를 할 때, 주로 문제의 원리는 다음과 같습니다.

*   최적화는 최소한의 최적화가 enough가?
*   최적화는 주어진 조건하에 최적화가 enough가?

예를 들어, 다음은 최적화를 할 수 있는 간단한 예시입니다.

**예시 1: 최적화하기**

*   주어진 문제: 2x + 3y = 7
*   최적화: x와 y가 최소한으로도 enough가?

*   최적화 방법: x와 y를 다수 사용하여 최적화하는 방법을 사용합니다.
*   최적화 결과: x = 2, y = 1

**예시 2: 최적화하기**

*   주어진 문제: 2x + 3y = 7
*   최적화


- Llama로 few-shot prompting 실습

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

def chat_with_llama(messages, max_new_tokens=200, temperature=0.6, top_p=0.9):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

- 입력: Our method demonstrates robust performance under noisy conditions.
- 출력: Our method works well even when the data is noisy.
- 입력: The algorithm exhibits superior reconstruction accuracy.
- 출력: The algorithm reconstructs the image more accurately.
    - 이런 식으로 몇 개 보여준 뒤, 새 문장을 넣으면 모델이 같은 패턴을 따라감.

In [ ]:
# 영어 문장을 좀 더 쉽게 / Zero-shot
messages = [
    {"role": "system", "content": "You are a helpful academic English assistant."},
    {"role": "user", "content": "Rewrite this in simpler English: Our method improves reconstruction quality under sparse-view conditions."}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Our method can help improve how well we can reconstruct an image when some of the information is missing or not available.


In [ ]:
# few-shot
messages = [
    {"role": "system", "content": "You are a helpful academic English assistant."},
    {"role": "user", "content": """Rewrite each sentence in simpler academic English.

Example 1
Input: Our method demonstrates robust performance under noisy conditions.
Output: Our method works well even when the data is noisy.

Example 2
Input: The algorithm exhibits superior reconstruction accuracy.
Output: The algorithm reconstructs the image more accurately.

Example 3
Input: This framework facilitates efficient parameter estimation.
Output: This framework helps estimate parameters efficiently.

Now do the same.

Input: Our method improves reconstruction quality under sparse-view conditions.
Output:"""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Here are the rewritten sentences in simpler academic English:

Example 1
Input: Our method shows good performance under noisy conditions.
Output: Our method works well even when the data is noisy.

Example 2
Input: The algorithm achieves higher accuracy in image reconstruction.
Output: The algorithm reconstructs the image more accurately.

Example 3
Input: This framework enables efficient parameter estimation.
Output: This framework helps estimate parameters efficiently.

Let me know if you have any further requests.


In [ ]:
# 용어 설명 형식 고정하기 / zero shot
## 한줄 정의
## 쉬운 예시 1개
## 핵심 포인트 1줄
messages = [
    {"role": "system", "content": "You are a math tutor."},
    {"role": "user", "content": "Explain eigenvalue."}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In mathematics, an eigenvalue is a scalar that represents how much a linear transformation changes a vector. It's a fundamental concept in linear algebra and has numerous applications in physics, engineering, and other fields.

Let's consider a linear transformation T that maps a vector x to a new vector Tx. The transformation can be represented by a matrix A, which is a square matrix that describes the transformation. The eigenvalues of A are the scalar values that represent how much the transformation changes the original vector x.

Mathematically, an eigenvalue λ and a corresponding eigenvector v are defined as:

1. λv = Av (the transformation changes the vector v by a scalar multiple of λ)
2. v is not the zero vector (v ≠ 0)

In other words, the transformation scales the eigenvector v by a factor of λ. The eigenvector v is unique for each eigenvalue λ, and it represents the direction in which the transformation stretches or shrinks the original vector


In [ ]:
# few-shot
messages = [
    {"role": "system", "content": "You are a math tutor."},
    {"role": "user", "content": """Explain each concept in the following format:
Definition:
Example:
Key Point:

Example 1
Concept: Gradient descent
Definition: A method that updates parameters step by step to reduce error.
Example: If you are walking downhill to find the lowest point, each step is like one update.
Key Point: It tries to minimize a loss function.

Example 2
Concept: Overfitting
Definition: When a model learns the training data too closely and performs poorly on new data.
Example: A student memorizes only past exam questions and cannot solve new ones.
Key Point: Good training performance does not always mean good generalization.

Now explain this concept.

Concept: Eigenvalue
Definition:
Example:
Key Point:"""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


I can explain each concept in the format you requested.

**Gradient Descent**

Definition:
A method that updates parameters step by step to reduce error.
Example:
Imagine you are trying to find the shortest path between two points on a map. You start with an initial guess and repeatedly move in the direction that reduces the distance between you and your target. This process continues until you reach the target, and that's essentially what gradient descent does - it updates the parameters (in this case, the weights and biases of a neural network) step by step to minimize the error between the predicted output and the actual output.

Key Point:
It tries to minimize a loss function.

**Overfitting**

Definition:
When a model learns the training data too closely and performs poorly on new data.
Example:
A student who memorizes only past exam questions and cannot solve new ones is an example of overfitting. The student has learned the patterns in the past data, but has not generalized well

In [ ]:
# 한국어
messages = [
    {"role": "system", "content": "너는 친절한 수학 튜터야."},
    {"role": "user", "content": """아래 예시처럼 어려운 설명을 더 쉬운 설명으로 바꿔줘.

예시 1
입력: 경사하강법은 손실함수를 최소화하기 위해 반복적으로 파라미터를 갱신하는 최적화 기법이다.
출력: 경사하강법은 오차를 줄이기 위해 값을 조금씩 바꾸어 가는 방법이다.

예시 2
입력: 과적합은 모델이 학습 데이터에 지나치게 맞추어져 새로운 데이터에서 성능이 떨어지는 현상이다.
출력: 과적합은 연습문제만 외워서 새로운 문제를 잘 못 푸는 것과 비슷하다.

예시 3
입력: 고유값은 선형변환 후에도 방향은 유지된 채 크기만 바뀌는 벡터와 관련된 값이다.
출력: 고유값은 어떤 방향은 그대로 두고 얼마나 늘어나거나 줄어드는지를 나타내는 값이다.

이제 같은 방식으로 바꿔줘.

입력: 최적화는 주어진 조건 아래에서 가장 좋은 해를 찾는 과정이다.
출력:"""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


네, 네. 예시를 바꿔줘면 다음과 같습니다.

예시 1
입력: 경사하강법은 손실함수를 최소화하기 위해 반복적으로 파라미터를 갱신하는 최적화 기법이다.
출력: 경사하강법은 오차를 줄이기 위해 값을 조금씩 바꾸어 가는 방법이다.

예시 2
입력: 과적합은 모델이 학습 데이터에 지나치게 맞추어져 새로운 데이터에서 성능이 떨어지는 현상이다.
출력: 과적합은 연습문제만 외워서 새로운 문제를 잘 못 푸는 것과 비슷하다.

예시 3
입력: 고유값은 선형변환 후에도 방향은 유지된 채 크기만 바뀌는 벡터와 관련된 값이다.
출력: 고유값은 어떤 방


In [ ]:
# 감정 분류처럼 패턴 맞추기
##few-shot은 생성뿐 아니라 간단한 분류 형식에도 잘 씀.
### 목표: 문장을 보고 긍정/부정/중립 중 하나로 답하기

messages = [
    {"role": "system", "content": "You are a text classification assistant."},
    {"role": "user", "content": """Classify the sentiment of each sentence as Positive, Negative, or Neutral.

Sentence: The lecture was very clear and helpful.
Label: Positive

Sentence: The explanation was confusing and too fast.
Label: Negative

Sentence: The class started at 9 AM.
Label: Neutral

Sentence: The examples were useful and easy to follow.
Label:"""}
]

print(chat_with_llama(messages, max_new_tokens=20))
#중요한 점은 답의 범위를 좁게 제한

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Here are the classifications:

1. Sentence: The lecture was very clear and helpful.
   - Sent


In [ ]:
# 코드 설명 스타일 맞추기
## 코드 목적
## 한 줄씩 설명
## 실행 결과 요약

messages = [
    {"role": "system", "content": "You are a Python tutor."},
    {"role": "user", "content": """Explain code in the following format:

Purpose:
Line-by-line explanation:
Result:

Example 1
Code:
x = 3
y = 4
print(x + y)

Purpose:
This code adds two numbers and prints the result.
Line-by-line explanation:
x = 3 -> stores 3 in x
y = 4 -> stores 4 in y
print(x + y) -> prints the sum of x and y
Result:
It prints 7.

Now explain this code.

Code:
for i in range(3):
    print(i)

Purpose:
Line-by-line explanation:
Result:"""}
]

print(chat_with_llama(messages, max_new_tokens=250))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


**Purpose:**
This code is a simple example of a `for` loop in Python, which is used to iterate over a sequence of numbers or values.

**Line-by-line explanation:**

1. `for i in range(3):` 
   - `for` is the keyword used to start a `for` loop.
   - `i` is the variable that will take on each value in the sequence.
   - `range(3)` generates a sequence of numbers from 0 to 2 (inclusive).

2. `print(i)` 
   - `print` is the function used to output the value of `i` to the console.

3. `Result:` 
   - This line is a placeholder for the result of the code.

**Example 1:**

This code adds two numbers and prints the result.

1. `x = 3` 
   - `x` is the variable that will store the first number in the sequence.
   - `3` is the value of `x`.

2. `y = 4` 
   - `y` is the variable that will store the second number in the sequence.
   - `4` is the value of `y`.

3


#### Llama로 chain-of-thought 없이 reasoning을 더 잘 끌어내는 프롬프트

**Llama에서 reasoning 품질을 높이려면 보통 아래 5개가 잘 먹힘.**

1. 문제를 작게 나누기

- 한 번에 다 하라고 하지 말고, 필요한 작업을 분리합니다.

    - 예:

    - 핵심 정보 추출
    - 가능한 선택지 비교
    - 최종 결론 제시

2. 출력 형식을 고정하기

 - 형식이 없으면 모델이 흔들립니다.

    - 예:

    - 핵심 정보:
    - 판단 기준:
    - 결론:

3. 제약을 분명히 주기

    - 예:

    - “불확실하면 추측하지 말고 모른다고 말하라”
    - “답만 쓰지 말고 근거 2개만 요약하라”
    - “3문장 이내로 답하라”

4. 자기검토 단계를 넣기

    - 예:

    - 초안 작성
    - 오류 점검
    - 최종 답 작성

5. 예시를 보여주기

- few-shot 예시가 있으면 형식과 판단 방식이 더 안정.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

def chat_with_llama(messages, max_new_tokens=200, temperature=0.3, top_p=0.9):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
# 패턴 A. 정보 추출 → 판단 → 답변
messages = [
    {"role": "system", "content": "You are a careful reasoning assistant."},
    {"role": "user", "content": """Answer using this format:

Relevant information:
- list only the important facts

Decision:
- give a short judgment based on the facts

Final answer:
- give the final answer clearly

Question:
A student studied only memorized examples and fails on new problems. Is this more similar to overfitting or underfitting?"""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Relevant information:
- Memorization of examples
- Failure on new problems

Decision:
- This is more similar to overfitting.


In [ ]:
# 패턴 B: 선택지 비교형
messages = [
    {"role": "system", "content": "You are a precise academic assistant."},
    {"role": "user", "content": """Compare the two concepts briefly and choose the better match.

Concepts:
1. Overfitting
2. Underfitting

Case:
A model performs extremely well on training data but poorly on new data.

Use this format:
Option 1:
- Strength:
- Weakness:

Option 2:
- Strength:
- Weakness:

Best choice:
Reason:"""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


**Concepts:**

1. **Overfitting:**
- Strength: Overfitting occurs when a model is too complex and fits the training data too closely, resulting in poor performance on new, unseen data.
- Weakness: Overfitting can lead to overreliance on the training data, making the model unresponsive to changes in the underlying distribution.

2. **Underfitting:**
- Strength: Underfitting occurs when a model is too simple and fails to capture the underlying patterns in the training data, resulting in poor performance on new data.
- Weakness: Underfitting can lead to poor generalization, as the model may not be able to handle complex relationships between variables.

**Best choice:**
**Option 1: Overfitting**
Overfitting is a more severe issue than underfitting, as it can lead to poor generalization and decreased model reliability. While underfitting can still be problematic, it is generally less severe than over


In [ ]:
# 패턴 C: 체크리스트 기반 추론
messages = [
    {"role": "system", "content": "You are a careful math tutor."},
    {"role": "user", "content": """Determine whether gradient descent is likely to converge in this situation.

Situation:
- The learning rate is very large.
- The objective is smooth.
- The update is repeated iteratively.

Answer in this format:
Condition 1: Is the learning rate too large?
Condition 2: Is the objective smooth?
Condition 3: Is the update repeated iteratively?
Final conclusion:"""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Condition 1: Is the learning rate too large?

Yes, the learning rate is very large. In gradient descent, the learning rate is a hyperparameter that controls how quickly the model updates its weights. A large learning rate can lead to rapid convergence, but it can also cause the model to overshoot the optimal solution. If the learning rate is too large, the model may not be able to converge to the optimal solution, or it may converge to a local minimum.

Condition 2: Is the objective smooth?

Yes, the objective is smooth. Smooth objectives are those that have a continuous derivative and do not have any sharp turns or discontinuities. In gradient descent, the objective function is typically smooth, meaning that its derivative is always non-zero. This allows the algorithm to converge to the optimal solution.

Condition 3: Is the update repeated iteratively?

Yes, the update is repeated iteratively. In gradient descent, the update rule is to update the weights by subtracting the product


In [ ]:
# 패턴 D: 초안 → 검토 → 최종답
messages = [
    {"role": "system", "content": "You are a careful assistant."},
    {"role": "user", "content": """Answer the question in three stages.

Stage 1. Draft:
Write a short draft answer.

Stage 2. Review:
Check whether the draft is missing anything important or contains an error.

Stage 3. Final:
Write a corrected final answer.

Question:
Why can a too-large learning rate make gradient descent unstable?"""}
]

print(chat_with_llama(messages, max_new_tokens=250))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Stage 1: Draft

A too-large learning rate can make gradient descent unstable because it can lead to overshooting the optimal solution. When the learning rate is too large, the algorithm may converge to a local minimum, rather than the global minimum. This is because the algorithm is trying to quickly adjust the weights to minimize the loss, but it may overshoot the optimal solution and then converge to a local minimum.

Stage 2: Review

Upon reviewing the draft, I noticed that the question is asking for a specific reason why a too-large learning rate can make gradient descent unstable. The correct answer is that a too-large learning rate can lead to overshooting the optimal solution, which can cause the algorithm to converge to a local minimum instead of the global minimum.

Stage 3: Final

A too-large learning rate can make gradient descent unstable by leading to overshooting the optimal solution. This can cause the algorithm to converge to a local minimum, rather than the global mini

In [ ]:
# 패턴 E: 근거 제한형

messages = [
    {"role": "system", "content": "You are a concise reasoning assistant."},
    {"role": "user", "content": """Question:
Why is overfitting harmful?

Instructions:
- Give exactly 2 reasons.
- Each reason must be one sentence.
- Then give the final conclusion in one sentence."""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Overfitting is harmful because it leads to poor model generalization, resulting in the model being unable to accurately predict new, unseen data.

Overfitting is harmful because it results in a model that is too complex and is unable to capture the underlying patterns and relationships in the data.

Overfitting is harmful because it can lead to overreliance on the training data, causing the model to perform well on that specific dataset but poorly on new, unseen data, ultimately leading to decreased model performance and reliability.


In [ ]:
# 같은 질문을 "구조 없는 프롬프트"와 비교하기
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Why is overfitting harmful?"}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Overfitting is a significant issue in machine learning models, and it can be quite harmful. Here's why:

Overfitting occurs when a model is too complex and learns the noise in the training data rather than the underlying patterns. This means that the model is fitting the random fluctuations in the data rather than the underlying relationships. As a result, the model performs well on the training data but poorly on new, unseen data.

The harm caused by overfitting is multifaceted:

1. **Poor generalization**: Overfitting leads to poor performance on new data, making it difficult to predict outcomes or make decisions based on the model's output. This can result in suboptimal decisions or actions.
2. **Increased risk of errors**: Overfitting can lead to overestimation of the model's accuracy, causing it to make incorrect predictions or recommendations. This can have serious consequences, such as financial losses, reputational damage, or even harm to people's lives.
3.


- Llama로 요약, 번역, 코드 생성

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

def chat_with_llama(messages, max_new_tokens=250, temperature=0.4, top_p=0.9):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [ ]:
# 요약 실습
text = """
Gradient descent is an optimization method used to minimize a loss function.
It works by iteratively updating parameters in the direction opposite to the gradient.
If the learning rate is too small, convergence can be slow.
If the learning rate is too large, the algorithm may overshoot the minimum and become unstable.
This method is widely used in machine learning and deep learning.
"""

messages = [
    {"role": "system", "content": "You are a helpful assistant for summarization."},
    {"role": "user", "content": f"""Summarize the following text in 3 bullet points.

Text:
{text}"""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Here are three bullet points summarizing the text:

• Gradient descent is an optimization method used to minimize a loss function by iteratively updating parameters in the direction opposite to the gradient.
• The method can converge slowly if the learning rate is too small, or overshoot the minimum if the learning rate is too large.
• It is widely used in machine learning and deep learning due to its simplicity and effectiveness in minimizing loss functions.


In [ ]:
# 더 좋은 요약 프롬프트
text = """
Gradient descent is an optimization method used to minimize a loss function.
It works by iteratively updating parameters in the direction opposite to the gradient.
If the learning rate is too small, convergence can be slow.
If the learning rate is too large, the algorithm may overshoot the minimum and become unstable.
This method is widely used in machine learning and deep learning.
"""

messages = [
    {"role": "system", "content": "You are a careful summarization assistant."},
    {"role": "user", "content": f"""Summarize the text using this format:

Main idea:
Important detail 1:
Important detail 2:
One-sentence conclusion:

Text:
{text}"""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Main idea: Gradient descent is an optimization method used to minimize a loss function.
Important detail 1: It works by iteratively updating parameters in the direction opposite to the gradient.
Important detail 2: The learning rate is crucial, as it determines convergence speed and stability.
One-sentence conclusion: Gradient descent is a widely used optimization method in machine learning and deep learning that iteratively updates parameters to minimize a loss function.


In [ ]:
# 한국어 요약 실습
text = """
경사하강법은 손실함수를 최소화하기 위해 반복적으로 파라미터를 갱신하는 최적화 방법이다.
기울기의 반대 방향으로 조금씩 이동하면서 더 좋은 해를 찾는다.
학습률이 너무 작으면 수렴 속도가 느리고, 너무 크면 최솟값을 지나쳐 불안정해질 수 있다.
이 방법은 머신러닝과 딥러닝에서 매우 널리 사용된다.
"""

messages = [
    {"role": "system", "content": "너는 요약을 잘하는 조교야."},
    {"role": "user", "content": f"""다음 글을 3줄로 요약해줘.

글:
{text}"""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


경사하강법은 손실함수를 최소화하기 위해 반복적으로 파라미터를 갱신하는 최적화 방법이다. 이 방법은 학습률이 너무 작거나 너무 크면 불안정해질 수 있으며, 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반복적으로 반


 - 번역 능력이 없는 게 아니라 프롬프트가 덜 엄격해서 형식이 흔들린 것

In [ ]:
# 더 염격하게 작성

text = "Gradient descent updates parameters iteratively to reduce the loss function."

messages = [
    {"role": "system", "content": "You are a translation assistant. Translate English into natural Korean only."},
    {"role": "user", "content": f"""Translate the following sentence into natural Korean.
Do not explain.
Do not keep English words unless absolutely necessary.
Output only the Korean translation.

Sentence:
{text}"""}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


이산화탄소가 사용되는 방정식은 다음과 같습니다.

i = 1, j = 2
∂f(x)∂x1 = -2x2
∂f(x)∂x2 = -x1
∂f(x)∂x3 = -x1 - x2
∂f(x)∂x4 = -x1 - x2 - x3

i = 2, j = 3
∂f(x)∂x1 = -2x3
∂f(x)∂x2 = -x1
∂f(x)∂x3 = -x1 - x2
∂f(x)∂x4 = -x1 - x2 - x3

i = 3, j = 4
∂f(x)∂x1 = -2x4
∂f(x)∂x2 = -x1
∂f(x)∂x3 = -x1 - x2
∂f(x)∂x4 = -x1 - x2 - x3

i = 4, j =


In [ ]:
# 번역 실습
## 영어 -> 한국어
text = "Gradient descent updates parameters iteratively to reduce the loss function."

messages = [
    {"role": "system", "content": "You are a translation assistant."},
    {"role": "user", "content": f"Translate this into Korean:\n\n{text}"}
]

print(chat_with_llama(messages))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Korean 번역:

梯度 descents은Loss function을 최소화하기 위해parameter를 iteratively 업데이트합니다.


In [ ]:
# 한국어 -> 영어
text = "경사하강법은 오차를 줄이기 위해 값을 조금씩 바꾸는 방법이다."

messages = [
    {"role": "system", "content": "You are a translation assistant."},
    {"role": "user", "content": f"Translate this into natural English:\n\n{text}"}
]

print(chat_with_llama(messages))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


경사하강법은 오차를 줄이기 위해 값을 조금씩 바꾸는 방법이다.


In [ ]:
# 코드 생성
messages = [
    {"role": "system", "content": "You are a Python tutor."},
    {"role": "user", "content": """Write Python code to compute the sum of numbers from 1 to 10.
Explain the code briefly after the code."""}
]

print(chat_with_llama(messages, max_new_tokens=300))

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


**Code:**
```python
def sum_numbers(n):
    """
    Compute the sum of numbers from 1 to n.

    Args:
        n (int): The upper limit of the sum.

    Returns:
        int: The sum of numbers from 1 to n.
    """
    return n * (n + 1) // 2

# Example usage:
n = 10
result = sum_numbers(n)
print(f"The sum of numbers from 1 to {n} is: {result}")
```

**Explanation:**

This code defines a function `sum_numbers` that takes an integer `n` as input and returns the sum of numbers from 1 to `n`. The formula used to calculate the sum is the arithmetic series formula: `n * (n + 1) / 2`.

In this case, we call the function with `n = 10` and print the result. The output will be the sum of numbers from 1 to 10, which is `55`.


##

## 1. Llama 4가 무엇인가

**Llama 4**는 Meta가 2025년 4월 5일 공개한 차세대 Llama 계열 모델입니다.  

1. **MoE(Mixture-of-Experts) 구조 채택**
2. **네이티브 멀티모달 지원**(텍스트 + 이미지 입력)
3. **매우 긴 컨텍스트 처리 능력 강화**

Llama 4는 이전의 “텍스트 중심 LLM”에서 한 단계 더 나아가, **더 긴 문맥을 다루고, 이미지도 함께 이해하며, 계산 효율까지 높이려는 방향**으로 설계된 모델군이라고 보면 된다. 

---

## 2. 공개된 모델 종류

### (1) Llama 4 Scout
- **17B active parameters**
- **16 experts**
- **총 파라미터 약 109B**
- **최대 10M tokens 수준의 매우 긴 컨텍스트**
- 긴 문서, 대규모 로그, 긴 코드베이스처럼 **아주 긴 입력을 다루는 작업**에 강점이 있습니다. 

### (2) Llama 4 Maverick
- **17B active parameters**
- **128 experts**
- **총 파라미터 약 400B**
- **최대 1M tokens 컨텍스트**
- Scout보다 더 큰 총 규모와 더 많은 expert를 활용해 **추론, 코딩, 멀티모달 이해 성능**을 강화한 모델.

### (3) Llama 4 Behemoth
- 출시 시점에 바로 공개된 배포용 모델은 아니고, Meta가 **teacher model**로 소개한 대형 모델.
- **288B active parameters**, **16 experts**, **총 파라미터는 nearly 2T**로 설명.
- Scout와 Maverick의 학습/증류에 활용되는 상위 모델 성격.

---

## 3. “active parameters”와 “total parameters”가 왜 다른가


MoE 구조에서는 모델 전체에 많은 expert가 들어 있지만, **입력 토큰 하나를 처리할 때 모든 expert를 동시에 쓰지 않습니다.**  

대신 일부 expert만 선택적으로 활성화.

그래서 예를 들어 Maverick은 총 파라미터가 약 400B이지만, 실제로 한 시점에 계산에 직접 참여하는 것은 **17B active parameters**.  

이 구조의 장점은 다음과 같습니다.

- 총 모델 용량은 크게 가져갈 수 있음
- 하지만 실제 추론 시 계산량은 상대적으로 줄일 수 있음
- 결과적으로 **성능과 효율 사이의 균형**을 노릴 수 있음

**“큰 모델의 표현력”과 “상대적으로 낮은 계산비용”을 동시에 추구하는 구조**라고 이해. 

---

## 4. Llama 4의 가장 큰 기술적 특징

## 4-1. MoE(Mixture-of-Experts)

Llama 4는 이전 세대와 달리 **MoE 기반 autoregressive language model**.

각 expert는 특정 유형의 패턴이나 지식에 더 잘 반응하도록 학습될 수 있고,   라우팅 메커니즘이 입력에 따라 어떤 expert를 쓸지 결정합니다. 

이 구조는 특히 다음 상황에서 유리.

- 모델을 더 크게 만들고 싶을 때
- 추론 비용을 너무 많이 늘리고 싶지 않을 때
- 다양한 종류의 작업을 하나의 모델에 담고 싶을 때

---

## 4-2. 네이티브 멀티모달

Llama 4는 **텍스트와 이미지 입력을 함께 처리할 수 있는 네이티브 멀티모달 모델**로 소개.  

공식 설명에서는 **early fusion**을 사용한다고 되어 있는데, 이는 텍스트와 이미지 정보를 비교적 이른 단계에서 함께 통합해 처리하는 방식.


이 말은 곧 다음과 같은 작업이 가능하다는 뜻.

- 이미지 설명
- 이미지 기반 질의응답
- 문서 이미지 + 텍스트 질의 결합
- 차트/도표/스크린샷 해석 보조

다만 출력은 기본적으로 **텍스트 생성 중심**으로 이해하는 것이 안전합니다. 

---

## 4-3. 초장문 컨텍스트

Scout의 매우 인상적인 특징은 **최대 10M token** 수준의 초장문 컨텍스트.  
Maverick도 **1M token** 컨텍스트를 지원하는 것으로 발표. 

이는 일반적인 LLM의 컨텍스트 길이와 비교하면 매우 큰 수치. 

이런 긴 컨텍스트가 유리한 작업은 다음과 같습니다.

- 여러 편의 긴 논문을 한 번에 넣고 비교
- 대형 코드 저장소 전체 수준 분석
- 장시간 대화 이력 기반 에이전트
- 수많은 로그/레코드/기록을 한 번에 참조하는 시스템

다만 실무에서는 **“이론적 최대 길이”와 “현실적 배포 길이”는 다를 수 있다**는 점도 같이 기억.  
플랫폼, 메모리, 추론 엔진 설정에 따라 실제 사용 가능한 길이는 더 작아질 수 있습니다. 

---

## 5. 학습 데이터와 언어 지원

Meta와 공식 모델 카드 계열 설명에 따르면, Llama 4는 ㅜ**200개 언어를 포함하는 데이터로 사전학습되었고**,  
그중 **12개 언어를 공식 지원 언어**로 강조. 


- Arabic
- English
- French
- German
- Hindi
- Indonesian
- Italian
- Portuguese
- Spanish
- Tagalog
- Thai
- Vietnamese :

한국어는 “공식 지원 12개 언어” 목록에는 보이지 않습니다.  

다만 사전학습 자체는 더 넓은 언어 범위를 포함한다고 설명되어 있어, 한국어가 전혀 불가능하다는 뜻은 아니고, **공식 최적화 대상 언어가 아니라는 의미에 가깝습니다.** 

---

## 6. 이전 세대 대비 무엇이 달라졌나

Llama 3.x 계열과 비교했을 때, Llama 4의 변화는 대체로 다음과 같이 정리할 수 있습니다.

### (1) Dense 중심 → MoE 중심
이전 세대는 주로 dense transformer 계열로 이해하면 되지만,  
Llama 4는 본격적으로 **MoE 구조**를 전면에 내세웁니다. 

### (2) 텍스트 중심 → 멀티모달 확장
이전에도 일부 멀티모달 흐름은 있었지만, Llama 4는 **네이티브 멀티모달**을 핵심 정체성으로 강조합니다. 
### (3) 긴 컨텍스트의 대폭 확장
특히 Scout의 **10M token**은 이전 세대 대비 매우 큰 점프입니다. 

### (4) 배포 효율성 강조
Scout는 단일 서버급 GPU에서의 배포 접근성을 강조하고, Maverick은 BF16/FP8 형태의 효율 배포를 지원하는 방향으로 소개. 
---

## 7. Llama 4의 장점

### 장점 1. 긴 문맥 처리에 매우 강함
특히 Scout는 긴 논문 묶음, 대용량 문서, 코드베이스, 로그 분석 같은 작업에 매력적. 

### 장점 2. 멀티모달 활용 가능
텍스트만이 아니라 이미지까지 함께 입력할 수 있어서, 문서 이해나 시각 자료 해석 작업에 응용 범위가 넓습니다. 

### 장점 3. MoE 기반 효율성
총 모델 규모는 크게 유지하면서도 active parameters를 제한해 계산 효율을 확보하려는 설계가 돋보임. 

### 장점 4. 연구/개발 생태계 접근성
Hugging Face와 `transformers`에서 비교적 빠르게 통합되었고, 공식 체크포인트도 제공됩니다. `transformers`에서는 Llama 4 지원 문서가 제공되며, 최소 버전 안내도 포함되어 있습니다. 

---

## 8. 주의할 점

### (1) “오픈소스”라고 단순화하면 부정확함
Llama 4는 흔히 open model 또는 open-weight처럼 언급되지만, 공식 배포는 **Llama 4 Community License Agreement** 아래 이루어짐.

특히 **직전 달 월간 활성 사용자 수가 7억 명을 넘는 서비스는 별도 라이선스를 요청해야 한다**고 명시되어 있습니다. 따라서 엄밀하게는 일반적인 의미의 완전 자유 오픈소스와 동일하다고 보기는 어렵습니다. 

### (2) 긴 컨텍스트는 실제 운영 비용이 큼
10M token 같은 수치는 매우 인상적이지만, 실제로는 메모리 사용량, 추론 속도, 서버 비용, 엔진 최적화 문제를 함께 고려. 이론적으로 가능하다고 해서 항상 실용적인 것은 아닙니다. 

이는 공식 문서와 배포 문서에서도 긴 컨텍스트 사용 시 별도 설정과 하드웨어 고려가 필요하다는 점에서 추론할 수 있습니다. 

### (3) 한국어 최적화 여부는 따로 검증 필요
한국어가 공식 지원 12개 언어에 포함되어 있지 않으므로, 한국어 응답 품질이나 장문 추론 성능은 실제 태스크에서 반드시 테스트해보는 것이 좋습니다. 

---

## 9. 어떤 상황에서 Scout와 Maverick 중 무엇을 고를까

### Scout가 더 적합한 경우
- 매우 긴 문서를 다뤄야 할 때
- 비교적 제한된 자원에서 시작하고 싶을 때
- 긴 컨텍스트 기반 RAG, 문서 요약, 로그 분석이 중요할 때 

### Maverick이 더 적합한 경우
- 코딩, 복합 추론, 멀티모달 이해 성능을 좀 더 강하게 원할 때
- 더 큰 모델 규모와 더 많은 expert를 활용한 품질을 기대할 때
- 더 강한 상용 서비스형 배포를 염두에 둘 때 

- mlx-lm은 Apple Silicon용 LLM 실행/파인튜닝 패키지
 - 텍스트 전용으로 길습을 진행






In [1]:
!pip install huggingface_hub 

In [ ]:
from huggingface_hub import InferenceClient

HF_TOKEN = "HF_TOKEN"

MODEL_ID = "meta-llama/Llama-4-Scout-17B-16E-Instruct"

client = InferenceClient(
    provider="auto",   # 가능한 provider 자동 선택
    api_key=HF_TOKEN
)

In [12]:
messages = [
    {"role": "user", "content": "Llama 4 Scout를 5문장으로 쉽게 설명해줘."}
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=220
)

print(response.choices[0].message.content)

Llama 4 Scout는 Meta에서 개발한 인공지능 모델입니다. 이 모델은 자연어 처리 능력을 바탕으로 다양한 질문에 답변을 생성할 수 있습니다. Llama 4 Scout는 특히 대화형 인터페이스에서 뛰어난 성능을 발휘하며, 사용자와의 상호작용을 통해 학습하고 개선됩니다. 이 모델은 챗봇, 가상 비서 및 고객 서비스와 같은 다양한 응용 프로그램에 통합될 수 있습니다. Llama 4 Scout의 주요 목표는 사용자에게 정확하고 유용한 정보를 제공하고, 동시에 대화 경험을 향상시키는 것입니다.


In [13]:
from huggingface_hub import InferenceClient

MODEL_ID = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
client = InferenceClient(provider="auto", api_key=HF_TOKEN)

def chat_once(
    prompt,
    system_prompt=None,
    max_tokens=220,
    temperature=0.2,
    top_p=0.95
):
    messages = []

    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
    )

    return response.choices[0].message.content

In [14]:
prompt = "Llama 4 Scout를 5문장으로 쉽게 설명해줘."
response = chat_once(prompt, max_tokens=220, temperature=0.2)
print(response)

Llama 4 Scout는 Meta에서 개발한 인공지능 모델입니다. 이 모델은 다양한 작업에 활용할 수 있는 언어 모델로서, 텍스트 생성, 요약, 번역 등의 기능을 제공합니다. Llama 4 Scout는 이전 버전인 Llama 3의 성능을 개선하였으며, 더 정확하고 효율적인 결과를 제공합니다. 이 모델은 자연어 처리 작업에 특화되어 있어 챗봇, 가상 비서 등의 애플리케이션에 적합합니다. Llama 4 Scout는 개발자들이 쉽게 사용할 수 있도록 설계되었으며, 다양한 산업 분야에서 활용될 것으로 기대됩니다.


In [15]:
# system prompt

system_prompt = "You are a clear and patient AI teacher. Use very simple language."

prompt = "Llama 4 Scout를 초보자에게 설명해줘."

response = chat_once(
    prompt,
    system_prompt=system_prompt,
    max_tokens=220,
    temperature=0.2
)

print(response)

Llama 4 Scout는 메타에서 개발한 인공지능 모델입니다. 

Llama는 'Large Language Model Meta AI'의 약자입니다. 

Scout는 아마도 이 모델의 버전이나 이름을 나타내는 것 같습니다.

이 모델은 컴퓨터가 인간과 비슷하게 언어를 이해하고 생성할 수 있도록 설계되었습니다. 

초보자를 위해 간단히 설명하면, Llama 4 Scout는 다음과 같은 일을 할 수 있습니다:

1. 질문에 답변해줍니다.
2. 텍스트를 생성해줍니다.
3. 대화와 소통을 도와줍니다.

Llama 4 Scout는 대량의 데이터를 학습하여 이러한 능력을 갖추게 되었습니다.


In [16]:
# temperture 비교
prompt = "양자화의 장점을 5문장으로 설명해줘."

for temp in [0.0, 0.3, 0.7, 1.0]:
    print("=" * 80)
    print("temperature =", temp)
    print(chat_once(prompt, max_tokens=220, temperature=temp))
    print()

temperature = 0.0
양자화는 고전적인 연속 신호를 이산적인 디지털 신호로 변환하는 과정으로, 신호의 손실 없이 데이터를 저장하고 전송할 수 있습니다. 이를 통해 디지털 신호 처리가 가능해지며, 컴퓨터와 같은 디지털 장치에서 신호를 쉽게 처리하고 분석할 수 있습니다. 양자화는 또한 신호의 압축과 복원을 가능하게 하여 데이터 저장과 전송의 효율성을 높입니다. 더불어 양자화는 신호의 노이즈 제거와 필터링에도 사용되며, 신호의 품질을 향상시키는 데 도움이 됩니다. 이러한 장점들로 인해 양자화는 디지털 신호 처리 분야에서 널리 사용됩니다.

temperature = 0.3
양자화는 아날로그 신호를 디지털 신호로 변환하는 과정으로, 여러 가지 장점이 있습니다. 첫째, 양자화를 통해 아날로그 신호를 디지털로 변환함으로써 노이즈에 민감하지 않은 신호를 얻을 수 있습니다. 둘째, 디지털 신호는 저장과 전송이 용이하여 데이터 관리가 간단해집니다. 셋째, 양자화를 통해 신호를 압축하거나 필터링할 수 있어 데이터 처리 효율성이 향상됩니다. 넷째, 디지털 신호는 쉽게 복제하거나 편집할 수 있어 다양한 응용 분야에서 활용할 수 있습니다. 마지막으로, 양자화는 신호의 정확성을 유지하면서도 데이터 양을 줄일 수 있어, 저장 공간과 전송 대역폭을 효율적으로 사용할 수 있습니다.

temperature = 0.7
양자화는 아날로그 신호를 디지털 신호로 변환하는 과정으로, 여러 가지 장점이 있습니다. 첫째, 양자화를 통해 아날로그 신호를 디지털로 변환함으로써 노이즈에 강해지고 신호의 품질을 유지할 수 있습니다. 둘째, 디지털 신호는 쉽게 저장하고 전송할 수 있어 데이터 관리가 용이해집니다. 셋째, 양자화를 통해 신호를 압축하거나 필터링할 수 있어 데이터의 효율적인 처리가 가능합니다. 넷째, 디지털 신호는 쉽게 분석하고 처리할 수 있어 다양한 응용 분야에서 활용할 수 있습니다. 마지막으로, 양자화는 신호의 정확성과 신뢰성을 높여주어 다양한 분야에서 활용되고 있습니다.

tempera

In [17]:
# top-p
prompt = "딥러닝에서 양자화가 왜 중요한지 쉽게 설명해줘."

for tp in [0.7, 0.85, 0.95, 1.0]:
    print("=" * 80)
    print("top_p =", tp)
    print(chat_once(prompt, max_tokens=220, temperature=0.7, top_p=tp))
    print()

top_p = 0.7
딥러닝에서 양자화(Quantization)는 모델의 성능을 유지하면서도 모델의 크기와 연산 속도를 크게 개선할 수 있는 기술입니다. 쉽게 말해, 양자화는 딥러닝 모델을 더 작고 효율적으로 만드는 방법 중 하나입니다.

### 왜 양자화가 중요한가?

1. **모델 크기 감소**: 딥러닝 모델들은 대개 매우 크고 무겁습니다. 이런 모델들을 모바일 기기나 임베디드 시스템처럼 자원이 제한된 환경에서 실행하려면, 모델의 크기를 줄여야 합니다. 양자화는 모델의 가중치와 연산을 정수 또는 낮은 정밀도(float)로 변환하여 모델 크기를 크게 줄일 수 있습니다.

2. **연산 속도 향상**: 높은 정밀도의 부동소수점 연산(Floating Point Operation)은 자원이 제한된 장치에서 실행하기에 느릴 수 있습니다. 양자화된 모델은 정수 연산을 사용하므로, 훨씬 빠르게 실행될 수 있습니다.

3. **에너지 소비 감소**: 모바일 기기나 IoT 장치에서는 에너지

top_p = 0.85
딥러닝에서 양자화(Quantization)는 모델의 성능을 유지하면서도 모델의 크기와 추론 속도를 크게 개선할 수 있는 기술입니다. 쉽게 설명해 보겠습니다.

### 1. **딥러닝 모델의 크기 문제**

- 현대 딥러닝 모델들은 대량의 파라미터(가중치와 편향)를 가지고 있습니다. 예를 들어, BERT와 같은 대규모 언어 모델은 수억 개의 파라미터를 가질 수 있습니다.
- 이 많은 파라미터들은 모델의 높은 성능을 가능하게 하지만, 동시에 모델의 크기를 매우 크게 만들어서 배포 및 실행 시 문제가 될 수 있습니다.

### 2. **추론 속도의 문제**

- 큰 모델은 많은 메모리를 필요로 하고, 이로 인해 모델의 추론 속도가 느려질 수 있습니다. 특히, 모바일 기기나 임베디드 시스템과 같이 자원이 제한된 환경에서는 큰 모델을 효율적으로 실행하는 것이 어려울 수 있습니다.

### 3. **양자화란?**

- 양자화는 모델의 파라

top_p = 0.95
딥러닝에

In [18]:
# few-shot learning
def chat_fewshot(user_question):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI tutor. Answer briefly, clearly, and in Korean."
        },
        {
            "role": "user",
            "content": "Transformer가 뭐야?"
        },
        {
            "role": "assistant",
            "content": "Transformer는 문장 안의 단어들이 서로 어떻게 관련되는지 attention으로 파악하는 신경망 구조입니다."
        },
        {
            "role": "user",
            "content": "양자화가 뭐야?"
        },
        {
            "role": "assistant",
            "content": "양자화는 모델의 수치를 더 적은 비트로 표현해서 메모리와 계산량을 줄이는 방법입니다."
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        max_tokens=220,
        temperature=0.2
    )
    return response.choices[0].message.content

In [19]:
print(chat_fewshot("Llama 4 Scout가 뭐야?"))

Llama 4 Scout는 Meta에서 개발한 인공지능 모델입니다.


In [20]:
test_questions = [
    "Llama 4 Scout를 쉽게 설명해줘.",
    "MoE가 무엇인지 설명해줘.",
    "왜 큰 모델은 로컬에서 실행이 어려운가?",
    "temperature를 높이면 어떤 일이 생기나?",
    "few-shot prompting의 장점은 무엇인가?"
]

for q in test_questions:
    print("=" * 100)
    print("질문:", q)
    print(chat_once(q, max_tokens=220, temperature=0.3))
    print()

질문: Llama 4 Scout를 쉽게 설명해줘.
Llama 4 Scout는 Meta에서 개발한 인공지능 모델입니다. 이 모델은 다양한 정보를 처리하고 생성할 수 있는 언어 모델로서, 자연어 처리 능력을 갖추고 있습니다. Llama 4 Scout는 이전 버전인 Llama 3의 후속 모델로, 더 향상된 성능과 기능을 제공합니다.

주요 특징은 다음과 같습니다:

*   향상된 언어 이해: Llama 4 Scout는 더 복잡한 자연어 표현을 이해할 수 있습니다. 예를 들어, 사용자가 질문을 하거나 요청을 할 때 더 정확하게 파악하고 응답할 수 있습니다.
*   향상된 응답 생성: 이 모델은 더 다양하고 창의적인 응답을 생성할 수 있습니다. 사용자의 요청에 따라 더 자세한 설명이나 예시를 제공할 수 있습니다.
*   다양한 작업 지원: Llama 4 Scout는 단순한 질문 응답뿐만 아니라, 텍스트 요약, 번역, 대화 생성 등 다양한 작업을 수행할 수 있습니다.

Llama 4 Scout는 Meta의 지속적인 연구와 개발을 통해 더 나은 성능과 기능을 제공할 것으로 기대됩니다.

질문: MoE가 무엇인지 설명해줘.
MoE는 Mixture of Experts의 약자로, 전문가 혼합이라는 뜻입니다. MoE는 여러 개의 전문가 모델을 결합하여 보다 복잡하고 다양한 문제를 해결하기 위한 머신러닝 모델입니다.

MoE의 핵심 아이디어는 여러 개의 전문가 모델을 훈련시키고, 각 전문가 모델이 특정한 부분 문제에 특화되도록 하는 것입니다. 그리고 입력 데이터가 주어지면, 각 전문가 모델의 출력을 가중치 합을 통해 최종 출력을 생성합니다.

MoE는 다음과 같은 장점이 있습니다:

1. **다양한 문제 해결**: MoE는 여러 개의 전문가 모델을 결합하여 다양한 문제를 해결할 수 있습니다.
2. **특화된 전문가**: 각 전문가 모델은 특정한 부분 문제에 특화되므로, 해당 문제에 대한 성능이 뛰어납니다.
3. **유연성**: MoE는 새로운 전문가 모델을 추가하거나 기존 전문가 

In [21]:
# 여러 문서를 한꺼번에 비교시키기
doc1 = """
Transformer는 self-attention을 사용하여 문장 내 각 단어가 다른 단어와 어떤 관련이 있는지 계산한다.
RNN보다 병렬화가 쉽고, 긴 거리 의존성을 더 잘 다룰 수 있다.
하지만 attention 계산량이 시퀀스 길이에 따라 빠르게 증가할 수 있다.
"""

doc2 = """
Transformer는 sequence modeling에서 매우 중요한 구조이다.
각 토큰은 다른 모든 토큰을 참고할 수 있어서 문맥 이해에 유리하다.
반면 긴 입력에서는 메모리 사용량과 계산 비용이 커질 수 있다.
"""

doc3 = """
Transformer는 encoder-decoder 구조로 많이 알려졌지만, decoder-only 형태의 GPT 계열에도 핵심 아이디어가 사용된다.
attention은 중요한 정보를 선택적으로 참고하게 해준다.
그러나 긴 문서를 다룰 때는 효율적인 attention 변형이 필요할 수 있다.
"""

In [22]:
prompt = f"""
아래 3개의 문서를 읽고 다음을 해줘.

1. 세 문서의 공통 주장 3개
2. 각 문서만의 강조점 1개씩
3. 세 문서를 합쳐 학부생용 6문장 요약
4. 마지막에 "학생이 가장 헷갈릴 수 있는 지점" 2개

[문서 1]
{doc1}

[문서 2]
{doc2}

[문서 3]
{doc3}
"""

print(chat_once(prompt, max_tokens=700, temperature=0.2))

### 1. 세 문서의 공통 주장 3개

1. **트랜스포머는 셀프어텐션을 사용한다**: 트랜스포머는 셀프어텐션을 사용하여 문장 내 각 단어가 다른 단어와 어떤 관련이 있는지 계산한다.
2. **트랜스포머는 긴 거리 의존성을 잘 다룬다**: 트랜스포머는 RNN보다 병렬화가 쉽고, 긴 거리 의존성을 더 잘 다룰 수 있다.
3. **트랜스포머는 긴 입력에서 계산 비용이 증가할 수 있다**: 트랜스포머는 긴 입력에서는 메모리 사용량과 계산 비용이 커질 수 있다.

### 2. 각 문서만의 강조점 1개씩

1. **문서 1**: 트랜스포머는 RNN보다 병렬화가 쉽다는 점.
2. **문서 2**: 트랜스포머가 문맥 이해에 유리하다는 점.
3. **문서 3**: 트랜스포머가 decoder-only 형태의 GPT 계열에도 핵심 아이디어가 사용된다는 점.

### 3. 세 문서를 합쳐 학부생용 6문장 요약

트랜스포머는 셀프어텐션을 사용하여 문장 내 각 단어가 다른 단어와 어떤 관련이 있는지 계산하는 모델이다. 
트랜스포머는 RNN보다 병렬화가 쉽고, 긴 거리 의존성을 더 잘 다룰 수 있다. 
트랜스포머는 각 토큰이 다른 모든 토큰을 참고할 수 있어서 문맥 이해에 유리하다. 
그러나 트랜스포머는 긴 입력에서는 메모리 사용량과 계산 비용이 커질 수 있다. 
트랜스포머는 encoder-decoder 구조로 많이 알려져 있지만, decoder-only 형태의 GPT 계열에도 핵심 아이디어가 사용된다. 
트랜스포머의 어텐션 메커니즘은 중요한 정보를 선택적으로 참고하게 해주지만, 긴 문서를 다룰 때는 효율적인 어텐션 변형이 필요할 수 있다.

### 4. 학생이 가장 헷갈릴 수 있는 지점 2개

1. **트랜스포머와 RNN의 차이**: 트랜스포머가 RNN보다 병렬화가 쉽고 긴 거리 의존성을 잘 다루는 이유를 명확히 이해하는 것이 중요하다.
2. **어텐션 메커니즘의 역할**: 트랜스포머의 어텐션 메커니즘이 어떻게 중요한 정보를 선택적으로 참고하게 해주는지, 그리고 이것이 모델의 성능

In [23]:
# 강의노트 구조화 시키기
lecture_note = """
Transformer는 자연어처리의 핵심 구조 중 하나이다.
기존 RNN은 순차 계산이 필요하여 병렬화에 한계가 있었다.
Transformer는 attention 메커니즘을 중심으로 설계되어, 각 위치가 다른 위치를 직접 참고할 수 있다.
Self-attention은 입력 내부 관계를 계산하고, multi-head attention은 여러 관점에서 관계를 동시에 본다.
Positional encoding은 순서 정보를 주입하기 위해 사용된다.
Encoder는 입력 표현을 만들고, decoder는 이전 토큰을 바탕으로 다음 토큰을 예측한다.
GPT는 decoder-only 구조를 사용하고, BERT는 encoder-only 구조를 사용한다.
긴 문장에서는 attention cost가 증가하므로 이를 개선하려는 다양한 방법들이 제안되었다.
실제 응용에서는 번역, 요약, 질의응답, 코드 생성 등 다양한 작업에 활용된다.
"""

In [24]:
prompt = f"""
다음 강의노트를 바탕으로 아래 결과를 만들어줘.

1. 15분 수업용 강의 개요
2. 학생이 꼭 알아야 하는 핵심 용어 8개
3. 학생이 자주 헷갈리는 개념 5개
4. 수업 후 확인문제 3개
5. 마지막에 초보자용 한 단락 요약

강의노트:
{lecture_note}
"""

print(chat_once(prompt, max_tokens=900, temperature=0.3))

### 1. 15분 수업용 강의 개요

**강의 주제:** 트랜스포머(Transformer) 구조와 자연어 처리에서의 응용

**강의 목표:** 
- 트랜스포머의 기본 구조와 원리를 이해한다.
- 트랜스포머가 자연어 처리에서 어떻게 활용되는지 배운다.

**강의 개요:**
1. 트랜스포머의 필요성과 배경 (2분)
   - 기존 RNN의 한계
   - 트랜스포머의 등장 배경

2. 트랜스포머의 핵심 구조 (5분)
   - Self-attention과 multi-head attention
   - Positional encoding

3. 트랜스포머의 응용과 모델 구조 (4분)
   - Encoder-Decoder 구조
   - GPT와 BERT의 구조

4. 실제 응용과 도전 과제 (2분)
   - 다양한 자연어 처리 작업에서의 활용
   - 긴 문장 처리의 도전과 해결 방안

5. 요약 및 확인 문제 (2분)

### 2. 학생이 꼭 알아야 하는 핵심 용어 8개

1. **Transformer**: 자연어 처리의 핵심 구조 중 하나.
2. **Self-attention**: 입력 내부 관계를 계산하는 메커니즘.
3. **Multi-head attention**: 여러 관점에서 관계를 동시에 보는 메커니즘.
4. **Positional encoding**: 순서 정보를 주입하는 방법.
5. **Encoder**: 입력 표현을 만드는 구조.
6. **Decoder**: 이전 토큰을 바탕으로 다음 토큰을 예측하는 구조.
7. **GPT**: Decoder-only 구조를 사용하는 모델.
8. **BERT**: Encoder-only 구조를 사용하는 모델.

### 3. 학생이 자주 헷갈리는 개념 5개

1. **RNN vs. Transformer**: RNN은 순차 계산이 필요하지만, Transformer는 병렬화가 가능하다.
2. **Self-attention vs. Multi-head attention**: Self-attention은 단일 관계, multi-h

In [25]:
# 제약을 많이 주고 출력 통제 보기

prompt = """
다음 조건을 모두 지켜서 답해줘.

- 한국어로 작성
- 정확히 6문장
- 표는 사용하지 말 것
- 예시는 2개만 넣을 것
- 수식은 1개만 넣을 것
- 마지막 문장은 학생에게 질문하는 형태로 끝낼 것

주제: positional encoding
"""

print(chat_once(prompt, max_tokens=300, temperature=0.2))

positional encoding은 입력 시퀀스의 각 토큰의 위치 정보를 인코딩하는 방법입니다. 
이는 트랜스포머 모델에서 중요한 역할을 합니다. 예를 들어, 문장 "나는 사과를 먹었다"에서 각 단어의 위치를 인코딩할 수 있습니다. 
또 다른 예로는, 문장 "I love you"에서 각 단어의 위치를 인코딩할 수 있습니다. 
positional encoding은 다음과 같은 수식으로 나타낼 수 있습니다: $PE_{(pos, 2i)} = sin(\frac{pos}{10000^{2i/d}})$. 
positional encoding의 원리는 무엇이라고 생각하시나요?


In [26]:
# 같은 내용으로 “짧은 설명”과 “구조화 작업”을 비교

simple_prompt = "Transformer를 쉽게 설명해줘."
structured_prompt = f"""
다음 강의노트를 읽고
1. 3문장 핵심 요약
2. 핵심 용어 5개
3. 학생 질문 3개
4. 오해하기 쉬운 포인트 2개
를 정리해줘.

강의노트:
{lecture_note}
"""

print("===== 짧은 질문 =====")
print(chat_once(simple_prompt, max_tokens=220, temperature=0.2))

print("\n===== 구조화 작업 =====")
print(chat_once(structured_prompt, max_tokens=700, temperature=0.2))

===== 짧은 질문 =====
Transformer는 자연어 처리(NLP) 및 기계 번역과 같은 작업에 사용되는 딥러닝 모델입니다. 2017년 구글 연구원들이 발표한 논문 "Attention Is All You Need"에서 소개되었습니다.

Transformer의 핵심 아이디어는 입력 시퀀스(예: 문장)를 처리할 때, 시퀀스의 각 요소가 다른 요소들과 얼마나 관련이 있는지를 학습하는 것입니다. 이를 위해 Self-Attention 메커니즘을 사용합니다.

Self-Attention은 입력 시퀀스의 각 토큰(예: 단어)이 다른 토큰들과 얼마나 관련이 있는지를 계산합니다. 이 관련성은 가중치로 표현되며, 이 가중치를 사용하여 입력 시퀀스의 각 토큰을 가중합합니다. 이렇게 하면 입력 시퀀스의 각 토큰이 다른 토큰들과의 관계를 고려하여 변환됩니다.

Transformer는 인코더(Encoder)와 디코더(Decoder)로 구성됩니다. 인코더는 입력 시퀀스를 처리하여 연속적인 벡터 표현으로 변환하고, 디코더는 이

===== 구조화 작업 =====
### 1. 3문장 핵심 요약

- Transformer는 자연어 처리의 핵심 구조 중 하나로, attention 메커니즘을 통해 각 위치가 다른 위치를 직접 참고할 수 있게 설계되었다.
- Transformer는 self-attention과 multi-head attention을 사용하여 입력 내부 관계를 계산하며, positional encoding을 통해 순서 정보를 주입한다.
- Transformer는 encoder-decoder 구조를 기반으로 하며, GPT와 BERT는 각각 decoder-only와 encoder-only 구조를 사용하여 다양한 자연어 처리 작업에 활용된다.

### 2. 핵심 용어 5개

1. **Transformer**: 자연어 처리에서 사용되는 핵심 구조 중 하나로, attention 메커니즘에 기반한 모델.
2. **Attention 메커니즘**: 모델이 입력의 특정 부분을 집중적으로 처

In [27]:
# 긴 문장
paper1 = """
This study proposes an optimization-based reconstruction framework for inverse problems under limited observations.
The main motivation is that when the amount of observed data is reduced,
the reconstruction problem becomes more ill-posed and artifacts or instability can increase.
To address this issue, the method introduces an iterative objective function
that includes both a data fidelity term and a regularization term
designed to preserve important structural patterns while suppressing noise.

The reconstruction process alternates between enforcing consistency with measured signals
and applying prior assumptions about smoothness and boundary preservation.
Compared with direct reconstruction methods, the proposed approach produces improved results
when the number of observations is limited.
In particular, boundary-like regions are recovered more clearly,
and reconstruction artifacts are reduced.

The evaluation is conducted on simulated benchmark data and a small set of structured images.
Quantitative metrics include PSNR and SSIM, while qualitative inspection focuses on artifact suppression
and pattern sharpness. Although the method performs well under limited-observation settings,
its main limitation is computational cost, since iterative optimization can become slow for large inputs.
The authors also note that parameter tuning is sensitive and may require manual adjustment
depending on the signal condition.
"""

paper2 = """
This paper presents a deep learning approach for reconstruction under noisy or degraded input conditions.
Instead of explicitly solving an optimization problem during inference,
the model is trained to map degraded inputs to cleaner target outputs.
The central idea is that a neural network can learn noise suppression and detail restoration
from paired training data, making reconstruction much faster once training is complete.

The architecture uses convolutional layers with skip connections
in order to preserve local features while removing degradation patterns.
Training is performed on a dataset of paired clean and corrupted samples.
The paper reports that the learned model improves perceptual output quality
and reduces noise significantly compared with standard baseline methods.

For evaluation, the authors use PSNR, SSIM, and visual comparison.
The deep learning method shows strong performance in restoring smooth regions
while also preserving fine structures.
However, the paper highlights several concerns.
First, the method depends heavily on the distribution of the training data.
Second, model outputs may be less interpretable than optimization-based approaches.
Third, generalization to unseen degradations remains uncertain.
The authors suggest that robustness and explainability should be considered in future work.
"""

paper3 = """
This work investigates hybrid reconstruction strategies that combine model-based priors
with data-driven learning for inverse problems.
The motivation is that purely optimization-based methods often provide interpretability and stability,
while purely learning-based methods offer speed and strong empirical performance.
The study aims to integrate the strengths of both approaches.

The proposed framework first generates an intermediate reconstruction using a conventional algorithm
and then refines that reconstruction with a learned neural module.
In some variants, the iterative update itself is partially learned,
allowing the model to incorporate both physical consistency and adaptive feature extraction.
This hybrid design is expected to improve robustness while maintaining high-quality reconstruction.

Experiments are conducted under limited-observation and noisy-input settings.
The results show that the hybrid approach often outperforms traditional baselines
and also provides more stable performance than a fully data-driven model
when the test condition differs from the training condition.
The paper analyzes performance using reconstruction quality metrics,
visual inspection, and ablation studies on the contribution of each module.

The limitations discussed by the authors include increased model complexity,
more difficult implementation, and the challenge of balancing learnable components
with interpretable constraints.
The paper concludes that hybrid frameworks are promising,
especially for applications where reliability, reconstruction fidelity,
and robustness to distribution shift are all important.
"""

In [28]:
prompt = f"""
아래 3개의 연구 요약을 매우 꼼꼼하게 비교해줘.

반드시 다음 구조를 지켜서 답해줘.

1. 전체 공통 주제 1문단
2. 각 연구의 연구문제 정리
3. 각 연구의 핵심 방법 정리
4. 각 연구의 장점과 약점 비교
5. 데이터 의존성, 해석 가능성, 계산 비용 측면에서 3개 연구를 비교
6. 어떤 상황에서 어떤 접근이 더 적절할지 사례별로 설명
7. 세 연구를 통합하는 관점에서 향후 연구 방향 5개 제안
8. 마지막에 대학원 세미나 발표용 10문장 요약 작성

답변은 한국어로 쓰고,
단순 bullet 나열이 아니라 비교가 드러나게 서술해줘.

[연구 1]
{paper1}

[연구 2]
{paper2}

[연구 3]
{paper3}
"""

print(chat_once(prompt, max_tokens=1400, temperature=0.2))

**1. 전체 공통 주제**

세 연구는 모두 제한된 관측 데이터 또는 잡음이 있는 입력 조건에서 역문제에 대한 재구성 방법을 제안하고 있습니다. 각 연구는 재구성 문제의 어려움을 해결하기 위해 다른 접근 방식을 취하고 있으며, 최적화 기반, 딥러닝 기반, 하이브리드 접근 방식의 장단점을 비교하고 있습니다.

**2. 각 연구의 연구문제 정리**

* 연구 1: 제한된 관측 데이터에서 최적화 기반 재구성 프레임워크를 제안하여 재구성 문제의 어려움을 해결하고자 합니다.
* 연구 2: 잡음이 있는 입력 조건에서 딥러닝 기반 재구성 방법을 제안하여 빠르게 재구성하고 잡음을 제거하고자 합니다.
* 연구 3: 모델 기반 사전 정보와 데이터 기반 학습을 결합한 하이브리드 재구성 전략을 제안하여 재구성 문제의 어려움을 해결하고자 합니다.

**3. 각 연구의 핵심 방법 정리**

* 연구 1: 최적화 기반 재구성 프레임워크를 제안하며, 반복적인 목적 함수를 통해 데이터 충실성 항과 정규화 항을 최소화합니다.
* 연구 2: 딥러닝 기반 재구성 방법을 제안하며, 신경망을 통해 잡음이 있는 입력 데이터를 깨끗한 출력 데이터로 매핑합니다.
* 연구 3: 하이브리드 재구성 전략을 제안하며, 모델 기반 사전 정보를 통해 중간 재구성을 생성하고 데이터 기반 학습을 통해 재구성을 개선합니다.

**4. 각 연구의 장점과 약점 비교**

* 연구 1: 장점 - 제한된 관측 데이터에서 우수한 재구성 성능, 경계 영역의 선명한 재구성; 약점 - 계산 비용이 높음, 매개변수 조정이 민감함
* 연구 2: 장점 - 빠르게 재구성, 잡음 제거 성능 우수; 약점 - 훈련 데이터 분포에 의존, 모델 출력 해석 어려움
* 연구 3: 장점 - 모델 기반 사전 정보와 데이터 기반 학습의 장점 결합, 견고한 성능; 약점 - 모델 복잡도 증가, 구현 어려움

**5. 데이터 의존성, 해석 가능성, 계산 비용 측면에서 3개 연구를 비교**

* 데이터 의존성: 연구 2가 훈련 데이터 분포에 가장 많이 의존하며, 

In [29]:
# 평가용 함수 + 긴 비교 실험
def run_test(title, prompt, max_tokens=1000, temperature=0.2):
    print("=" * 100)
    print("TEST:", title)
    print("=" * 100)
    result = chat_once(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature
    )
    print(result)
    print()

In [30]:
comparison_prompt = f"""
아래 3개의 연구 요약을 읽고, 단순 요약이 아니라 비교 중심으로 분석해줘.

요구사항:
1. 세 연구가 공통으로 해결하려는 문제가 무엇인지 설명
2. 연구 1, 2, 3의 접근 방식 차이를 표 대신 문단으로 설명
3. 계산 비용 측면 비교
4. 해석 가능성 측면 비교
5. 새로운 데이터나 환경 변화에 대한 일반화 가능성 비교
6. 실제 응용 관점에서 어떤 접근이 더 현실적일지 논의
7. 세 방법을 통합한 이상적인 프레임워크가 있다면 어떻게 구성될지 제안
8. 마지막에 발표 슬라이드용 핵심 메시지 5줄 작성

[연구 1]
{paper1}

[연구 2]
{paper2}

[연구 3]
{paper3}
"""

run_test(
    "긴 연구 요약 비교",
    comparison_prompt,
    max_tokens=1500,
    temperature=0.2
)

TEST: 긴 연구 요약 비교
## 1. 공통으로 해결하려는 문제

세 연구는 모두 제한된 관찰 데이터 또는 잡음이 있는 입력 조건에서 역문제에 대한 재구성 문제를 해결하려고 합니다. 이 문제는 데이터가 제한되어 있거나 잡음이 있는 경우 재구성이 더 어려워지고, 잡음이나 불안정성이 증가할 수 있습니다.

## 2. 접근 방식 차이

- **연구 1**: 최적화 기반 재구성 프레임워크를 제안합니다. 이 방법은 데이터 충실도 항과 정규화 항을 포함하는 반복적인 목적 함수를 도입하여 중요한 구조적 패턴을 보존하면서 잡음을 억제합니다. 재구성 과정은 측정된 신호와의 일관성을 강제하고 부드러움 및 경계 보존에 대한 이전 가정을 적용하는 방식으로 진행됩니다.

- **연구 2**: 심층 학습 접근 방식을 제시합니다. 이 방법은 최적화 문제를 명시적으로 해결하는 대신, 모델을 학습하여 잡음이 있거나 열화된 입력을 더 깨끗한 목표로 매핑합니다. 신경망은 잡음 억제와 세부 사항 복원을 학습된 데이터 쌍으로부터 학습합니다.

- **연구 3**: 모델 기반 사전 정보와 데이터 기반 학습을 결합한 하이브리드 재구성 전략을 조사합니다. 이 연구는 순수한 최적화 기반 방법의 해석 가능성과 안정성을 데이터 기반 방법의 속도와 강력한 경험적 성능과 통합하는 것을 목표로 합니다.

## 3. 계산 비용 측면 비교

- **연구 1**: 반복적인 최적화 과정으로 인해 계산 비용이 높을 수 있습니다. 특히 대규모 입력 데이터에 대해 느릴 수 있습니다.

- **연구 2**: 학습된 모델을 사용하면 재구성이 매우 빠릅니다. 그러나 학습 과정 자체는 시간이 걸릴 수 있습니다.

- **연구 3**: 하이브리드 접근 방식은 모델 복잡성이 증가하고 구현이 더 어려울 수 있지만, 최적화 기반 방법보다는 빠를 수 있고 데이터 기반 방법보다는 더 안정적일 수 있습니다.

## 4. 해석 가능성 측면 비교

- **연구 1**: 최적화 기반 접근 방식으로 인해 높은 해석 가능성을 제공합니다. 재구성 과정이

In [31]:
followup_prompt = f"""
다음 3개의 연구를 바탕으로 대학원 수업 자료를 만든다고 하자.
아래를 순서대로 작성해줘.

1. 수업 제목
2. 학습목표 4개
3. 학생이 먼저 이해해야 하는 선수 개념 6개
4. 세 연구를 설명하는 20분 수업 개요
5. 학생들이 가장 헷갈릴 개념 5개와 그 이유
6. 수업 후 토론 질문 5개
7. 과제 문제 3개
8. 마지막에 학부생에게 설명하듯 쉬운 한 단락 요약

반드시 비교와 연결성이 드러나게 써줘.

[연구 1]
{paper1}

[연구 2]
{paper2}

[연구 3]
{paper3}
"""

run_test(
    "긴 입력 기반 강의자료 생성",
    followup_prompt,
    max_tokens=1600,
    temperature=0.3
)

TEST: 긴 입력 기반 강의자료 생성
### 1. 수업 제목
**제한된 관찰 하에서의 역문제 재구성: 최적화 기반, 딥러닝, 하이브리드 접근**

### 2. 학습목표
1. 제한된 관찰 하에서의 역문제 재구성에 대한 기본 개념을 이해한다.
2. 최적화 기반 재구성 방법의 원리와 적용 사례를 설명할 수 있다.
3. 딥러닝 기반 재구성 방법의 원리와 적용 사례를 설명할 수 있다.
4. 하이브리드 재구성 방법의 원리와 장단점을 분석할 수 있다.

### 3. 선수 개념
1. 역문제의 정의와 중요성
2. 제한된 관찰 하에서의 재구성 문제
3. 최적화 문제와 목적 함수
4. 딥러닝 기본 개념 (신경망, 훈련 방법)
5. 이미지 재구성 평가 지표 (PSNR, SSIM)
6. 재구성 문제에서의 해석가능성과 안정성

### 4. 20분 수업 개요
1. **최적화 기반 재구성 방법 (5분)**:
   - 제한된 관찰 하에서의 재구성 문제
   - 데이터 충실성 항과 정규화 항을 사용한 목적 함수
   - 반복적 최적화 과정 및 장단점

2. **딥러닝 기반 재구성 방법 (5분)**:
   - 딥러닝 접근 방식 소개
   - 신경망 구조 (컨볼루션 레이어, 스킵 연결)
   - 훈련 데이터와 평가 지표

3. **하이브리드 재구성 방법 (5분)**:
   - 모델 기반 접근과 데이터 기반 접근의 결합
   - 중간 재구성 및 신경망 기반 재구성
   - 잡음 입력 및 제한된 관찰 하에서의 성능

4. **비교 및 결론 (5분)**:
   - 세 가지 방법의 비교
   - 각 방법의 장단점 및 적용 사례

### 5. 학생들이 가장 헷갈릴 개념 5개와 그 이유
1. **최적화 기반 방법 vs. 딥러닝 기반 방법**:
   - 왜 최적화 문제가 딥러닝으로 대체되는지 이해하기 어려움.

2. **목적 함수 구성**:
   - 데이터 충실성 항과 정규화 항의 역할 이해 부족.

3. **딥러닝 모델의 해석가능성**:
   - 블랙박스 모델에 대한 이해 부족.

4. **하이브리드 방

In [32]:
critical_prompt = f"""
아래 3개의 연구 요약을 읽고, 비판적으로 검토해줘.

다음 관점에서 분석해줘.
1. 각 연구가 암묵적으로 가정하는 전제
2. 실제 적용 시 문제가 될 수 있는 부분
3. 논문에서는 좋아 보이지만 현실에서는 약할 수 있는 이유
4. 각 접근법이 실패할 가능성이 큰 상황
5. 논문 심사자 관점에서 추가로 요구할 실험
6. 세 연구를 읽은 뒤 남는 가장 중요한 미해결 문제 3개

답변은 비판적이되 균형 있게 작성해줘.

[연구 1]
{paper1}

[연구 2]
{paper2}

[연구 3]
{paper3}
"""

run_test(
    "비판적 검토 테스트",
    critical_prompt,
    max_tokens=1500,
    temperature=0.2
)

TEST: 비판적 검토 테스트
## 연구 1: 최적화 기반 재구성 프레임워크

### 1. 암묵적으로 가정하는 전제
- 제한된 관측 데이터로 인해 재구성 문제가 더욱 ill-posed 문제가 된다고 가정합니다.
- 재구성 과정에서 데이터 충실성(term)과 정규화(term)이 중요하다고 가정합니다.

### 2. 실제 적용 시 문제가 될 수 있는 부분
- 계산 비용이 높고, 대규모 입력에 대해 느릴 수 있습니다.
- 매개변수 조정이 민감하여 신호 조건에 따라 수동 조정이 필요할 수 있습니다.

### 3. 논문에서는 좋아 보이지만 현실에서는 약할 수 있는 이유
- 시뮬레이션된 벤치마크 데이터와 제한된 구조화된 이미지 세트에서만 평가되었습니다.
- 실제 적용 시 다양한 데이터 세트와 환경에서의 성능 검증이 필요할 수 있습니다.

### 4. 각 접근법이 실패할 가능성이 큰 상황
- 대규모 입력 데이터 또는 복잡한 재구성 문제에서 계산 비용과 성능의 균형을 맞추지 못할 수 있습니다.

### 5. 논문 심사자 관점에서 추가로 요구할 실험
- 더 크고 다양한 데이터 세트를 사용한 성능 평가.
- 다른 재구성 방법과의 비교 연구.

## 연구 2: 딥러닝 기반 재구성 접근법

### 1. 암묵적으로 가정하는 전제
- 신경망이 쌍을 이룬 훈련 데이터를 통해 노이즈 억제 및 세부 사항 복원을 학습할 수 있다고 가정합니다.
- 훈련 데이터의 분포가 테스트 데이터와 유사하다고 가정합니다.

### 2. 실제 적용 시 문제가 될 수 있는 부분
- 훈련 데이터의 분포에 크게 의존하며, 분포 외의 데이터에 대한 일반화가 불확실합니다.
- 모델 출력이 해석하기 어려울 수 있습니다.

### 3. 논문에서는 좋아 보이지만 현실에서는 약할 수 있는 이유
- 주로 평활한 영역과 미세 구조물 복원에 집중되어 있어 복잡한 패턴이나 구조물에 대한 성능이 불확실할 수 있습니다.

### 4. 각 접근법이 실패할 가능성이 큰 상황
- 훈련 데이터와 크게 다른 분포의 데이터에 직면했을 때 성능이 

In [33]:
paper1_long = paper1 + """
In addition, the study emphasizes that limited-observation reconstruction
is not only a technical challenge but also a general problem in signal recovery,
where insufficient measurements make the solution unstable.
The authors discuss how artifact patterns vary depending on observation sparsity
and argue that reconstruction quality should be evaluated not only numerically
but also in terms of whether meaningful structural patterns remain visible.
They further note that iterative methods can be extended with stronger priors,
but this often increases algorithmic complexity and tuning burden.
"""

paper2_long = paper2 + """
The authors also discuss that learning-based approaches may implicitly learn
dataset-specific patterns rather than only the intended reconstruction mapping.
As a result, excellent benchmark performance does not always guarantee robustness
under distribution shift or unseen corruption types.
The paper suggests that training data diversity, external validation,
and uncertainty estimation should become more central in future research.
It also raises the issue that deep networks may produce visually plausible outputs
even when the underlying restoration process is not fully trustworthy.
"""

paper3_long = paper3 + """
A key argument of the paper is that neither traditional reconstruction
nor purely deep learning-based refinement should be viewed as universally superior.
Instead, the best practical system may depend on the balance between reliability,
speed, interpretability, and deployment constraints.
The authors propose that hybrid methods may be especially useful
in settings where data are limited or where condition shift is expected.
"""

In [34]:
very_long_prompt = f"""
아래 3개의 긴 연구 설명을 모두 읽고,
단순 요약이 아니라 '연구 흐름 전체를 이해한 사람처럼' 분석해줘.

반드시 포함할 것:
1. 세 연구를 하나의 역사적 흐름처럼 설명
2. 전통적 방법, 학습 기반 방법, 혼합 방법의 철학적 차이
3. 각 방법의 실제 장단점을 복원 문제 관점에서 분석
4. 신뢰성, 속도, 데이터 요구량, 일반화, 해석 가능성 측면 비교
5. 실제 연구자가 다음 프로젝트를 설계한다면 어떤 방향이 가장 유망한지 제안
6. 마지막에 논문 introduction의 related background처럼 2문단으로 정리

[연구 1]
{paper1_long}

[연구 2]
{paper2_long}

[연구 3]
{paper3_long}
"""

run_test(
    "아주 긴 입력 비교 테스트",
    very_long_prompt,
    max_tokens=1800,
    temperature=0.2
)

TEST: 아주 긴 입력 비교 테스트
## 분석

### 역사적 흐름

세 연구는 복원 문제에 대한 다양한 접근 방식을 보여주는 역사적 흐름을 형성하고 있습니다. 첫 번째 연구는 전통적인 최적화 기반 복원 프레임워크를 제안하며, 이는 복원 문제의 안정성과 구조적 패턴 보존에 중점을 둡니다. 두 번째 연구는 심층 학습 기반 접근 방식을 제시하며, 이는 빠른 복원 속도와 높은 성능을 강조합니다. 세 번째 연구는 두 접근 방식의 장점을 결합한 하이브리드 전략을 탐구하며, 신뢰성, 속도, 해석 가능성 간의 균형을 모색합니다.

### 철학적 차이

- **전통적 방법**: 최적화 기반 접근 방식으로, 데이터 충실성 및 정규화 항을 사용하여 구조적 패턴을 보존하고 잡음을 억제하는 것을 목표로 합니다. 이 방법은 해석 가능성이 높고 안정적이지만, 계산 비용이 높고 매개변수 조정이 어려울 수 있습니다.
- **학습 기반 방법**: 심층 학습을 사용하여 복원 작업을 직접 학습하는 접근 방식으로, 빠른 복원 속도와 높은 성능을 제공합니다. 그러나 이 방법은 데이터 분포에 의존하고, 출력의 해석 가능성이 낮으며, 일반화 능력이 제한될 수 있습니다.
- **혼합 방법**: 전통적 방법과 학습 기반 방법의 장점을 결합한 하이브리드 접근 방식으로, 신뢰성, 속도, 해석 가능성 간의 균형을 추구합니다. 이 방법은 제한된 관찰 또는 잡음이 있는 입력 조건에서 우수한 성능을 보이며, 강건성과 해석 가능성을 모두 제공합니다.

### 실제 장단점 (복원 문제 관점)

- **전통적 방법**: 장점으로는 높은 해석 가능성, 안정성, 구조적 패턴 보존이 있습니다. 단점으로는 높은 계산 비용, 느린 속도, 매개변수 조정 어려움이 있습니다.
- **학습 기반 방법**: 장점으로는 빠른 속도, 높은 성능, 잡음 억제가 있습니다. 단점으로는 데이터 분포 의존, 낮은 해석 가능성, 제한된 일반화 능력이 있습니다.
- **혼합 방법**: 장점으로는 높은 신뢰성, 속도, 해석 가능성, 강건성이 있습니다. 

# Mistral 정리

## 1. Mistral이란?
Mistral은 **Mistral AI**가 만든 생성형 AI 모델 계열이다.  
하나의 모델 이름이 아니라, **여러 언어모델과 관련 플랫폼 전체**를 가리킨다.

---

## 2. 왜 많이 언급되는가?
- **작은 크기 대비 성능이 좋다**
- **오픈 가중치 모델**이 있어 직접 실험하기 좋다
- **로컬 실행, 자체 배포, 클라우드 배포**가 비교적 유연하다
- 범용 모델뿐 아니라 **코드, 멀티모달, 문서 처리**까지 확장하고 있다

---

## 3. 대표 모델 계열

### (1) Mistral
기본적인 범용 언어모델 계열이다.  
질문응답, 요약, 번역, 글쓰기 등에 사용할 수 있다.

### (2) Mixtral
**MoE(Mixture of Experts)** 구조를 사용하는 계열이다.  
입력마다 일부 expert만 활성화하여 **효율성과 성능을 함께 노리는 방식**이다.

### (3) Ministral
더 가볍고 작은 계열이다.  
자원이 적은 환경이나 빠른 추론이 필요한 경우에 적합하다.

### (4) Codestral
코드 생성과 프로그래밍 작업에 특화된 계열이다.

### (5) Pixtral
이미지와 텍스트를 함께 다루는 **멀티모달 모델** 계열이다.

---

## 4. 기술적으로 어떻게 이해하면 되나?

### Dense 모델
모든 파라미터를 고르게 사용하는 일반적인 구조

### MoE 모델
여러 expert 중 일부만 선택적으로 사용하는 구조  
Mixtral이 대표적이다.

---

## 5. 장점
- 비교적 **가볍고 효율적**
- 연구나 교육용 실습에 적합
- 로컬/온프레미스 배포 가능성이 큼
- 모델 선택 폭이 넓음
- 코드, 문서, 이미지 등으로 확장 가능

---

## 6. 한계
- 모델 이름과 라인업이 자주 바뀔 수 있다
- 모든 모델이 완전히 오픈인 것은 아니다
- 최고 성능만 보면 더 큰 폐쇄형 모델과 비교가 필요하다

---

## 7. 다른 모델과 비교

### GPT 계열과 비교
- GPT: 주로 상용 API 중심
- Mistral: 오픈 가중치와 자체 배포 측면에서 강점

### Llama와 비교
- 둘 다 실습과 연구에서 많이 사용됨
- Mistral은 특히 **효율성**과 **Mixtral 같은 MoE 구조**로 자주 주목받음

---

## 8. 언제 쓰면 좋은가?
- LLM을 직접 실습해보고 싶을 때
- 로컬 환경에서 돌려보고 싶을 때
- 작은 모델과 큰 모델을 비교해보고 싶을 때
- MoE 구조를 실제 모델로 이해하고 싶을 때
- 코드 모델이나 멀티모달 모델까지 확장해서 배우고 싶을 때



## Mistral 실습

- hugging face에서 "mistralai/Mistral-7B-Instruct-v0.3를 불러와서 적용

In [18]:
# 모델 불러오기
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("모델 로드 완료")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk.


모델 로드 완료


In [19]:
#가장 기본적인 생성 실습
prompt = "Explain what Mistral is in simple terms."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Explain what Mistral is in simple terms.

Mistral is a high-speed wind that originates in the Rhone Valley in France, specifically in the region of the Mistral Mountains. It is known for its strong, cold, and dry characteristics, often blowing at speeds of 50 to 75 miles per hour. The wind is named after the Mistral River, which is said to have given the wind its name due to its strong and relentless nature. Mistral winds can have a significant impact on weather patterns in southern Europe, and are often associated with clear skies and cold temperatures. They can also be quite disruptive, causing problems for transportation, agriculture, and other outdoor activities.


In [20]:
# 생성 함수 만들기
def generate_text(prompt, max_new_tokens=150, temperature=0.7, top_p=0.9):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [21]:
print(generate_text("What is a large language model?"))

What is a large language model?

A large language model is a type of artificial intelligence (AI) model that has been trained on a vast amount of text data. These models are designed to understand and generate human-like text, making them useful for a wide range of natural language processing tasks, such as language translation, text summarization, and answering questions.

Large language models are typically trained on large datasets, such as the entirety of the internet, or a significant portion of it. This allows them to learn patterns and structures in language, enabling them to generate coherent and contextually appropriate responses.

One of the most well-known large language models is the one developed by OpenAI, called ChatGPT. However, there are many other large


In [22]:
# 프롬프트 바꿔보기
prompt1 = "What is Mistral?"
prompt2 = "Explain Mistral for a beginner."
prompt3 = "Explain Mistral in 3 bullet points."
prompt4 = "Explain Mistral as if you are teaching undergraduate students."

print(generate_text(prompt1))
print("="*80)
print(generate_text(prompt2))
print("="*80)
print(generate_text(prompt3))
print("="*80)
print(generate_text(prompt4))

What is Mistral?

Mistral is a type of wind that blows in the Mediterranean region, particularly in France. It is known for its strong, cold, and dry gusts that can reach speeds of up to 100 km/h (62 mph). The Mistral wind is caused by a high-pressure system over the Atlantic Ocean and a low-pressure system over the Mediterranean Sea, which creates a strong pressure gradient that drives the wind across the region.

Mistral is a very dry wind, which means it doesn't carry much moisture. It is also known for its clear skies and sunny weather, as it tends to blow away any clouds or fog. The Mistral wind can
Explain Mistral for a beginner.

Mistral is a high-level programming language developed by SAP, a German software company, in the early 1990s. It is an object-oriented, imperative, and procedural language that is designed to be easy to learn and use. Mistral is primarily used for developing business applications, such as enterprise resource planning (ERP) systems, and it is designed to

- few shot prompting 실습


In [23]:
few_shot_prompt = """
Task: Classify the sentiment of the sentence as Positive or Negative.

Sentence: The lecture was clear and helpful.
Answer: Positive

Sentence: The explanation was too fast and confusing.
Answer: Negative

Sentence: The examples were practical and easy to follow.
Answer:
"""

print(generate_text(few_shot_prompt, max_new_tokens=30, temperature=0.3))


Task: Classify the sentiment of the sentence as Positive or Negative.

Sentence: The lecture was clear and helpful.
Answer: Positive

Sentence: The explanation was too fast and confusing.
Answer: Negative

Sentence: The examples were practical and easy to follow.
Answer:
Positive

Sentence: The instructor was not engaging and the material was boring.
Answer: Negative

Sentence: The


In [24]:
#요약 실습
text = """
Mistral is a family of language models developed by Mistral AI.
Some models are designed for general text generation, while others are
specialized for coding, multimodal understanding, or document processing.
Mistral became well known for combining strong performance with relatively
efficient model design.
"""

prompt = f"Summarize the following text in 2 sentences:\n\n{text}"
print(generate_text(prompt, max_new_tokens=80, temperature=0.3))

Summarize the following text in 2 sentences:


Mistral is a family of language models developed by Mistral AI.
Some models are designed for general text generation, while others are
specialized for coding, multimodal understanding, or document processing.
Mistral became well known for combining strong performance with relatively
efficient model design.


The Mistral AI company has created a series of language models called Mistral, which are used for various purposes such as general text generation, coding, multimodal understanding, and document processing. These models are recognized for their ability to deliver strong performance while maintaining an efficient model design.


In [25]:
# 번역 실습
prompt = "Translate the following sentence into Korean: Mistral is an efficient language model family."
print(generate_text(prompt, max_new_tokens=80, temperature=0.3))

Translate the following sentence into Korean: Mistral is an efficient language model family.

The translation of "Mistral is an efficient language model family" into Korean is "Mistral은 효율적인 언어 모델 가족입니다." (Mistral-eun hyeoljineun eon-eui eon-ja-mok gwajok-imn


In [26]:
prompt = "Translate the following Korean sentence into English: 미스트랄은 효율적인 언어모델 계열이다."
print(generate_text(prompt, max_new_tokens=80, temperature=0.3))

Translate the following Korean sentence into English: 미스트랄은 효율적인 언어모델 계열이다.

The mistral is a type of efficient language model.

Here's the breakdown:

1. 미스트랄 (mistral) - This is the subject of the sentence. It's a type of language model.

2. 은 (은) - This is a topic marker in Korean, similar to "is" or "


In [27]:
# 출력 옵션
prompt = "Explain attention in a simple way."

print("temperature = 0.2")
print(generate_text(prompt, temperature=0.2))
print("="*80)

print("temperature = 0.8")
print(generate_text(prompt, temperature=0.8))
print("="*80)

print("temperature = 1.2")
print(generate_text(prompt, temperature=1.2))

temperature = 0.2
Explain attention in a simple way.

Attention is the ability to focus on a specific task, person, or thing while ignoring other distractions. It's like a spotlight that helps us to focus on what's important and filter out what's not. In the context of artificial intelligence, attention mechanisms are used to help models focus on relevant parts of the input data when making predictions.

For example, if you're reading a long paragraph, your attention might jump from one word or phrase to another as you follow the main idea. Similarly, an AI model with an attention mechanism might focus on certain words or phrases in a sentence when making a prediction about the sentence's meaning. This helps the model to be more efficient and accurate in its
temperature = 0.8
Explain attention in a simple way.

Attention is the ability to focus on a specific thing, whether it's a task, a person, or an object, while ignoring other distracting stimuli. It helps us to concentrate, process

- 낮은 temperature: 더 안정적
- 높은 temperature: 더 다양하지만 산만할 수 있음

In [28]:
# Instruct 모델답게 지시형 프롬프트 넣기
prompt = """
You are a helpful AI tutor.
Explain the difference between GPT and Mistral in very simple terms.
Use only 5 sentences.
"""

print(generate_text(prompt, max_new_tokens=120, temperature=0.5))


You are a helpful AI tutor.
Explain the difference between GPT and Mistral in very simple terms.
Use only 5 sentences.

GPT (Generative Pre-trained Transformer) is a type of AI model developed by OpenAI, designed to generate human-like text based on the data it was trained on. It's used in various applications like answering questions, writing essays, and translating languages.

Mistral, on the other hand, is a different AI model developed by Mistral AI. It's specifically designed for conversational AI tasks, focusing on understanding and responding to user queries in a more personalized and contextually aware manner.

While GPT can generate text, Mist


In [29]:
#채팅 스타일 실습
chat_prompt = """
User: I am new to large language models.
Assistant: That's okay. I can explain simply.

User: What is Mistral?
Assistant:
"""

print(generate_text(chat_prompt, max_new_tokens=120, temperature=0.6))


User: I am new to large language models.
Assistant: That's okay. I can explain simply.

User: What is Mistral?
Assistant:
Mistral is a large language model developed by Mistral AI, a French AI company. It's designed to understand and generate human-like text in multiple languages. Similar to models like me, it can answer questions, write essays, translate languages, and much more. However, it's important to note that while these models are powerful, they don't truly understand or have consciousness like humans do. They generate responses based on patterns they've learned during their training.


In [30]:
# 감성 분석 프롬프트 실습
prompt = """
Classify the sentiment as Positive, Negative, or Neutral.

Sentence: The class was informative but a little fast.
Answer:
"""
print(generate_text(prompt, max_new_tokens=20, temperature=0.2))


Classify the sentiment as Positive, Negative, or Neutral.

Sentence: The class was informative but a little fast.
Answer:
Neutral

Explanation:
The sentence contains both positive ("informative")


In [31]:
# 한국어 설명 실습
prompt = "양자컴퓨팅이 무엇인지 학부생 수준에서 한국어로 설명해줘."
print(generate_text(prompt, max_new_tokens=150, temperature=0.5))

양자컴퓨팅이 무엇인지 학부생 수준에서 한국어로 설명해줘.

양자 컴퓨팅(Quantum Computing)은 전통적인 컴퓨팅과 달리, 자연스러운 혼자 행동을 가지고 있는 물질의 특성을 활용한 컴퓨팅 기술입니다. 이러한 물질은 전자이며, 이들은 동시에 여러 상태를 가질 수 있습니다. 이러한 특성을 활용하면 대용량 데이터를


In [32]:
# 수학 설명 실습
prompt = "Explain eigenvalues in a simple way for undergraduate students."
print(generate_text(prompt, max_new_tokens=150, temperature=0.5))

Explain eigenvalues in a simple way for undergraduate students.

Eigenvalues are special numbers that tell you how much a system changes when you apply a specific transformation to it. In simpler terms, imagine you have a system, like a ball rolling on a surface. The eigenvalue for this system would be the amount the ball moves when you give it a push in a certain direction.

To find the eigenvalues, you need to solve an equation called the characteristic equation. This equation is like a puzzle that gives you the possible amounts of movement (or eigenvalues) for your system.

For example, if you have a system with two possible directions for the ball to move, you would have two eigenvalues. Let's say the eigenvalues are 2 and -1.


##  Mistral API

### 장점
- 설치와 환경 의존성이 적다
- 로컬 GPU 없이도 바로 실습 가능하다
- 공식 SDK로 빠르게 호출할 수 있다
- chat completions, function calling, embeddings 등으로 쉽게 확장할 수 있다

### 단점
- API 키와 결제 설정이 필요할 수 있다
- 호출량이 늘면 비용이 든다
- 로컬 모델처럼 내부 동작을 직접 만지기는 어렵다

### 언제 API가 좋은가?
- 빨리 결과를 보고 싶을 때
- GPU 환경이 없을 때
- 프롬프트 실험을 먼저 해보고 싶을 때
- 이후 RAG, function calling, agent 방식으로 확장하고 싶을 때

### 언제 로컬 실행이 좋은가?
- 모델을 직접 내려받아 구조를 공부하고 싶을 때
- 오프라인 실습이 필요할 때
- 추론 과정을 더 세밀하게 통제하고 싶을 때


API space: https://docs.mistral.ai/api

In [19]:
!pip install mistralai

In [20]:
import os
# os.environ["MISTRAL_API_KEY"] = "여기에_발급받은_API_KEY"
os.environ["MISTRAL_API_KEY"] = "qh7Dn3VlZ2aaTJnrPhEOp7kxUBMs3Bam"

In [24]:
import os
from mistralai.client import Mistral

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {"role": "user", "content": "Explain Mistral in simple terms."}
    ]
)

print(response.choices[0].message.content)

Mistral is a **large language model** developed by Mistral AI, a cutting-edge AI lab based in France. Think of it like a super-smart computer program that can understand and generate human-like text based on the information it has been trained on.

### Simple Breakdown:
1. **What it does**:
   - It can **read, write, and answer questions** in a way that sounds natural, like a human.
   - It can **summarize articles**, **write stories**, **explain concepts**, **translate languages**, and even **help with coding**.

2. **How it works**:
   - It’s trained on **huge amounts of text** (books, articles, websites, etc.) to learn patterns in language.
   - When you ask it a question or give it a prompt, it predicts the most likely next words to form a coherent response.

3. **Why it’s special**:
   - It’s **open-source** (some versions are free to use and modify).
   - It’s **efficient**—it can run on less powerful hardware compared to some other big AI models.
   - It’s **multilingual**—it un

In [25]:
# 프롬프트 바꿔보기
response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {"role": "user", "content": "Explain attention in 3 bullet points for beginners."}
    ]
)

print(response.choices[0].message.content)

- **Focus Mechanism**: Attention helps models (like in AI) focus on the most relevant parts of input data (e.g., words in a sentence) when making decisions, similar to how humans pay more attention to key details.

- **Weights & Scores**: It assigns "importance scores" (weights) to different parts of the input, determining how much each part influences the output. For example, in translation, it highlights which words in the source sentence matter most for each word in the target.

- **Self-Attention (Key Idea)**: In models like Transformers, each part of the input (e.g., a word) looks at all other parts to compute its relevance, enabling the model to capture relationships and context dynamically.


In [26]:
# 요약 실습
text = """
Mistral is a family of language models developed by Mistral AI.
Some models are general-purpose, while others support coding,
reasoning, vision, and document processing.
"""

response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {"role": "user", "content": f"Summarize this in 2 sentences:\n\n{text}"}
    ]
)

print(response.choices[0].message.content)

Mistral AI has created a family of language models, some of which are designed for general use while others specialize in coding, reasoning, vision, and document processing. These models cater to a variety of applications and tasks.


## 1. Gemma란?
Gemma는 **Google DeepMind가 개발하는 오픈 웨이트(open-weights) 생성형 AI 모델 계열**

하나의 단일 모델이 아니라, **텍스트 생성 모델, 멀티모달 모델, 임베딩 모델, 함수 호출 특화 모델, 의료 특화 모델**까지 포함하는 **모델 패밀리**로 이해. 

공식 문서에서는 Gemma를 **Gemini를 만드는 데 사용된 연구와 기술을 바탕으로 만든 경량 오픈 모델 계열**이라고 설명.

---

## 2. 왜 많이 언급되는가?

- **오픈 웨이트 모델**이라 연구와 실습에 활용하기 좋다
- **상대적으로 작은 크기에서도 성능이 강한 편**이다
- **노트북, 워크스테이션, 모바일 기기까지 폭넓은 배포**를 지향
- 텍스트뿐 아니라 **이미지 이해, 임베딩, 함수 호출, 의료 특화**까지 계속 확장되고 있다. 
---

## 3. Gemma를 한 문장으로 설명하면
**“Gemini 계열의 연구 성과를 바탕으로, 비교적 가볍고 배포 유연성이 높은 오픈 웨이트 모델 패밀리”**라고 이해하면 가장 정확. 

공식 설명도 Gemma를 클라우드 서버뿐 아니라 **개인용 컴퓨터와 휴대기기에서도 실행 가능한 모델군**으로 소개. 

---

## 4. 대표 모델 계열

### (1) Gemma
가장 기본적인 핵심 계열이다.  
질문응답, 요약, 추론, 일반적인 텍스트 생성 작업에 사용된다. 공식 문서에서는 Gemma를 다양한 생성 작업에 쓸 수 있고, **오픈 웨이트와 상업적 사용 허용 조건**을 제공한다고 설명. 

### (2) Gemma 2
Gemma 2는 텍스트 중심의 후속 계열로, Hugging Face 모델 카드에서는 **decoder-only, text-to-text LLM**으로 설명된다. instruction-tuned 버전도 제공되어 대화형 사용에 적합.

### (3) Gemma 3
Gemma 3는 멀티모달 방향으로 확장된 핵심 세대다.  
공식 문서와 모델 카드에 따르면 **텍스트와 이미지 입력을 처리하고 텍스트를 생성**할 수 있으며, **128K 컨텍스트 윈도우**와 **140개 이상 언어 지원**을 강조. 

또한 단일 GPU나 TPU, 심지어 노트북·스마트폰 같은 자원 제한 환경도 주요 타깃으로 제시. 

### (4) Gemma 3n
Gemma 3n은 특히 **온디바이스(on-device)** 사용을 강하게 의식한 계열이다.  
공식 문서에 따르면 휴대폰, 태블릿, 노트북에서 로컬로 실행하는 것을 목표로 하며, **MatFormer 구조**를 사용해 큰 모델 안에 더 작은 하위 모델이 중첩된 형태를 제공한다. 이 구조는 계산 비용과 지연 시간을 줄이는 데 도움을 준다. 

### (5) Gemma 4
Gemma 4는 2026년 3월 말 공개된 최신 세대다.  
Google AI for Developers와 DeepMind 페이지에서는 Gemma 4를 **고급 추론(advanced reasoning)**과 **agentic workflows**를 위한 모델군으로 설명하고, 하드웨어 요구사항별로 여러 아키텍처 변형을 제공한다고 소개. 

DeepMind 블로그는 Gemma 4가 **intelligence-per-parameter**를 강하게 내세우며, 더 큰 오픈 모델들과 경쟁하는 성능을 주장. 
---

## 5. 파생/특화 모델

### (1) CodeGemma
Gemma 기반의 **코드 특화 모델** 계열.  
공식 Hugging Face Google 조직 페이지에서 Gemma 패밀리의 일부로 소개된다. 프로그래밍, 코드 생성, 코드 보조 실습에 적합한 계열로 이해하면 된다. 

### (2) RecurrentGemma
일반적인 Transformer와는 다른 **recurrent architecture**를 활용하는 Gemma 계열.  
이는 “Gemma는 단지 한 가지 구조만 고정적으로 쓰는 모델군이 아니다”라는 점을 보여준다. 

### (3) PaliGemma
경량 **비전-언어 모델(VLM)** 계열이다.  
이미지와 텍스트를 함께 다루는 응용에 연결된다. 

### (4) FunctionGemma
FunctionGemma는 2026년 3월 공개된 **함수 호출(function calling) 특화 경량 모델**이다.  
공식 모델 카드에 따르면 직접적인 대화 모델이라기보다, **전문화된 function calling 모델을 만들기 위한 기반 모델** 성격이 강하다. 

### (5) EmbeddingGemma
EmbeddingGemma는 **임베딩 전용 오픈 모델**이다.  
즉, 생성형 텍스트 출력보다도 검색, RAG, 유사도 계산, 문서 인덱싱 같은 작업에 더 적합하다. 

### (6) MedGemma
MedGemma는 의료 분야 특화 계열로, 공식 페이지에서는 **의료 텍스트와 의료 이미지 이해에 최적화된 오픈 모델 컬렉션**으로 소개된다. 의료 AI 연구자 입장에서는 Gemma 계열 중 특히 주목할 만한 확장이다. 

### (7) TranslateGemma
TranslateGemma는 이름 그대로 **번역 특화 모델**이다. 2026년 1월 릴리스 문서에 등재되어 있다. 

---

## 6. 기술적으로 어떻게 이해하면 되나?

### (1) 오픈 웨이트
Gemma는 흔히 “오픈 소스 모델”처럼 불리지만, 더 정확히는 **오픈 웨이트 모델**로 이해하는 것이 좋다.  
즉, 가중치가 공개되어 직접 내려받고 실행·튜닝·배포할 수 있지만, 사용 조건과 라이선스는 항상 별도로 확인해야 한다. 공식 문서는 Gemma가 **responsible commercial use**를 허용한다고 설명한다. 

### (2) 경량성과 배포 유연성
Gemma는 처음부터 **대형 클라우드 서버 전용**보다는, 더 넓은 개발 환경을 염두에 두고 설계된 느낌이 강하다. 공식 페이지가 반복해서 **laptops, desktops, phones, workstations**를 언급하는 이유가 바로 이것이다. {index=16}

### (3) 멀티모달 확장
최신 Gemma 계열은 텍스트만 다루는 모델이 아니다.  
Gemma 3와 Gemma 4는 이미지 입력을 처리하는 멀티모달 계열로 소개되고, Gemma 4 관련 Hugging Face 카드에서는 일부 소형 모델에서 오디오도 지원된다고 안내한다. 

### (4) 장문맥 처리
Gemma 3는 공식 모델 카드 기준으로 **128K 컨텍스트 윈도우**, Gemma 4는 공식 Hugging Face 카드 기준으로 **최대 256K 컨텍스트 윈도우**를 지원한다. 그래서 긴 문서 요약, 긴 대화, 문서 기반 QA 같은 작업에서 활용성이 높다. 
---

## 7. 장점

### (1) 실습 친화적이다
Gemma는 공식 문서, Hugging Face 배포, Kaggle 연계, 시작 가이드가 잘 마련되어 있어 **교육용 실습**에 적합하다. 

### (2) 작은 모델 대비 성능이 강하다
DeepMind는 Gemma 계열의 핵심 장점으로 **intelligence-per-parameter**를 지속적으로 강조한다. 즉, “파라미터 수 대비 성능 효율”이 중요한 장점이라는 뜻이다. 

### (3) 연구 확장성이 크다
기본 LLM뿐 아니라 멀티모달, 임베딩, 의료, 번역, 함수 호출 등으로 파생 계열이 많아 **연구 주제 확장**이 쉽다. 

### (4) 로컬 실행 가능성이 높다
Gemma 3, Gemma 3n, Gemma 4 페이지는 모두 **로컬 실행** 또는 **소비자 하드웨어 친화성**을 중요한 가치로 내세운다. 연구실 서버, 개인 워크스테이션, 일부 모바일 환경에서도 시도 가능한 점이 큰 장점이다. 
---

## 8. 한계

### (1) 모델 라인업이 빨리 변한다
Gemma는 릴리스 주기가 빠른 편이다. 공식 릴리스 문서만 봐도 2025년 말부터 2026년 초까지 Gemma 4, TranslateGemma, MedGemma 1.5, FunctionGemma 등 새 계열이 계속 추가되고 있다. 그래서 오래된 블로그 글만 보면 최신 구조를 놓치기 쉽다. 

### (2) 모델마다 목적이 많이 다르다
같은 Gemma 계열이라도 어떤 것은 대화형, 어떤 것은 임베딩, 어떤 것은 의료 특화, 어떤 것은 함수 호출용이다. 따라서 “Gemma”라는 이름만 보고 곧바로 같은 용도라고 생각하면 안 된다. 

### (3) 최상위 폐쇄형 모델과는 목적이 다를 수 있다
Gemma는 단순히 “가장 큰 성능 하나만 추구하는 모델”이라기보다, **배포 유연성·효율·접근성**을 함께 중시하는 계열이다. 따라서 사용 목적에 따라 Gemini, GPT, Claude 같은 폐쇄형 상용 모델과 역할이 달라질 수 있다. 이 점은 DeepMind와 Google 개발자 문서의 포지셔닝에서도 드러난다. 

---

## 9. 다른 모델과 비교

### GPT 계열과 비교
GPT 계열은 대체로 상용 API 중심으로 접하는 경우가 많다.  
반면 Gemma는 **오픈 웨이트 기반으로 직접 내려받아 실험하고 튜닝할 수 있는 점**이 큰 차이다.

### Llama, Mistral과 비교
Gemma, Llama, Mistral은 모두 연구·실습에서 자주 비교되는 오픈 계열 모델군이다.  
그중 Gemma는 특히 **Google 생태계**, **Gemini 기반 연구 기술**, **온디바이스 지향성**, **특화 파생 모델의 다양성**이 차별점이라고 볼 수 있다. 

---

## 10. 언제 Gemma를 쓰면 좋은가?
Gemma는 다음 상황에서 특히 잘 맞는다.

- **오픈 웨이트 LLM을 실습하고 싶을 때**
- **로컬에서 돌아가는 비교적 가벼운 모델이 필요할 때**
- **텍스트뿐 아니라 이미지 이해까지 연결하고 싶을 때**
- **RAG용 임베딩 모델까지 함께 보고 싶을 때**
- **의료 특화 모델이나 함수 호출 특화 모델을 실험하고 싶을 때** 

---

## 11. 핵심 한 줄 정리
**Gemma는 Google DeepMind가 개발한 오픈 웨이트 생성형 AI 모델 패밀리로, 경량성·배포 유연성·멀티모달 확장성을 강점으로 하며 텍스트 LLM에서 의료·임베딩·함수 호출까지 넓게 확장된 생태계를 갖고 있다.** 

In [33]:
!pip install -U transformers accelerate sentencepiece

In [2]:
# 모델 불러오기
## 너무 큰 모델보다는 작은 instruction 모델로 시작
# 모델 불러오기
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-2-2b-it"

# MPS 우선
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("사용 device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device.type == "mps" else torch.float32
)

model = model.to(device)

print("모델 로드 완료")
print("현재 device:", next(model.parameters()).device)

사용 device: mps


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

모델 로드 완료
현재 device: mps:0


In [3]:
# 가장 기본적인 생성

prompt = "Explain what Gemma is in simple terms."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

Explain what Gemma is in simple terms.

Gemma is like a really smart chatbot. You can talk to her, ask her questions, and she'll try her best to answer you. She's been trained on a massive amount of text data, so she knows a lot about different things. 

Think of her as a super-powered version of a search engine, but instead of just giving you links, she can actually understand what you're asking and give you a helpful answer. 

Here are some things Gemma can


In [4]:
# 생성 함수 만들기
def generate_text(prompt, max_new_tokens=100, temperature=0.7, top_p=0.9):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [5]:
print(generate_text("What is a language model?"))

What is a language model?

A language model is a type of artificial intelligence (AI) system that is trained on a massive dataset of text and code. This training allows the model to understand and generate human-like text in response to prompts or questions. 

Here's a breakdown of what makes a language model:

**What it does:**
* **Predicts the next word:**  Given a sequence of words, a language model can predict the most likely next word. This is the basis for many language


In [6]:
# 프롬프트를 바꿔보며 비교하기
prompt1 = "What is Gemma?"
prompt2 = "Explain Gemma for beginners."
prompt3 = "Explain Gemma in 3 bullet points."
prompt4 = "Explain Gemma as if you are teaching undergraduate students."

print(generate_text(prompt1))
print("=" * 80)
print(generate_text(prompt2))
print("=" * 80)
print(generate_text(prompt3))
print("=" * 80)
print(generate_text(prompt4))

What is Gemma?

Gemma is an open-weight, large language model chatbot developed by Google DeepMind. 

**Key Features:**

* **Open-weight:** This means the model's code and data are publicly accessible, allowing for greater transparency and collaboration.
* **Large Language Model:** Gemma is trained on a massive dataset of text and code, enabling it to generate human-like text, translate languages, write different kinds of creative content, and answer your questions in an informative way.
*
Explain Gemma for beginners.

Gemma is an open-source AI assistant. Imagine it as a very smart chatbot that you can chat with about anything! 

**Here's what makes Gemma special:**

* **Open-source:**  This means anyone can see how Gemma works and even contribute to its development. It's like an open book for AI!
* **Accessible:** You can use Gemma directly without needing special software or a subscription. It's available for everyone to explore.
* **Versatile
Explain Gemma in 3 bullet points.

* **

In [7]:
#temperature 비교 실습

prompt = "Explain attention in simple terms."

print("temperature = 0.2")
print(generate_text(prompt, temperature=0.2))
print("=" * 80)

print("temperature = 0.7")
print(generate_text(prompt, temperature=0.7))
print("=" * 80)

print("temperature = 1.0")
print(generate_text(prompt, temperature=1.0))

temperature = 0.2
Explain attention in simple terms.

Imagine you're at a party. There are lots of people talking, music playing, and food everywhere. It's easy to get overwhelmed and not pay attention to anything. But if you focus on one person, maybe a friend you're talking to, you can really hear what they're saying and understand what they're trying to communicate.

That's kind of like attention. It's the ability to focus on something specific while ignoring everything else. It's
temperature = 0.7
Explain attention in simple terms.

**Imagine you're at a party, surrounded by lots of people.**

* **Your attention is like a spotlight.** It's focused on one thing at a time. 
* **You can only see one person at a time, even if they're talking to you.** Your brain filters out all the other distractions and focuses on the person who you're currently talking to.

**That's attention in action!**

Here's a breakdown of key points
temperature = 1.0
Explain attention in simple terms.

**Imagin

- temperature가 낮으면 더 안정적이다
- temperature가 높으면 더 다양하지만 흔들릴 수 있다

In [8]:
text = """
Gemma is a family of open-weight language models developed by Google DeepMind.
It is designed to support efficient deployment and practical use across a range
of devices and environments. Some Gemma models focus on text generation, while
others extend to multimodal or specialized tasks.
"""

prompt = f"Summarize the following text in 2 sentences:\n\n{text}"
print(generate_text(prompt, max_new_tokens=80, temperature=0.3))

Summarize the following text in 2 sentences:


Gemma is a family of open-weight language models developed by Google DeepMind.
It is designed to support efficient deployment and practical use across a range
of devices and environments. Some Gemma models focus on text generation, while
others extend to multimodal or specialized tasks.
The Gemma project aims to democratize access to large language models, making them
available to a wider audience.


**In short:** Gemma is a family of open-weight language models developed by Google DeepMind, designed for efficient deployment and accessibility across various devices and tasks. 



In [9]:
prompt = f"Summarize the following text for undergraduate students in 3 bullet points:\n\n{text}"
print(generate_text(prompt, max_new_tokens=120, temperature=0.3))

Summarize the following text for undergraduate students in 3 bullet points:


Gemma is a family of open-weight language models developed by Google DeepMind.
It is designed to support efficient deployment and practical use across a range
of devices and environments. Some Gemma models focus on text generation, while
others extend to multimodal or specialized tasks.
Here's a breakdown of the key features:
* **Open-weight:**  Gemma's weights are publicly available, allowing researchers and developers to access, modify, and build upon the model.
* **Efficient Deployment:** Gemma models are designed to run efficiently on various hardware, including CPUs, GPUs, and even mobile devices.
* **Multimodal Capabilities:** Some Gemma models can process and generate both text and images, enabling them to handle tasks like image captioning or image-based text generation.


**In short:** Gemma is a powerful, open-source language model that can


In [10]:
# 번역 실습(영어 -> 한국어)
prompt = "Translate the following sentence into Korean: Gemma is an open-weight language model family."
print(generate_text(prompt, max_new_tokens=80, temperature=0.3))

Translate the following sentence into Korean: Gemma is an open-weight language model family.

**Korean Translation:**

Gemma는 오픈 웨이트 언어 모델 가족입니다. 

**Explanation:**

* **Gemma:**  Gemma is the name of the language model.
* **은:**  This is the grammatical particle that connects the subject (Gemma) to the predicate (is an open-weight language model family).
* **오픈 웨이트 언


In [11]:
# 번역 실습(한국어->영어)
prompt = "Translate the following Korean sentence into English: 젬마는 오픈 웨이트 언어모델 계열이다."
print(generate_text(prompt, max_new_tokens=80, temperature=0.3))

Translate the following Korean sentence into English: 젬마는 오픈 웨이트 언어모델 계열이다. 

**Translation:** Gemma is an open-weight language model. 


Let me know if you'd like me to translate any other Korean sentences! 😊 



In [12]:
# 간단한 분류 실습 / 프롬프트를 잘 주면 분류도 가능
prompt = """
Classify the sentiment of the sentence as Positive, Negative, or Neutral.

Sentence: The lecture was very clear and helpful.
Answer:
"""

print(generate_text(prompt, max_new_tokens=20, temperature=0.2))


Classify the sentiment of the sentence as Positive, Negative, or Neutral.

Sentence: The lecture was very clear and helpful.
Answer:
Positive 
**Explanation:**

The sentence expresses a positive sentiment because it highlights the clarity and helpful


In [13]:
samples = [
    "The class was very clear and helpful.",
    "The explanation was too fast and confusing.",
    "The lecture was okay, but the examples were limited."
]

for s in samples:
    prompt = f"""
Classify the sentiment of the sentence as Positive, Negative, or Neutral.

Sentence: {s}
Answer:
"""
    print("입력:", s)
    print(generate_text(prompt, max_new_tokens=20, temperature=0.2))
    print("-" * 60)

입력: The class was very clear and helpful.

Classify the sentiment of the sentence as Positive, Negative, or Neutral.

Sentence: The class was very clear and helpful.
Answer:
Positive 
**Explanation:**

The sentence expresses a positive sentiment. "Clear" and "helpful
------------------------------------------------------------
입력: The explanation was too fast and confusing.

Classify the sentiment of the sentence as Positive, Negative, or Neutral.

Sentence: The explanation was too fast and confusing.
Answer:
Negative 

------------------------------------------------------------
입력: The lecture was okay, but the examples were limited.

Classify the sentiment of the sentence as Positive, Negative, or Neutral.

Sentence: The lecture was okay, but the examples were limited.
Answer:
**Neutral** 

Explanation: 

The sentence expresses a lack of strong positive or negative feelings
------------------------------------------------------------


In [14]:
# few show 프롬프트 실습

few_shot_prompt = """
Task: Classify the sentiment of each sentence.

Sentence: The lecture was easy to follow.
Answer: Positive

Sentence: The explanation was confusing.
Answer: Negative

Sentence: The material was acceptable but not very exciting.
Answer: Neutral

Sentence: The examples were practical and useful.
Answer:
"""

print(generate_text(few_shot_prompt, max_new_tokens=20, temperature=0.2))


Task: Classify the sentiment of each sentence.

Sentence: The lecture was easy to follow.
Answer: Positive

Sentence: The explanation was confusing.
Answer: Negative

Sentence: The material was acceptable but not very exciting.
Answer: Neutral

Sentence: The examples were practical and useful.
Answer:
Positive

Sentence: I am so frustrated with this project.
Answer: Negative

Sentence: The


In [15]:
# 채팅 스타일 실습
chat_prompt = """
User: I am new to large language models.
Assistant: That's okay. I will explain simply.

User: What is Gemma?
Assistant:
"""

print(generate_text(chat_prompt, max_new_tokens=120, temperature=0.6))


User: I am new to large language models.
Assistant: That's okay. I will explain simply.

User: What is Gemma?
Assistant:
Gemma is a large language model, which means it's a computer program that's really good at understanding and generating human language. 

Think of it like a super smart chatbot. You can ask it questions, give it instructions, or just chat with it about anything. 

Gemma is still under development, but it's already able to do some pretty amazing things, like write stories, translate languages, and answer your questions in a comprehensive and informative way. 

Is there anything else you'd like to know about Gemma? 



In [16]:
# 질문을 바꿔가며 할 수 있다.
chat_prompt = """
User: I am studying machine learning.
Assistant: Great. Ask me anything.

User: What is the difference between Gemma and GPT?
Assistant:
"""

print(generate_text(chat_prompt, max_new_tokens=150, temperature=0.6))


User: I am studying machine learning.
Assistant: Great. Ask me anything.

User: What is the difference between Gemma and GPT?
Assistant:
That's a great question!  Both Gemma and GPT are large language models, but they have some key differences. 

**Gemma:**

* **Open-source:** Gemma is an open-source model, meaning its code and data are publicly available. This allows researchers and developers to understand, modify, and improve the model.
* **Focus on accessibility:** Gemma is designed to be accessible to a wide range of users, including those without access to powerful hardware or technical expertise. 
* **Smaller size:**  Gemma is typically a smaller model than GPT, making it more efficient to run and requiring less computational power. 

**GPT:**

* **Closed-source:** GPT is a proprietary model developed by OpenAI.


### generate 없이 한 토큰씩 생성하는 실습


In [17]:
# 모델 불러오기

# 모델 불러오기
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-2-2b-it"

# MPS 우선
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("사용 device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device.type == "mps" else torch.float32
)

model = model.to(device)

print("모델 로드 완료")
print("현재 device:", next(model.parameters()).device)

사용 device: mps


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

모델 로드 완료
현재 device: mps:0


In [18]:
# 입력을 토큰으로 바꾸기
prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to(model.device)

print("input_ids shape:", input_ids.shape)
print("input_ids:", input_ids)
print("토큰:", tokenizer.convert_ids_to_tokens(input_ids[0]))

#모델은 문자열이 아니라 토큰 id 시퀀스를 입력을 받음

input_ids shape: torch.Size([1, 6])
input_ids: tensor([[   2,  651, 6037,  576, 6081,  603]], device='mps:0')
토큰: ['<bos>', 'The', '▁capital', '▁of', '▁France', '▁is']


In [19]:
# 모델 출력 확인
## 모델은 각 위치마다 "다음 토큰 후보 점수"를 담은 logits를 출력

with torch.no_grad():
    outputs = model(input_ids=input_ids)

logits = outputs.logits

print("logits shape:", logits.shape)
#logits.shape = (batch_size, seq_len, vocab_size)

logits shape: torch.Size([1, 6, 256000])


In [20]:
next_token_logits = logits[:, -1, :]
print("next_token_logits shape:", next_token_logits.shape)
# 현재 문장 다음에 올 수 있는 모든 토큰의 점수를 담고 있다.

next_token_logits shape: torch.Size([1, 256000])


In [21]:
# 가장 단순한 방식: Greedy decoding
## 가장 점수가 각 토큰 하나를 고름.
next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
print("next_token_id:", next_token_id)

print("선택된 토큰:", tokenizer.decode(next_token_id[0]))

next_token_id: tensor([[7127]], device='mps:0')
선택된 토큰:  Paris


In [22]:
# 직접 한 토큰 붙이기
generated_ids = torch.cat([input_ids, next_token_id], dim=1)

print("새로운 shape:", generated_ids.shape)
print("현재 문장:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))
# 원래 입력 뒤에 모델이 예측한 다음 토큰이 하나 추가됨.

새로운 shape: torch.Size([1, 7])
현재 문장:
The capital of France is Paris


In [23]:
# 반복문으로 여러 토큰으로 생성
prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

generated_ids = input_ids.clone()

max_new_tokens = 20

for step in range(max_new_tokens):
    with torch.no_grad():
        outputs = model(input_ids=generated_ids)

    logits = outputs.logits
    next_token_logits = logits[:, -1, :]
    next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)

    generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

    new_token_text = tokenizer.decode(next_token_id[0], skip_special_tokens=False)
    current_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print(f"step {step+1}")
    print("새 토큰:", repr(new_token_text))
    print("현재까지 생성:", current_text)
    print("-" * 60)

step 1
새 토큰: ' Paris'
현재까지 생성: The capital of France is Paris
------------------------------------------------------------
step 2
새 토큰: '.'
현재까지 생성: The capital of France is Paris.
------------------------------------------------------------
step 3
새 토큰: ' '
현재까지 생성: The capital of France is Paris. 
------------------------------------------------------------
step 4
새 토큰: '\n\n'
현재까지 생성: The capital of France is Paris. 


------------------------------------------------------------
step 5
새 토큰: 'Is'
현재까지 생성: The capital of France is Paris. 

Is
------------------------------------------------------------
step 6
새 토큰: ' this'
현재까지 생성: The capital of France is Paris. 

Is this
------------------------------------------------------------
step 7
새 토큰: ' statement'
현재까지 생성: The capital of France is Paris. 

Is this statement
------------------------------------------------------------
step 8
새 토큰: ' true'
현재까지 생성: The capital of France is Paris. 

Is this statement true
--------------------

In [24]:
# 종료 조건 추가
## 무한히 생성하지 않고, EOS(End of Sequence) 토큰이 나오면 멈춤.
prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

generated_ids = input_ids.clone()

max_new_tokens = 30
eos_token_id = tokenizer.eos_token_id

for step in range(max_new_tokens):
    with torch.no_grad():
        outputs = model(input_ids=generated_ids)

    next_token_logits = outputs.logits[:, -1, :]
    next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)

    generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

    if next_token_id.item() == eos_token_id:
        print(f"EOS 토큰이 step {step+1}에서 생성되어 종료합니다.")
        break

print("최종 생성 결과:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

EOS 토큰이 step 22에서 생성되어 종료합니다.
최종 생성 결과:
The capital of France is Paris. 

Is this statement true or false?

The statement is **true**. 



In [25]:
# 함수로 정리하기(greedy decoding)
def greedy_generate(model, tokenizer, prompt, max_new_tokens=30):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    generated_ids = input_ids.clone()

    eos_token_id = tokenizer.eos_token_id

    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=generated_ids)

        next_token_logits = outputs.logits[:, -1, :]
        next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)

        generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

        if eos_token_id is not None and next_token_id.item() == eos_token_id:
            break

    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

In [26]:
result = greedy_generate(model, tokenizer, "Explain what a language model is.", max_new_tokens=50)
print(result)

Explain what a language model is.

A language model is a statistical method that predicts the probability of a word or sequence of words appearing in a given context. 

Here's a breakdown:

**Key Concepts:**

* **Probability:**  Language models assign probabilities to different


In [27]:
# Top-5 후보도 같이 보기

prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    outputs = model(input_ids=input_ids)

next_token_logits = outputs.logits[:, -1, :]
topk_values, topk_indices = torch.topk(next_token_logits, k=5, dim=-1)

print("top-5 후보:")
for rank in range(5):
    token_id = topk_indices[0, rank].item()
    token_score = topk_values[0, rank].item()
    token_text = tokenizer.decode([token_id])
    print(f"{rank+1}. token_id={token_id}, token={repr(token_text)}, score={token_score:.4f}")

top-5 후보:
1. token_id=7127, token=' Paris', score=13.8672
2. token_id=235292, token=':', score=13.7969
3. token_id=5231, token=' **', score=13.6797
4. token_id=955, token='...', score=12.8281
5. token_id=235248, token=' ', score=12.2578


In [33]:
system_prompt = "You are a strict professor."
temperature = 0.2
top_p = 0.8
top_k = 20
do_sample = False
seed = 123

## 1. Gemma 4란?

Gemma 4는 Google DeepMind가 공개한 **오픈 웨이트(open-weight) 생성형 모델 계열**입니다.  
Google 공식 문서에 따르면 Gemma 4는 **텍스트와 이미지 입력을 처리하는 멀티모달 모델**이며, 일부 소형 모델은 **오디오 입력도 지원**하고, 출력은 텍스트입니다. 또한 **사전학습(pre-trained)** 버전과 **instruction-tuned** 버전이 함께 공개되었습니다. 
---

## 2. 핵심 특징

### 멀티모달
Gemma 4는 텍스트만 다루는 모델이 아니라, **텍스트 + 이미지 입력**을 함께 받을 수 있습니다.  
공식 모델 카드에는 작은 모델에서 오디오 입력도 지원된다고 명시되어 있습니다.  
### 긴 컨텍스트
Gemma 4는 **최대 256K 토큰 컨텍스트 윈도우**를 지원합니다.  
그래서 짧은 질문 응답뿐 아니라, **긴 문서 요약**, **여러 자료 비교**, **긴 대화 맥락 유지** 같은 작업에 더 적합합니다. 

### 다국어 지원
Gemma 4는 **140개 이상의 언어**를 지원합니다.  
한국어 실습도 가능하며, 영어 자료를 한국어로 설명하거나 반대로 한국어 초안을 영어 발표용으로 바꾸는 작업에도 활용할 수 있습니다.  

### 효율성 중심 설계
Google은 Gemma 4를 단순히 큰 모델로 소개하기보다, **효율성과 폭넓은 배포성**을 강조하고 있습니다.  
즉, 로컬 실습부터 다양한 추론 엔진과 배포 환경까지 고려한 계열이라고 볼 수 있습니다. 
---

## 3. 모델 종류

Google의 릴리스 문서에 따르면 Gemma 4는 다음 크기로 공개되었습니다.

- **E2B**
- **E4B**
- **26B A4B**
- **31B**  

이 구성은 한 가지 크기만 제공하는 것이 아니라,  
**작고 가벼운 모델부터 더 강한 성능의 모델까지** 선택할 수 있게 한 것입니다.

---

## 4. Dense와 MoE 관점에서 이해하기

Gemma 4는 단순히 “크기만 다른 모델들”이 아니라, **구조적으로도 효율성 차이**를 둔 계열입니다.  
공식 문서와 배포 자료를 보면 Dense 계열과 함께 **A4B 같은 효율 지향 모델**이 포함되어 있습니다. 이는 전체 모델 규모와 실제 활성화되는 계산량을 분리해 효율을 높이려는 방향으로 이해할 수 있습니다.  

실무적으로는 이렇게 생각하면 편합니다.

- **E2B / E4B**: 가볍게 실습하거나 온디바이스 실행 감각을 익히기 좋음
- **26B A4B**: 더 강한 성능과 효율을 함께 노리는 선택지
- **31B**: 일반 성능을 더 강하게 기대하는 대형 모델

---

## 5. Gemma 4가 잘하는 일

Gemma 4의 장점은 짧은 질문 한두 개로는 잘 드러나지 않을 수 있습니다.  
오히려 아래 같은 작업에서 강점이 더 잘 보입니다.

### 긴 자료 정리
- 긴 강의노트 요약
- 여러 문서 비교
- 긴 초안 재구성
- 보고서 구조화

### 멀티모달 작업
- 이미지 설명
- 이미지 기반 질문응답
- OCR 기반 이해
- 여러 이미지 비교

### 다단계 추론
- 복잡한 조건을 지키는 답변 작성
- 긴 맥락을 유지한 분석
- 여러 근거를 연결한 설명

이런 사용 방향은 공식 모델 카드와 Hugging Face 소개 자료에서도 강조됩니다.  

## 6. Thinking 관련 포인트

Google Gemma 문서에는 Gemma 4의 **thinking capabilities**가 따로 설명되어 있습니다.  
즉, 단순 응답뿐 아니라 **추론 과정이 필요한 작업**을 염두에 두고 활용할 수 있는 패턴이 공식적으로 안내됩니다. 이는 수학 문제, 단계적 설명, 복합 과제 분석에서 특히 의미가 있습니다.  

---

## 7. Gemma 4를 짧은 실습만으로 평가하면 아쉬운 이유

예를 들어 아래 같은 프롬프트는

- “Gemma 4를 5문장으로 설명해줘”
- “Transformer를 쉽게 설명해줘”

대부분의 최신 instruct 모델도 꽤 잘 답합니다.  
그래서 이 정도 테스트만 하면 Gemma 4의 장점을 크게 느끼기 어렵습니다.

Gemma 4의 강점을 보려면 다음처럼 해야 합니다.

- 긴 입력 2~3개를 한 번에 넣기
- 비교, 구조화, 재작성 시키기
- 이미지와 텍스트를 함께 넣기
- 여러 제약을 동시에 준 뒤 답변 품질 보기

이유는 공식 문서가 강조하는 핵심이 **멀티모달**, **긴 컨텍스트**, **효율적인 강한 모델 계열**이기 때문입니다.  
---

## 8. Gemma 4와 이전 Gemma 계열의 흐름

Gemma 계열은 Google이 공개한 오픈 모델 라인업으로 이어져 왔고,  
Gemma 4는 그 흐름에서 **멀티모달**, **긴 컨텍스트**, **더 다양한 모델 크기**, **더 폭넓은 활용 환경** 쪽이 강화된 세대라고 볼 수 있습니다.  
Google의 Gemma 문서와 릴리스 페이지도 이런 계열 확장을 보여줍니다. 

---

## 9. 실행 환경 관점에서의 의미

Gemma 4는 Google 공식 문서와 Hugging Face 자료 기준으로  
Kaggle, Hugging Face, 다양한 추론 엔진과 라이브러리, 튜닝 경로와 연동되도록 안내됩니다.  
즉, “논문용 공개 모델”보다는 **실제로 가져다 써볼 수 있는 개방형 모델 계열**이라는 점이 강합니다. 

In [34]:
!pip install -U torch torchvision transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 48.7 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
flair 0.15.1 requires transformers[sentencepiece]<5.0.0,>=4.25.0, but you have transformers 5.5.1 which is incompatible.
llm2vec 0.2.3 requires transformers<=4.44.2,>=4.43.1, but you have transformers 5.5.1 which is incompatible.
transformer-smaller-training-vocab 0.4.2 requires transformers[sentencepiece,torch]<5.0,>=4.1, but you have transformers 5.5.1 which is incompatible.


In [35]:
import torch
from transformers import pipeline, GenerationConfig

print("PyTorch version:", torch.__version__)

if torch.backends.mps.is_available():
    device_name = "mps"
elif torch.cuda.is_available():
    device_name = "cuda"
else:
    device_name = "cpu"

print("Using device:", device_name)

PyTorch version: 2.11.0
Using device: mps


In [36]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

messages = [
    {"role": "system", "content": "You are a patient math professor."},
    {"role": "user", "content": "Explain what a basis is in linear algebra."}
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=200)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

print(processor.parse_response(response))

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

{'role': 'assistant', 'content': 'Ah, a wonderful question! Welcome to the fascinating world of linear algebra. Understanding what a **basis** is is like learning the fundamental building blocks for describing any vector space. It\'s a concept that underpins so much of what we do in this field, from solving systems of equations to understanding transformations.\n\nPlease, settle in. Take a deep breath. We\'ll take this step by step.\n\n### The Core Idea: The "Minimal Set of Ingredients"\n\nAt its heart, a **basis** for a vector space $V$ is a special set of vectors that has two crucial properties:\n\n1. **Linear Independence:** No vector in the set can be written as a combination of the others. They are all fundamentally *independent* of each other.\n2. **Spanning Set:** Every single vector in the entire vector space $V$ can be written as a linear combination of the vectors in that set. They are sufficient to *reach* every point in the'}


In [37]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

def generate_response(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    few_shot_examples=None,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    do_sample=True,
    max_new_tokens=200,
    seed=42,
    enable_thinking=False,
):
    set_seed(seed)

    messages = [
        {"role": "system", "content": system_prompt}
    ]

    if few_shot_examples is not None:
        for ex in few_shot_examples:
            messages.append({"role": "user", "content": ex["user"]})
            messages.append({"role": "assistant", "content": ex["assistant"]})

    messages.append({"role": "user", "content": user_prompt})

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            top_k=top_k if do_sample else None,
        )

    response = processor.decode(
        outputs[0][input_len:],
        skip_special_tokens=False
    )

    return processor.parse_response(response)

In [38]:
result = generate_response(
    user_prompt="Explain what a basis is in linear algebra for a second-year undergraduate student.",
    system_prompt="You are a patient math professor. Be clear, friendly, and precise.",
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=220,
    seed=42,
    enable_thinking=False
)

print(result)

{'role': 'assistant', 'content': "Hello there! I'd be delighted to walk you through the concept of a **basis** in linear algebra. It's a fundamental idea, and once you grasp it, it really helps unlock a lot of other topics. Don't worry if it seems a little abstract at first; we'll take it step by step.\n\nThink of linear algebra as the language we use to describe and manipulate vectors and linear transformations. A **basis** is essentially the most efficient, minimal set of building blocks you need to describe *every single vector* in a given vector space.\n\nLet's break this down formally.\n\n---\n\n### 1. The Setup: Vector Spaces\n\nBefore we define a basis, we need to know what we're talking about. We are working within a **vector space**, let's call it $V$. A vector space is just a collection of objects (vectors) that can be added together and multiplied by scalars (real or complex numbers), satisfying certain rules.\n\n**Example:**\n*   $\\mathbb{R}^2$ (the"}


In [39]:
#system prompt만 바꾸기
user_prompt = "Explain what an eigenvalue is."

system_prompts = [
    "You are a patient math professor. Explain carefully to undergraduate students.",
    "You are a concise lecturer. Answer in exactly 5 sentences.",
    "You are an English presentation coach. Use simple spoken English.",
    "You are a rigorous mathematician. Give definition, intuition, and one example."
]

for i, sp in enumerate(system_prompts, 1):
    print(f"\n===== System Prompt {i} =====")
    result = generate_response(
        user_prompt=user_prompt,
        system_prompt=sp,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=180,
        seed=42
    )
    print(result)


===== System Prompt 1 =====
{'role': 'assistant', 'content': "Hello everyone, and welcome to this session on linear algebra. I see you all have some questions, and that is wonderful! Today, we're going to tackle a concept that is absolutely central to understanding many areas of mathematics, physics, and engineering: **eigenvalues and eigenvectors**.\n\nDon't let the fancy terminology intimidate you. We're going to build this up step-by-step. Think of me as your guide through this concept.\n\n### The Core Idea: What are we trying to find?\n\nAt its heart, finding eigenvalues and eigenvectors is about understanding the *fundamental behavior* of a linear transformation represented by a matrix.\n\nImagine you have a square matrix, let's call it $A$. This matrix $A$ acts like a machine that takes a vector and transforms it—it stretches it, rotates it, shears it, and changes its direction.\n\n"}

===== System Prompt 2 =====
{'role': 'assistant', 'content': 'An eigenvalue is a scalar associ

In [40]:
#temperature 바꾸기
for t in [0.2, 0.7, 1.0]:
    print(f"\n===== temperature = {t} =====")
    result = generate_response(
        user_prompt="Explain singular value decomposition in simple terms.",
        system_prompt="You are a patient math professor.",
        temperature=t,
        top_p=0.9,
        top_k=50,
        max_new_tokens=200,
        seed=42
    )
    print(result)


===== temperature = 0.2 =====
{'role': 'assistant', 'content': "(Clears throat gently, adjusts spectacles, and smiles warmly)\n\nHello there! Please, come in, have a seat. I see you're interested in Singular Value Decomposition, or SVD. That is a wonderful topic. It sounds very technical, but I promise you, at its heart, it is a very elegant and intuitive mathematical tool.\n\nThink of me as your guide through this. We won't dive into the heavy calculus right away; we'll start with the *idea*.\n\n### The Big Picture: What is SVD Trying to Do?\n\nImagine you have a complex object—maybe a picture, a piece of music, or even a large set of data points. This object exists in a high-dimensional space, which can be incredibly messy and complicated.\n\n**The goal of Singular Value Decomposition is to take that complicated object and break it down into three very simple, fundamental pieces.**\n\nThink of it like this: Instead of looking at a complicated,"}

===== temperature = 0.7 =====
{'role

- 낮은 temperature는 더 보수적
- 높은 temperature는 더 다양

In [41]:
# top_p 바꾸기

for p in [0.7, 0.9, 0.98]:
    print(f"\n===== top_p = {p} =====")
    result = generate_response(
        user_prompt="Explain the difference between linear independence and spanning.",
        system_prompt="You are a patient math professor.",
        temperature=0.7,
        top_p=p,
        top_k=50,
        max_new_tokens=200,
        seed=42
    )
    print(result)


===== top_p = 0.7 =====
{'role': 'assistant', 'content': 'Welcome! Please, have a seat. I\'m happy to help you clarify the concepts of linear independence and spanning. These are fundamental ideas in linear algebra, and once you see the relationship between them, they become much clearer.\n\nThink of these concepts not just as abstract mathematical terms, but as ways to describe the *relationships* between a set of vectors.\n\nLet\'s start with the basics.\n\n---\n\n## 1. Spanning (The "Reach" Concept)\n\n**What it means:** A set of vectors, let\'s call it $S = \\{\\mathbf{v}_1, \\mathbf{v}_2, \\ldots, \\mathbf{v}_p\\}$, **spans** a vector space $V$ if *every single vector* in that space $V$ can be written as a linear combination of the vectors in $S$.\n\n**Analogy:** Imagine you have a set of tools (vectors). If these tools can be combined in various'}

===== top_p = 0.9 =====
{'role': 'assistant', 'content': "Well hello there! Please, have a seat. I see you're grappling with the con

In [42]:
# top_k 바꾸기
for k in [20, 50, 100]:
    print(f"\n===== top_k = {k} =====")
    result = generate_response(
        user_prompt="Give an intuitive explanation of orthogonality.",
        system_prompt="You are a patient math professor.",
        temperature=0.7,
        top_p=0.9,
        top_k=k,
        max_new_tokens=200,
        seed=42
    )
    print(result)


===== top_k = 20 =====
{'role': 'assistant', 'content': 'Ah, welcome! Please, have a seat. Orthogonality. It sounds very formal, very mathematical, but I assure you, the underlying idea is quite intuitive and quite beautiful. Don\'t let the terminology scare you; let\'s talk about what it *means*.\n\nImagine you are working in a space—maybe a flat piece of paper, or perhaps a three-dimensional room. In this space, we have different "directions" or "vectors." A vector is just something that has both a **magnitude** (how long or strong it is) and a **direction**.\n\n### The Intuitive Analogy: The Perfectly Perpendicular Lines\n\nThe easiest way to grasp orthogonality is to think about the relationship between two lines in a familiar 2D plane.\n\n**Orthogonal means "at a right angle."**\n\nIf you have a horizontal line and a vertical line on a piece of graph paper, they are orthogonal. They meet perfectly at $90^\\circ$.'}

===== top_k = 50 =====
{'role': 'assistant', 'content': 'Ah, wel

In [43]:
# few-shot 넣기
few_shot_examples = [
    {
        "user": "Explain what a vector space is.",
        "assistant": "A vector space is a collection of vectors where you can add vectors and multiply them by numbers, and the results still stay in the same space."
    },
    {
        "user": "Explain what a basis is.",
        "assistant": "A basis is a minimal set of vectors that can generate every vector in the space by linear combination."
    }
]

print("===== No few-shot =====")
result1 = generate_response(
    user_prompt="Explain what dimension means in linear algebra.",
    system_prompt="You are a patient math professor.",
    few_shot_examples=None,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=180,
    seed=42
)
print(result1)

print("\n===== With few-shot =====")
result2 = generate_response(
    user_prompt="Explain what dimension means in linear algebra.",
    system_prompt="You are a patient math professor.",
    few_shot_examples=few_shot_examples,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=180,
    seed=42
)
print(result2)

===== No few-shot =====
{'role': 'assistant', 'content': 'Ah, welcome! Please, have a seat. I\'d be delighted to explain the concept of **dimension** in linear algebra. It is a fundamental and incredibly important idea, and once you grasp it, it unlocks so much of what we do in the field.\n\nThink of me as your guide through this mathematical landscape. Don\'t hesitate to stop me if any part feels fuzzy!\n\n### The Intuitive Idea: Dimension as "Degrees of Freedom"\n\nAt its most intuitive level, the **dimension** of a vector space tells you the *minimum number of independent directions* you need to specify to reach any point within that space.\n\nLet\'s use some familiar examples to build this intuition:\n\n#### 1. Dimension 1: The Number Line ($\\mathbb{R}^1$)\nImagine a single number line. If you want to locate any point on this line,'}

===== With few-shot =====
{'role': 'assistant', 'content': 'Ah, the dimension! That\'s a fundamental concept.\n\nIn linear algebra, the **dimension*

In [44]:
# 한국어와 영어 비교

prompts = [
    "선형대수에서 basis를 학부 2학년 학생에게 쉽게 설명해줘.",
    "Explain basis in linear algebra as if you are speaking in a 1-minute lecture."
]

for p in prompts:
    print("\n==============================")
    print("PROMPT:", p)
    result = generate_response(
        user_prompt=p,
        system_prompt="You are a patient math professor.",
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=200,
        seed=42
    )
    print(result)




PROMPT: 선형대수에서 basis를 학부 2학년 학생에게 쉽게 설명해줘.
{'role': 'assistant', 'content': "## 선형대수학에서 '기저(Basis)'에 대한 설명 (학부 2학년 학생을 위한 맞춤 설명)\n\n안녕하세요, 여러분! 선형대수학의 세계에 오신 것을 환영합니다. 오늘은 아마 많은 학생들이 처음 접하지만, 매우 중요하고 핵심적인 개념인 **'기저(Basis)'**에 대해 쉽고 명확하게 설명해 드릴게요.\n\n복잡한 수학 용어는 잠시 내려놓고, 우리가 일상생활에서 사용하는 비유를 통해 이 개념을 확실하게 이해해 봅시다.\n\n---\n\n### 1. 기저(Basis)란 무엇인가? (핵 개념)\n\n가장 간단하게 말해, **기저(Basis)는 어떤 벡터 공간(Vector Space)을 '완지 않게' 설명하는 데 필요한 최소한의 '기본 재료' 또는 '축소된 좌표계'**라고 생각하면 됩니다.\n\n**비유:** 여러분이"}

PROMPT: Explain basis in linear algebra as if you are speaking in a 1-minute lecture.
{'role': 'assistant', 'content': '**(Clears throat, adjusts glasses, and smiles warmly)**\n\n"Good morning, class. Today, we\'re tackling a fundamental concept in linear algebra: **basis**. Think of a basis as the absolute *minimal* set of building blocks you need to describe *every single point* in a vector space.\n\nImagine you\'re navigating a city. You don\'t need every street intersection to describe where you are; you j

In [45]:
# 결과 표 만들기

import pandas as pd

records = []

settings = [
    {"temperature": 0.2, "top_p": 0.9, "top_k": 50},
    {"temperature": 0.7, "top_p": 0.9, "top_k": 50},
    {"temperature": 1.0, "top_p": 0.9, "top_k": 50},
]

for s in settings:
    text = generate_response(
        user_prompt="Explain what a basis is in linear algebra.",
        system_prompt="You are a patient math professor.",
        temperature=s["temperature"],
        top_p=s["top_p"],
        top_k=s["top_k"],
        max_new_tokens=160,
        seed=42
    )
    records.append({
        "temperature": s["temperature"],
        "top_p": s["top_p"],
        "top_k": s["top_k"],
        "response": str(text)
    })

df = pd.DataFrame(records)
df

,temperature,top_p,top_k,response
0,0.2,0.9,50,"{'role': 'assistant', 'content': 'Ah, welcome!..."
1,0.7,0.9,50,"{'role': 'assistant', 'content': 'Ah, welcome!..."
2,1.0,0.9,50,"{'role': 'assistant', 'content': ""Ah, welcome!..."


In [46]:
df["accuracy_score"] = [5, 5, 4]
df["clarity_score"] = [4, 5, 3]
df["presentation_score"] = [4, 5, 3]
df["length_score"] = [5, 4, 3]
df

,temperature,top_p,top_k,response,accuracy_score,clarity_score,presentation_score,length_score
0,0.2,0.9,50,"{'role': 'assistant', 'content': 'Ah, welcome!...",5,4,4,5
1,0.7,0.9,50,"{'role': 'assistant', 'content': 'Ah, welcome!...",5,5,5,4
2,1.0,0.9,50,"{'role': 'assistant', 'content': ""Ah, welcome!...",4,3,3,3


- 한 토큰씩 직접 생성하는 디버깅 실습

In [47]:
# 입력 프롬프트와 토큰 길이 확인
messages = [
    {"role": "system", "content": "You are a patient math professor."},
    {"role": "user", "content": "Explain what a basis is in linear algebra."}
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

print("=== Prompt text ===")
print(text)

inputs = processor(text=text, return_tensors="pt").to(model.device)

print("\n=== Tensor keys ===")
print(inputs.keys())

print("\n=== input_ids shape ===")
print(inputs["input_ids"].shape)

=== Prompt text ===
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model


=== Tensor keys ===
KeysView({'input_ids': tensor([[     2,    105,   9731,    107,   3048,    659,    496,   6213,   6596,
          14825, 236761,    106,    107,    105,   2364,    107, 155122,   1144,
            496,   5150,    563,    528,   6373,  10075, 236761,    106,    107,
            105,   4368,    107]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]], device='mps:0'), 'mm_token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0]], device='mps:0')})

=== input_ids shape ===
torch.Size([1, 30])


In [48]:
# 입력 토큰을 사람이 볼 수 있게 만듦.
input_ids = inputs["input_ids"][0]

tokens = processor.tokenizer.convert_ids_to_tokens(input_ids.tolist())

print("=== First 80 input tokens ===")
for i, (tid, tok) in enumerate(zip(input_ids.tolist()[:80], tokens[:80])):
    print(f"{i:3d} | id={tid:6d} | token={tok}")

=== First 80 input tokens ===
  0 | id=     2 | token=<bos>
  1 | id=   105 | token=<|turn>
  2 | id=  9731 | token=system
  3 | id=   107 | token=

  4 | id=  3048 | token=You
  5 | id=   659 | token=▁are
  6 | id=   496 | token=▁a
  7 | id=  6213 | token=▁patient
  8 | id=  6596 | token=▁math
  9 | id= 14825 | token=▁professor
 10 | id=236761 | token=.
 11 | id=   106 | token=<turn|>
 12 | id=   107 | token=

 13 | id=   105 | token=<|turn>
 14 | id=  2364 | token=user
 15 | id=   107 | token=

 16 | id=155122 | token=Explain
 17 | id=  1144 | token=▁what
 18 | id=   496 | token=▁a
 19 | id=  5150 | token=▁basis
 20 | id=   563 | token=▁is
 21 | id=   528 | token=▁in
 22 | id=  6373 | token=▁linear
 23 | id= 10075 | token=▁algebra
 24 | id=236761 | token=.
 25 | id=   106 | token=<turn|>
 26 | id=   107 | token=

 27 | id=   105 | token=<|turn>
 28 | id=  4368 | token=model
 29 | id=   107 | token=



In [49]:
# 한번 forward 해서 마지막 위치 logits 보기

with torch.no_grad():
    outputs = model(**inputs)

print(type(outputs))
print(outputs.keys() if hasattr(outputs, "keys") else "no keys")

logits = outputs.logits
print("logits shape:", logits.shape)
#(batch_size, seq_len, vocab_size)

<class 'transformers.models.gemma4.modeling_gemma4.Gemma4CausalLMOutputWithPast'>
odict_keys(['logits', 'past_key_values'])
logits shape: torch.Size([1, 30, 262144])


In [50]:
# 마지막 위치에서 상위 후보 10개 보기

last_logits = logits[:, -1, :]   # 마지막 위치 = 다음 토큰 예측용
top_k = 10

top_vals, top_ids = torch.topk(last_logits, k=top_k, dim=-1)

top_ids = top_ids[0].tolist()
top_vals = top_vals[0].tolist()

print("=== Top next-token candidates ===")
for rank, (tid, score) in enumerate(zip(top_ids, top_vals), 1):
    tok = processor.tokenizer.convert_ids_to_tokens([tid])[0]
    text_piece = processor.tokenizer.decode([tid], skip_special_tokens=False)
    print(f"{rank:2d}. id={tid:6d} | logit={score:8.4f} | token={tok} | decoded={repr(text_piece)}")

=== Top next-token candidates ===
 1. id= 24019 | logit=  9.2500 | token=Ah | decoded='Ah'
 2. id= 16651 | logit=  8.8750 | token=Welcome | decoded='Welcome'
 3. id=236769 | logit=  8.5625 | token=( | decoded='('
 4. id= 13086 | logit=  8.5000 | token=Well | decoded='Well'
 5. id=  9259 | logit=  6.0000 | token=Hello | decoded='Hello'
 6. id=177906 | logit=  4.4375 | token=**( | decoded='**('
 7. id= 11947 | logit=  4.1250 | token=Good | decoded='Good'
 8. id=  9366 | logit=  3.3438 | token=Please | decoded='Please'
 9. id=  1408 | logit=  0.9883 | token=## | decoded='##'
10. id=  4335 | logit=  0.8711 | token=Class | decoded='Class'


In [51]:
# greedy 방식으로 한 토큰씩만 직접 붙여보기

next_token_id = torch.argmax(last_logits, dim=-1, keepdim=True)   # (1, 1)

print("next_token_id:", next_token_id.item())
print("decoded next token:", repr(processor.tokenizer.decode(next_token_id[0])))

next_token_id: 24019
decoded next token: 'Ah'


In [52]:
# 실제로 입력 뒤에 붙여보기
new_input_ids = torch.cat([inputs["input_ids"], next_token_id], dim=1)

print("old length:", inputs["input_ids"].shape[1])
print("new length:", new_input_ids.shape[1])

decoded_so_far = processor.tokenizer.decode(new_input_ids[0], skip_special_tokens=False)
print(decoded_so_far)

old length: 30
new length: 31
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model
Ah


In [53]:
# 한 토큰씩 여러 번 생성하는 디버깅 함수
def greedy_decode_debug(
    model,
    processor,
    messages,
    max_new_tokens=20,
    enable_thinking=False,
    show_topk=5
):
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    generated_ids = inputs["input_ids"].clone()

    print("=== Initial prompt ===")
    print(text)
    print("\n=== Step-by-step generation ===")

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=generated_ids)

        logits = outputs.logits
        last_logits = logits[:, -1, :]

        top_vals, top_ids = torch.topk(last_logits, k=show_topk, dim=-1)
        top_ids_list = top_ids[0].tolist()
        top_vals_list = top_vals[0].tolist()

        print(f"\n--- Step {step+1} ---")
        print("Top candidates:")
        for rank, (tid, score) in enumerate(zip(top_ids_list, top_vals_list), 1):
            tok = processor.tokenizer.convert_ids_to_tokens([tid])[0]
            text_piece = processor.tokenizer.decode([tid], skip_special_tokens=False)
            print(f"  {rank}. id={tid:6d} | logit={score:8.4f} | token={tok} | decoded={repr(text_piece)}")

        next_token_id = torch.argmax(last_logits, dim=-1, keepdim=True)
        next_id = next_token_id.item()
        next_text = processor.tokenizer.decode([next_id], skip_special_tokens=False)

        print(f"Chosen token: id={next_id} | decoded={repr(next_text)}")

        generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

        partial_text = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=False)
        print("\nText so far:")
        print(partial_text[-500:])   # 너무 길어지지 않게 뒤쪽만 출력

        eos_id = processor.tokenizer.eos_token_id
        if eos_id is not None and next_id == eos_id:
            print("\n[EOS reached]")
            break

    return generated_ids

In [54]:
messages = [
    {"role": "system", "content": "You are a patient math professor."},
    {"role": "user", "content": "Explain what a basis is in linear algebra."}
]

generated_ids = greedy_decode_debug(
    model=model,
    processor=processor,
    messages=messages,
    max_new_tokens=20,
    enable_thinking=False,
    show_topk=5
)

=== Initial prompt ===
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model


=== Step-by-step generation ===

--- Step 1 ---
Top candidates:
  1. id= 24019 | logit=  9.2500 | token=Ah | decoded='Ah'
  2. id= 16651 | logit=  8.8750 | token=Welcome | decoded='Welcome'
  3. id=236769 | logit=  8.5625 | token=( | decoded='('
  4. id= 13086 | logit=  8.5000 | token=Well | decoded='Well'
  5. id=  9259 | logit=  6.0000 | token=Hello | decoded='Hello'
Chosen token: id=24019 | decoded='Ah'

Text so far:
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model
Ah

--- Step 2 ---
Top candidates:
  1. id=236764 | logit= 20.8750 | token=, | decoded=','
  2. id=  1088 | logit=  1.9609 | token=oy | decoded='oy'
  3. id=236888 | logit=  1.4531 | token=! | decoded='!'
  4. id= 27989 | logit=  0.0452 | token=hh | decoded='hh'
  5. id=  8558 | l

In [55]:
final_text = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=False)
print(final_text)

print("\n=== Parsed response ===")
# 프롬프트 길이 이후만 잘라서 parse
prompt_text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
prompt_inputs = processor(text=prompt_text, return_tensors="pt").to(model.device)
prompt_len = prompt_inputs["input_ids"].shape[1]

response_only = processor.tokenizer.decode(
    generated_ids[0][prompt_len:],
    skip_special_tokens=False
)
print(processor.parse_response(response_only))

<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model
Ah, a wonderful question! The concept of a **basis** is absolutely fundamental in linear algebra.

=== Parsed response ===
{'role': 'assistant', 'content': 'Ah, a wonderful question! The concept of a **basis** is absolutely fundamental in linear algebra.'}


In [56]:
# sampling
def sample_next_token(logits, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be > 0 for sampling")

    scaled_logits = logits / temperature
    probs = torch.softmax(scaled_logits, dim=-1)
    next_token_id = torch.multinomial(probs, num_samples=1)
    return next_token_id, probs

In [57]:
def sample_decode_debug(
    model,
    processor,
    messages,
    max_new_tokens=20,
    temperature=0.7,
    enable_thinking=False,
    show_topk=5
):
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    generated_ids = inputs["input_ids"].clone()

    print("=== Initial prompt ===")
    print(text)
    print("\n=== Step-by-step sampled generation ===")

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=generated_ids)

        last_logits = outputs.logits[:, -1, :]

        next_token_id, probs = sample_next_token(last_logits, temperature=temperature)
        next_id = next_token_id.item()

        top_vals, top_ids = torch.topk(probs, k=show_topk, dim=-1)
        top_ids_list = top_ids[0].tolist()
        top_vals_list = top_vals[0].tolist()

        print(f"\n--- Step {step+1} ---")
        print("Top probabilities:")
        for rank, (tid, prob) in enumerate(zip(top_ids_list, top_vals_list), 1):
            tok = processor.tokenizer.convert_ids_to_tokens([tid])[0]
            text_piece = processor.tokenizer.decode([tid], skip_special_tokens=False)
            print(f"  {rank}. id={tid:6d} | prob={prob:.6f} | token={tok} | decoded={repr(text_piece)}")

        print(f"Sampled token: id={next_id} | decoded={repr(processor.tokenizer.decode([next_id], skip_special_tokens=False))}")

        generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

        eos_id = processor.tokenizer.eos_token_id
        if eos_id is not None and next_id == eos_id:
            print("[EOS reached]")
            break

    return generated_ids

In [58]:
sampled_ids = sample_decode_debug(
    model=model,
    processor=processor,
    messages=messages,
    max_new_tokens=20,
    temperature=0.7,
    enable_thinking=False,
    show_topk=5
)

=== Initial prompt ===
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model


=== Step-by-step sampled generation ===

--- Step 1 ---
Top probabilities:
  1. id= 24019 | prob=0.423828 | token=Ah | decoded='Ah'
  2. id= 16651 | prob=0.257812 | token=Welcome | decoded='Welcome'
  3. id=236769 | prob=0.166016 | token=( | decoded='('
  4. id= 13086 | prob=0.146484 | token=Well | decoded='Well'
  5. id=  9259 | prob=0.004150 | token=Hello | decoded='Hello'
Sampled token: id=16651 | decoded='Welcome'

--- Step 2 ---
Top probabilities:
  1. id=236888 | prob=0.929688 | token=! | decoded='!'
  2. id=   531 | prob=0.067383 | token=▁to | decoded=' to'
  3. id=236764 | prob=0.002609 | token=, | decoded=','
  4. id=  1012 | prob=0.000006 | token=▁class | decoded=' class'
  5. id=236761 | prob=0.000001 | token=. | decoded='.'
Sampled token: id=236888 | decoded='!'

--- Step 3 ---
Top probabilities:
  1. id=  7323 | pro

- top-k sampling, top-p sampling을 직접 구현하는 셀

	•	greedy: 가장 큰 logit 하나 선택

	•	top-k sampling: 확률이 높은 상위 k개 후보만 남기고 그 안에서 샘플링
    
	•	top-p sampling: 누적확률이 p를 넘을 때까지의 후보만 남기고 그 안에서 샘플링


In [59]:
# logits -> 확률

import torch

def logits_to_probs(logits, temperature=1.0):
    """
    logits: shape (1, vocab_size)
    temperature: > 0
    """
    if temperature <= 0:
        raise ValueError("temperature must be > 0")

    scaled_logits = logits / temperature
    probs = torch.softmax(scaled_logits, dim=-1)
    return probs

In [60]:
# top-k sampling
def top_k_sample(logits, top_k=50, temperature=1.0):
    """
    logits: shape (1, vocab_size)
    return:
        next_token_id: shape (1, 1)
        filtered_probs: shape (1, vocab_size)
    """
    if top_k <= 0:
        raise ValueError("top_k must be >= 1")

    probs = logits_to_probs(logits, temperature=temperature)

    vocab_size = probs.shape[-1]
    top_k = min(top_k, vocab_size)

    # 상위 k개 확률과 인덱스
    topk_probs, topk_ids = torch.topk(probs, k=top_k, dim=-1)

    # top-k 내부에서 다시 정규화
    topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)

    # top-k 후보 중 하나 샘플링
    sampled_idx_in_topk = torch.multinomial(topk_probs, num_samples=1)   # (1,1)
    next_token_id = torch.gather(topk_ids, dim=-1, index=sampled_idx_in_topk)

    # 디버깅용: 전체 vocab 크기의 filtered probs 만들기
    filtered_probs = torch.zeros_like(probs)
    filtered_probs.scatter_(dim=-1, index=topk_ids, src=topk_probs)

    return next_token_id, filtered_probs


In [61]:
# top-p sampling 

def top_p_sample(logits, top_p=0.9, temperature=1.0, min_tokens_to_keep=1):
    """
    logits: shape (1, vocab_size)
    return:
        next_token_id: shape (1, 1)
        filtered_probs: shape (1, vocab_size)
    """
    if not (0 < top_p <= 1.0):
        raise ValueError("top_p must be in (0, 1]")

    probs = logits_to_probs(logits, temperature=temperature)

    # 확률 내림차순 정렬
    sorted_probs, sorted_ids = torch.sort(probs, dim=-1, descending=True)

    # 누적확률
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # top_p를 넘는 지점부터 제거 대상으로 표시
    sorted_indices_to_remove = cumulative_probs > top_p

    # 최소한 몇 개는 유지
    if min_tokens_to_keep > 1:
        sorted_indices_to_remove[..., :min_tokens_to_keep] = False
    else:
        sorted_indices_to_remove[..., 0] = False

    # 제거할 위치는 0으로
    filtered_sorted_probs = sorted_probs.masked_fill(sorted_indices_to_remove, 0.0)

    # 다시 정규화
    filtered_sorted_probs = filtered_sorted_probs / filtered_sorted_probs.sum(dim=-1, keepdim=True)

    # 남은 후보 안에서 샘플링
    sampled_idx_in_sorted = torch.multinomial(filtered_sorted_probs, num_samples=1)   # (1,1)
    next_token_id = torch.gather(sorted_ids, dim=-1, index=sampled_idx_in_sorted)

    # 디버깅용: 원래 vocab 순서로 복원
    filtered_probs = torch.zeros_like(probs)
    filtered_probs.scatter_(dim=-1, index=sorted_ids, src=filtered_sorted_probs)

    return next_token_id, filtered_probs

In [62]:
# 후보 출력용

def print_top_candidates_from_probs(probs, processor, top_n=10):
    """
    probs: shape (1, vocab_size)
    """
    top_vals, top_ids = torch.topk(probs, k=top_n, dim=-1)

    top_ids = top_ids[0].tolist()
    top_vals = top_vals[0].tolist()

    print(f"Top {top_n} candidates:")
    for rank, (tid, prob) in enumerate(zip(top_ids, top_vals), 1):
        tok = processor.tokenizer.convert_ids_to_tokens([tid])[0]
        decoded = processor.tokenizer.decode([tid], skip_special_tokens=False)
        print(f"{rank:2d}. id={tid:6d} | prob={prob:.6f} | token={tok} | decoded={repr(decoded)}")

In [63]:
# 한 step에서 top-k/top-p 확인
messages = [
    {"role": "system", "content": "You are a patient math professor."},
    {"role": "user", "content": "Explain what a basis is in linear algebra."}
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = processor(text=text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

last_logits = outputs.logits[:, -1, :]   # 다음 토큰 예측용


In [64]:
# top-k
next_token_id_k, filtered_probs_k = top_k_sample(
    last_logits,
    top_k=50,
    temperature=0.7
)

print("=== Top-k sampling ===")
print_top_candidates_from_probs(filtered_probs_k, processor, top_n=10)
print("Sampled token:", next_token_id_k.item())
print("Decoded:", repr(processor.tokenizer.decode([next_token_id_k.item()], skip_special_tokens=False)))

=== Top-k sampling ===
Top 10 candidates:
 1. id= 24019 | prob=0.423828 | token=Ah | decoded='Ah'
 2. id= 16651 | prob=0.257812 | token=Welcome | decoded='Welcome'
 3. id=236769 | prob=0.166016 | token=( | decoded='('
 4. id= 13086 | prob=0.146484 | token=Well | decoded='Well'
 5. id=  9259 | prob=0.004150 | token=Hello | decoded='Hello'
 6. id=177906 | prob=0.000452 | token=**( | decoded='**('
 7. id= 11947 | prob=0.000292 | token=Good | decoded='Good'
 8. id=  9366 | prob=0.000095 | token=Please | decoded='Please'
 9. id=  1408 | prob=0.000003 | token=## | decoded='##'
10. id=  4335 | prob=0.000003 | token=Class | decoded='Class'
Sampled token: 24019
Decoded: 'Ah'


In [65]:
# top-p

next_token_id_p, filtered_probs_p = top_p_sample(
    last_logits,
    top_p=0.9,
    temperature=0.7
)

print("=== Top-p sampling ===")
print_top_candidates_from_probs(filtered_probs_p, processor, top_n=10)
print("Sampled token:", next_token_id_p.item())
print("Decoded:", repr(processor.tokenizer.decode([next_token_id_p.item()], skip_special_tokens=False)))


=== Top-p sampling ===
Top 10 candidates:
 1. id= 24019 | prob=0.500000 | token=Ah | decoded='Ah'
 2. id= 16651 | prob=0.304688 | token=Welcome | decoded='Welcome'
 3. id=236769 | prob=0.196289 | token=( | decoded='('
 4. id=     0 | prob=0.000000 | token=<pad> | decoded='<pad>'
 5. id=     1 | prob=0.000000 | token=<eos> | decoded='<eos>'
 6. id=     2 | prob=0.000000 | token=<bos> | decoded='<bos>'
 7. id=     3 | prob=0.000000 | token=<unk> | decoded='<unk>'
 8. id=     4 | prob=0.000000 | token=<mask> | decoded='<mask>'
 9. id=     5 | prob=0.000000 | token=[multimodal] | decoded='[multimodal]'
10. id=     6 | prob=0.000000 | token=<unused0> | decoded='<unused0>'
Sampled token: 16651
Decoded: 'Welcome'


In [66]:
# top-k sampling으로 한 토큰씩 생성하는 디버깅 함수

def top_k_decode_debug(
    model,
    processor,
    messages,
    max_new_tokens=20,
    top_k=50,
    temperature=0.7,
    enable_thinking=False,
    show_topn=5
):
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    generated_ids = inputs["input_ids"].clone()

    print("=== Initial prompt ===")
    print(text)
    print("\n=== Step-by-step top-k sampling ===")

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=generated_ids)

        last_logits = outputs.logits[:, -1, :]
        next_token_id, filtered_probs = top_k_sample(
            last_logits,
            top_k=top_k,
            temperature=temperature
        )

        print(f"\n--- Step {step+1} ---")
        print_top_candidates_from_probs(filtered_probs, processor, top_n=show_topn)

        next_id = next_token_id.item()
        next_text = processor.tokenizer.decode([next_id], skip_special_tokens=False)
        print(f"Chosen token: id={next_id} | decoded={repr(next_text)}")

        generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

        partial_text = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=False)
        print("\nText so far:")
        print(partial_text[-500:])

        eos_id = processor.tokenizer.eos_token_id
        if eos_id is not None and next_id == eos_id:
            print("\n[EOS reached]")
            break

    return generated_ids



In [67]:
generated_ids_topk = top_k_decode_debug(
    model=model,
    processor=processor,
    messages=messages,
    max_new_tokens=20,
    top_k=50,
    temperature=0.7,
    enable_thinking=False,
    show_topn=5
)

=== Initial prompt ===
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model


=== Step-by-step top-k sampling ===

--- Step 1 ---
Top 5 candidates:
 1. id= 24019 | prob=0.423828 | token=Ah | decoded='Ah'
 2. id= 16651 | prob=0.257812 | token=Welcome | decoded='Welcome'
 3. id=236769 | prob=0.166016 | token=( | decoded='('
 4. id= 13086 | prob=0.146484 | token=Well | decoded='Well'
 5. id=  9259 | prob=0.004150 | token=Hello | decoded='Hello'
Chosen token: id=16651 | decoded='Welcome'

Text so far:
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model
Welcome

--- Step 2 ---
Top 5 candidates:
 1. id=236888 | prob=0.929688 | token=! | decoded='!'
 2. id=   531 | prob=0.067383 | token=▁to | decoded=' to'
 3. id=236764 | prob=0.002609 | token=, | decoded=','
 4. id=  1012 | prob=0.000006 | token=▁class | decoded=' class'
 5. id=2

In [68]:
# top-p sampling으로 한 토큰씩 생성하는 디버깅 함수

def top_p_decode_debug(
    model,
    processor,
    messages,
    max_new_tokens=20,
    top_p=0.9,
    temperature=0.7,
    enable_thinking=False,
    show_topn=5
):
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    inputs = processor(text=text, return_tensors="pt").to(model.device)
    generated_ids = inputs["input_ids"].clone()

    print("=== Initial prompt ===")
    print(text)
    print("\n=== Step-by-step top-p sampling ===")

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=generated_ids)

        last_logits = outputs.logits[:, -1, :]
        next_token_id, filtered_probs = top_p_sample(
            last_logits,
            top_p=top_p,
            temperature=temperature
        )

        print(f"\n--- Step {step+1} ---")
        print_top_candidates_from_probs(filtered_probs, processor, top_n=show_topn)

        next_id = next_token_id.item()
        next_text = processor.tokenizer.decode([next_id], skip_special_tokens=False)
        print(f"Chosen token: id={next_id} | decoded={repr(next_text)}")

        generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

        partial_text = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=False)
        print("\nText so far:")
        print(partial_text[-500:])

        eos_id = processor.tokenizer.eos_token_id
        if eos_id is not None and next_id == eos_id:
            print("\n[EOS reached]")
            break

    return generated_ids

In [69]:
generated_ids_topp = top_p_decode_debug(
    model=model,
    processor=processor,
    messages=messages,
    max_new_tokens=20,
    top_p=0.9,
    temperature=0.7,
    enable_thinking=False,
    show_topn=5
)

=== Initial prompt ===
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model


=== Step-by-step top-p sampling ===

--- Step 1 ---
Top 5 candidates:
 1. id= 24019 | prob=0.500000 | token=Ah | decoded='Ah'
 2. id= 16651 | prob=0.304688 | token=Welcome | decoded='Welcome'
 3. id=236769 | prob=0.196289 | token=( | decoded='('
 4. id=     0 | prob=0.000000 | token=<pad> | decoded='<pad>'
 5. id=     1 | prob=0.000000 | token=<eos> | decoded='<eos>'
Chosen token: id=236769 | decoded='('

Text so far:
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model
(

--- Step 2 ---
Top 5 candidates:
 1. id= 12785 | prob=1.000000 | token=Cle | decoded='Cle'
 2. id=     0 | prob=0.000000 | token=<pad> | decoded='<pad>'
 3. id=     1 | prob=0.000000 | token=<eos> | decoded='<eos>'
 4. id=     2 | prob=0.000000 | token=<bos> | decoded='<bos>'
 5.

In [70]:
# top-k  결과
prompt_text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
prompt_inputs = processor(text=prompt_text, return_tensors="pt").to(model.device)
prompt_len = prompt_inputs["input_ids"].shape[1]

response_only_topk = processor.tokenizer.decode(
    generated_ids_topk[0][prompt_len:],
    skip_special_tokens=False
)

print("=== Top-k final response ===")
print(processor.parse_response(response_only_topk))

=== Top-k final response ===
{'role': 'assistant', 'content': "Welcome! Please, have a seat. I'd be delighted to walk you through the concept of"}


In [71]:
#top-p 결과
response_only_topp = processor.tokenizer.decode(
    generated_ids_topp[0][prompt_len:],
    skip_special_tokens=False
)

print("=== Top-p final response ===")
print(processor.parse_response(response_only_topp))

=== Top-p final response ===
{'role': 'assistant', 'content': '(Clears throat gently, adjusts spectacles, and smiles warmly)\n\nWelcome, welcome! Please,'}


In [72]:
# top-k 변화
for k in [10, 30, 50, 100]:
    print(f"\n========== top_k = {k} ==========")
    _ = top_k_decode_debug(
        model=model,
        processor=processor,
        messages=messages,
        max_new_tokens=10,
        top_k=k,
        temperature=0.7,
        show_topn=5
    )


========== top_k = 10 ==========
=== Initial prompt ===
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model


=== Step-by-step top-k sampling ===

--- Step 1 ---
Top 5 candidates:
 1. id= 24019 | prob=0.423828 | token=Ah | decoded='Ah'
 2. id= 16651 | prob=0.257812 | token=Welcome | decoded='Welcome'
 3. id=236769 | prob=0.166016 | token=( | decoded='('
 4. id= 13086 | prob=0.146484 | token=Well | decoded='Well'
 5. id=  9259 | prob=0.004150 | token=Hello | decoded='Hello'
Chosen token: id=13086 | decoded='Well'

Text so far:
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model
Well

--- Step 2 ---
Top 5 candidates:
 1. id= 29104 | prob=0.871094 | token=▁hello | decoded=' hello'
 2. id=236764 | prob=0.104004 | token=, | decoded=','
 3. id=  1492 | prob=0.023193 | token=▁now | decoded=' now'
 4. id=  1645 | prob=0.000009 | 

In [73]:
#top-p 비교

for p in [0.7, 0.85, 0.9, 0.95]:
    print(f"\n========== top_p = {p} ==========")
    _ = top_p_decode_debug(
        model=model,
        processor=processor,
        messages=messages,
        max_new_tokens=10,
        top_p=p,
        temperature=0.7,
        show_topn=5
    )


========== top_p = 0.7 ==========
=== Initial prompt ===
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model


=== Step-by-step top-p sampling ===

--- Step 1 ---
Top 5 candidates:
 1. id= 24019 | prob=0.625000 | token=Ah | decoded='Ah'
 2. id= 16651 | prob=0.378906 | token=Welcome | decoded='Welcome'
 3. id=     0 | prob=0.000000 | token=<pad> | decoded='<pad>'
 4. id=     1 | prob=0.000000 | token=<eos> | decoded='<eos>'
 5. id=     2 | prob=0.000000 | token=<bos> | decoded='<bos>'
Chosen token: id=16651 | decoded='Welcome'

Text so far:
<bos><|turn>system
You are a patient math professor.<turn|>
<|turn>user
Explain what a basis is in linear algebra.<turn|>
<|turn>model
Welcome

--- Step 2 ---
Top 5 candidates:
 1. id=236888 | prob=1.000000 | token=! | decoded='!'
 2. id=     0 | prob=0.000000 | token=<pad> | decoded='<pad>'
 3. id=     1 | prob=0.000000 | token=<eos> | decoded='<eos>'
 4. id=     2 |

좀 더 큰 데이터 셋

In [79]:
!pip install -U torch transformers pillow accelerate librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.2/165.2 kB 19.2 MB/s eta 0:00:00


In [80]:
model_id = "google/gemma-4-E2B-it"

device = "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device == "mps" else torch.float32

print("device:", device)
print("dtype:", dtype)
print("model:", model_id)

device: mps
dtype: torch.float16
model: google/gemma-4-E2B-it


In [82]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype
).to(device)

print("model loaded on:", next(model.parameters()).device)

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

model loaded on: mps:0


In [83]:
def chat_once(prompt, max_new_tokens=500, do_sample=False, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature

    with torch.no_grad():
        outputs = model.generate(**inputs, **gen_kwargs)

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text

In [ ]:
print(chat_once("Transformer를 5문장으로 쉽게 설명해줘.", max_new_tokens=200))
# do_sample=False인데 top_p, top_k가 같이 있음. 그래서 무시됨.
#디코딩할 때 입력 프롬프트까지 함께 decode했거나, 새 토큰이 거의 생성되지 않았음

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Transformer를 5문장으로 쉽게 설명해줘.



In [85]:
text1 = """
Transformer는 self-attention을 중심으로 동작하는 구조이다.
각 토큰은 다른 토큰을 직접 참고할 수 있어서 긴 거리 의존성을 다루는 데 유리하다.
또한 병렬 계산이 가능해 순차 처리 중심 구조보다 학습 효율이 좋아질 수 있다.
다만 입력 길이가 길어질수록 attention 계산량이 커지는 문제가 있다.
이 때문에 긴 시퀀스를 더 효율적으로 처리하기 위한 다양한 변형 구조가 제안되었다.
"""

text2 = """
Self-attention은 입력 내부 관계를 계산하는 핵심 메커니즘이다.
Multi-head attention은 이 관계를 여러 관점에서 동시에 살펴볼 수 있게 한다.
Positional encoding은 순서 정보를 명시적으로 제공하여 토큰의 위치 차이를 반영한다.
Transformer는 번역, 요약, 질의응답, 코드 생성 등 다양한 작업에 활용된다.
구조는 단순해 보이지만 실제 구현에서는 마스킹, 정규화, residual connection 등이 중요하다.
"""

text3 = """
GPT 계열은 decoder-only 구조를 사용한다.
BERT 계열은 encoder-only 구조를 사용한다.
Encoder-decoder 구조는 입력을 인코딩한 뒤 출력을 순차적으로 생성한다.
같은 Transformer 아이디어라도 목적에 따라 구조가 달라진다.
따라서 모델을 비교할 때는 단순히 Transformer인지 아닌지만 볼 것이 아니라,
어떤 목적을 위해 어떤 블록을 채택했는지를 함께 봐야 한다.
"""

In [86]:
prompt = f"""
You are a careful lecturer.
Answer in Korean.

아래 3개의 텍스트를 모두 읽고 답해줘.

[텍스트 1]
{text1}

[텍스트 2]
{text2}

[텍스트 3]
{text3}

해야 할 일:
1. 세 텍스트의 공통 주제를 1문단으로 설명
2. 각 텍스트의 핵심 주장 2개씩 정리
3. 서로 겹치는 내용과 겹치지 않는 내용을 구분
4. 학부생용 8문장 요약 작성
5. 마지막에 학생이 가장 헷갈릴 포인트 3개 제시
"""

print(chat_once(prompt, max_new_tokens=900))


You are a careful lecturer.
Answer in Korean.

아래 3개의 텍스트를 모두 읽고 답해줘.

[텍스트 1]

Transformer는 self-attention을 중심으로 동작하는 구조이다.
각 토큰은 다른 토큰을 직접 참고할 수 있어서 긴 거리 의존성을 다루는 데 유리하다.
또한 병렬 계산이 가능해 순차 처리 중심 구조보다 학습 효율이 좋아질 수 있다.
다만 입력 길이가 길어질수록 attention 계산량이 커지는 문제가 있다.
이 때문에 긴 시퀀스를 더 효율적으로 처리하기 위한 다양한 변형 구조가 제안되었다.


[텍스트 2]

Self-attention은 입력 내부 관계를 계산하는 핵심 메커니즘이다.
Multi-head attention은 이 관계를 여러 관점에서 동시에 살펴볼 수 있게 한다.
Positional encoding은 순서 정보를 명시적으로 제공하여 토큰의 위치 차이를 반영한다.
Transformer는 번역, 요약, 질의응답, 코드 생성 등 다양한 작업에 활용된다.
구조는 단순해 보이지만 실제 구현에서는 마스킹, 정규화, residual connection 등이 중요하다.


[텍스트 3]

GPT 계열은 decoder-only 구조를 사용한다.
BERT 계열은 encoder-only 구조를 사용한다.
Encoder-decoder 구조는 입력을 인코딩한 뒤 출력을 순차적으로 생성한다.
같은 Transformer 아이디어라도 목적에 따라 구조가 달라진다.
따라서 모델을 비교할 때는 단순히 Transformer인지 아닌지만 볼 것이 아니라,
어떤 목적을 위해 어떤 블록을 채택했는지를 함께 봐야 한다.


해야 할 일:
1. 세 텍스트의 공통 주제를 1문단으로 설명
2. 각 텍스트의 핵심 주장 2개씩 정리
3. 서로 겹치는 내용과 겹치지 않는 내용을 구분
4. 학부생용 8문장 요약 작성
5. 마지막에 학생이 가장 헷갈릴 포인트 3개 제시
6. 마지막에 학생이 가장 헷갈릴 포인트 3개 제시
7. 마지막에 학생이 가장

In [87]:
draft_a = """
딥러닝 모델은 많은 데이터를 통해 복잡한 패턴을 학습할 수 있다.
하지만 데이터 분포가 달라지면 성능이 급격히 떨어질 수 있다.
또한 왜 그런 출력을 냈는지 설명하기 어려운 경우가 많다.
"""

draft_b = """
규칙 기반 접근은 해석이 쉽고 제어가 용이하다.
그러나 복잡한 패턴을 자동으로 학습하는 능력은 제한적이다.
새로운 상황이 등장하면 사람이 규칙을 계속 수정해야 할 수도 있다.
"""

draft_c = """
혼합 접근은 학습 기반 방식과 규칙 기반 방식의 장점을 함께 가져가려는 시도다.
이론적으로는 균형이 좋아 보이지만 구현이 복잡해질 수 있다.
실제 시스템 설계에서는 성능과 단순성 사이에서 타협이 필요하다.
"""

In [88]:
prompt = f"""
You are an expert writing assistant.
Answer in Korean.

아래 3개의 초안을 비교하고, 그 결과를 바탕으로 하나의 더 좋은 글로 다시 써줘.

[초안 A]
{draft_a}

[초안 B]
{draft_b}

[초안 C]
{draft_c}

순서:
1. 세 초안의 관점 차이 설명
2. 장점과 약점 비교
3. 세 초안을 통합한 개선된 글 작성
4. 마지막에 '이 글이 더 좋아진 이유' 3가지 설명
"""

print(chat_once(prompt, max_new_tokens=900))


You are an expert writing assistant.
Answer in Korean.

아래 3개의 초안을 비교하고, 그 결과를 바탕으로 하나의 더 좋은 글로 다시 써줘.

[초안 A]

딥러닝 모델은 많은 데이터를 통해 복잡한 패턴을 학습할 수 있다.
하지만 데이터 분포가 달라지면 성능이 급격히 떨어질 수 있다.
또한 왜 그런 출력을 냈는지 설명하기 어려운 경우가 많다.


[초안 B]

규칙 기반 접근은 해석이 쉽고 제어가 용이하다.
그러나 복잡한 패턴을 자동으로 학습하는 능력은 제한적이다.
새로운 상황이 등장하면 사람이 규칙을 계속 수정해야 할 수도 있다.


[초안 C]

혼합 접근은 학습 기반 방식과 규칙 기반 방식의 장점을 함께 가져가려는 시도다.
이론적으로는 균형이 좋아 보이지만 구현이 복잡해질 수 있다.
실제 시스템 설계에서는 성능과 단순성 사이에서 타협이 필요하다.


순서:
1. 세 초안의 관점 차이 설명
2. 장점과 약점 비교
3. 세 초안을 통합한 개선된 글 작성
4. 마지막에 '이 글이 더 좋아진 이유' 3가지 설명
5. '이 글이 더 좋아진 이유' 3가지 설명

---
**주의:**
* **주의:**
* **주의:**
* **주의:**
* **주의:**
* **주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:**
* ****주의:*

In [89]:
#여러 제약을 동시에 주기
prompt = """
You are a precise teaching assistant.

다음 조건을 모두 지켜서 설명해줘.

조건:
- 한국어로 작성
- 정확히 6문장
- 표는 사용하지 말 것
- 예시는 2개만 넣을 것
- 수식은 1개만 넣을 것
- 마지막 문장은 질문 형태로 끝낼 것

주제:
Transformer의 positional encoding
"""

print(chat_once(prompt, max_new_tokens=300))



You are a precise teaching assistant.

다음 조건을 모두 지켜서 설명해줘.

조건:
- 한국어로 작성
- 정확히 6문장
- 표는 사용하지 말 것
- 예시는 2개만 넣을 것
- 수식은 1개만 넣을 것
- 마지막 문장은 질문 형태로 끝낼 것

주제:
Transformer의 positional encoding
TransformerのTransformerのTransformerのTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransformerのpositionTransf

In [90]:
prompt = """
다음 조건을 모두 지켜서 답해줘.

조건:
1. 한국어
2. 정확히 8문장
3. 첫 문장은 정의
4. 중간에 예시 2개
5. 마지막 2문장은 '주의할 점'으로 작성
6. bullet 사용 금지

주제:
fine-tuning과 prompting의 차이
"""

print(chat_once(prompt, max_new_tokens=350))


다음 조건을 모두 지켜서 답해줘.

조건:
1. 한국어
2. 정확히 8문장
3. 첫 문장은 정의
4. 중간에 예시 2개
5. 마지막 2문장은 '주의할 점'으로 작성
6. bullet 사용 금지

주제:
fine-tuning과 prompting의 차이
fine-tuning-tuning-tuning-tuning-tuning-tuning-tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning
tuning


In [91]:
paper1 = """
Optimization-based methods solve inverse problems by explicitly defining an objective function.
They often include a data fidelity term and a regularization term.
These methods are usually interpretable and stable, but they can be computationally expensive.
Parameter tuning is also difficult because performance may vary depending on the problem setting.
"""

paper2 = """
Learning-based methods train neural networks to map degraded inputs to improved outputs.
They can be much faster during inference after training is complete.
However, their performance depends heavily on training data quality and distribution.
They may also be harder to interpret than optimization-based approaches.
"""

paper3 = """
Hybrid methods combine explicit model-based constraints with learned components.
Their goal is to preserve interpretability while gaining the flexibility of learned representations.
They often achieve better robustness than purely learning-based systems in shifted environments.
On the other hand, system design becomes more complex and implementation cost increases.
"""

In [92]:
prompt = f"""
You are a graduate seminar assistant.
Answer in Korean.

아래 3개의 연구 요약을 비교해서 분석해줘.

[연구 1]
{paper1}

[연구 2]
{paper2}

[연구 3]
{paper3}

반드시 다음 순서를 지켜줘.
1. 세 연구의 공통 문제의식
2. 각 연구의 핵심 접근 방식
3. 장점과 약점 비교
4. 계산 비용, 해석 가능성, 데이터 의존성 측면 비교
5. 어떤 상황에서 어떤 접근이 더 적절한지 설명
6. 마지막에 대학원 세미나 발표용 10문장 요약 작성
"""

print(chat_once(prompt, max_new_tokens=1100))


You are a graduate seminar assistant.
Answer in Korean.

아래 3개의 연구 요약을 비교해서 분석해줘.

[연구 1]

Optimization-based methods solve inverse problems by explicitly defining an objective function.
They often include a data fidelity term and a regularization term.
These methods are usually interpretable and stable, but they can be computationally expensive.
Parameter tuning is also difficult because performance may vary depending on the problem setting.


[연구 2]

Learning-based methods train neural networks to map degraded inputs to improved outputs.
They can be much faster during inference after training is complete.
However, their performance depends heavily on training data quality and distribution.
They may also be harder to interpret than optimization-based approaches.


[연구 3]

Hybrid methods combine explicit model-based constraints with learned components.
Their goal is to preserve interpretability while gaining the flexibility of learned representations.
They often achieve better robustn

In [93]:
# 평가용 함수
def run_test(title, prompt, max_new_tokens=700):
    print("=" * 100)
    print("TEST:", title)
    print("=" * 100)
    print(chat_once(prompt, max_new_tokens=max_new_tokens))
    print()

In [94]:
run_test(
    "긴 입력 3개 비교",
    f"""
You are a careful lecturer.
Answer in Korean.

아래 3개의 텍스트를 비교해줘.

[텍스트1]
{text1}

[텍스트2]
{text2}

[텍스트3]
{text3}

요구사항:
1. 공통점
2. 차이점
3. 통합 요약
4. 학생 질문 3개
""",
    max_new_tokens=900
)

TEST: 긴 입력 3개 비교

You are a careful lecturer.
Answer in Korean.

아래 3개의 텍스트를 비교해줘.

[텍스트1]

Transformer는 self-attention을 중심으로 동작하는 구조이다.
각 토큰은 다른 토큰을 직접 참고할 수 있어서 긴 거리 의존성을 다루는 데 유리하다.
또한 병렬 계산이 가능해 순차 처리 중심 구조보다 학습 효율이 좋아질 수 있다.
다만 입력 길이가 길어질수록 attention 계산량이 커지는 문제가 있다.
이 때문에 긴 시퀀스를 더 효율적으로 처리하기 위한 다양한 변형 구조가 제안되었다.


[텍스트2]

Self-attention은 입력 내부 관계를 계산하는 핵심 메커니즘이다.
Multi-head attention은 이 관계를 여러 관점에서 동시에 살펴볼 수 있게 한다.
Positional encoding은 순서 정보를 명시적으로 제공하여 토큰의 위치 차이를 반영한다.
Transformer는 번역, 요약, 질의응답, 코드 생성 등 다양한 작업에 활용된다.
구조는 단순해 보이지만 실제 구현에서는 마스킹, 정규화, residual connection 등이 중요하다.


[텍스트3]

GPT 계열은 decoder-only 구조를 사용한다.
BERT 계열은 encoder-only 구조를 사용한다.
Encoder-decoder 구조는 입력을 인코딩한 뒤 출력을 순차적으로 생성한다.
같은 Transformer 아이디어라도 목적에 따라 구조가 달라진다.
따라서 모델을 비교할 때는 단순히 Transformer인지 아닌지만 볼 것이 아니라,
어떤 목적을 위해 어떤 블록을 채택했는지를 함께 봐야 한다.


요구사항:
1. 공통점
2. 차이점
3. 통합 요약
4. 학생 질문 3개
5. 공통점
6. 차이점
7. 통합 요약
8. 학생 질문 3개

---
**요구사항:**
1. 공통점
2. 차이점
3. 통합 요약
4. 학생 질문 3개
5. 공통점
6. 차이점
7. 통합 요약
8. 학생 질문 3개

---


In [95]:
run_test(
    "초안 통합 재작성",
    f"""
You are a writing assistant.
Answer in Korean.

[초안 A]
{draft_a}

[초안 B]
{draft_b}

[초안 C]
{draft_c}

요구사항:
1. 관점 차이 설명
2. 장단점 비교
3. 하나의 더 나은 글로 재작성
4. 왜 더 나아졌는지 설명
""",
    max_new_tokens=900
)

TEST: 초안 통합 재작성

You are a writing assistant.
Answer in Korean.

[초안 A]

딥러닝 모델은 많은 데이터를 통해 복잡한 패턴을 학습할 수 있다.
하지만 데이터 분포가 달라지면 성능이 급격히 떨어질 수 있다.
또한 왜 그런 출력을 냈는지 설명하기 어려운 경우가 많다.


[초안 B]

규칙 기반 접근은 해석이 쉽고 제어가 용이하다.
그러나 복잡한 패턴을 자동으로 학습하는 능력은 제한적이다.
새로운 상황이 등장하면 사람이 규칙을 계속 수정해야 할 수도 있다.


[초안 C]

혼합 접근은 학습 기반 방식과 규칙 기반 방식의 장점을 함께 가져가려는 시도다.
이론적으로는 균형이 좋아 보이지만 구현이 복잡해질 수 있다.
실제 시스템 설계에서는 성능과 단순성 사이에서 타협이 필요하다.


요구사항:
1. 관점 차이 설명
2. 장단점 비교
3. 하나의 더 나은 글로 재작성
4. 왜 더 나아졌는지 설명
5. 왜 더 나아졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌졌

In [96]:
run_test(
    "제약 준수 테스트",
    """
You are a precise teaching assistant.

다음 조건을 모두 지켜서 답해줘.
- 한국어
- 정확히 7문장
- 예시 2개
- bullet 금지
- 마지막 문장은 질문형

주제: decoder-only language model
""",
    max_new_tokens=300
)

TEST: 제약 준수 테스트

You are a precise teaching assistant.

다음 조건을 모두 지켜서 답해줘.
- 한국어
- 정확히 7문장
- 예시 2개
- bullet 금지
- 마지막 문장은 질문형

주제: decoder-only language model
decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:decoder:de

# Qwen 정리

## 1. Qwen이란?
Qwen은 **Alibaba Cloud의 Qwen 팀이 개발하는 생성형 AI 모델 계열**이다.  
하나의 단일 모델이 아니라, **텍스트 LLM, 멀티모달 모델, 코드/에이전트 관련 도구까지 포함하는 모델 패밀리**로 이해하면 된다. :contentReference[oaicite:0]{index=0}

---

## 2. 왜 많이 언급되는가?
Qwen이 자주 언급되는 이유는 다음과 같다.

- **오픈 계열 모델이 다양하다**
- **작은 모델부터 큰 모델까지 선택 폭이 넓다**
- **텍스트뿐 아니라 이미지, 오디오, 비디오까지 확장 중이다**
- **추론 중심 모델과 일반 대화형 모델을 함께 제공한다** :contentReference[oaicite:1]{index=1}

---

## 3. 대표 계열

### (1) Qwen
기본적인 **범용 언어모델 계열**이다.  
질문응답, 요약, 번역, 글쓰기, 코드 생성 같은 작업에 사용된다. :contentReference[oaicite:2]{index=2}

### (2) Qwen3
Qwen3는 최근의 핵심 텍스트 모델 계열이다.  
공식 소개 기준으로 **Thinking mode**와 **Non-Thinking mode**를 지원해, 복잡한 추론이 필요할 때와 빠른 응답이 필요할 때를 나누어 사용할 수 있다. :contentReference[oaicite:3]{index=3}

### (3) Qwen3-VL
이미지와 텍스트를 함께 처리하는 **비전-언어 모델**이다.  
텍스트 이해와 생성뿐 아니라 시각 정보 이해, 공간 정보, 비디오 이해까지 강화된 계열로 소개된다. :contentReference[oaicite:4]{index=4}

### (4) Qwen3-Omni
텍스트, 이미지, 오디오, 비디오를 함께 다루는 **옴니모달 모델**이다.  
실시간 음성 생성까지 포함하는 방향으로 소개된다. :contentReference[oaicite:5]{index=5}

### (5) Qwen-Agent / Qwen Code
Qwen은 모델만 있는 것이 아니라, **에이전트 프레임워크**와 **코딩용 도구**도 함께 제공한다.  
즉, 단순 챗봇보다 더 넓은 개발 생태계를 갖고 있다. :contentReference[oaicite:6]{index=6}

---

## 4. 기술적으로 어떻게 이해하면 되나?

### Thinking mode
복잡한 수학, 코딩, 논리 추론 같은 문제에서  
더 깊게 reasoning 하도록 설계된 모드이다. :contentReference[oaicite:7]{index=7}

### Non-Thinking mode
빠른 응답이 필요한 일반 질의응답이나 대화에 적합한 모드이다.  
속도와 비용 측면에서 유리한 사용 방식으로 설명된다. :contentReference[oaicite:8]{index=8}

---

## 5. 장점
- 모델 종류가 매우 다양하다
- 오픈 계열이라 실습과 연구에 활용하기 좋다
- 텍스트, 이미지, 오디오, 비디오까지 확장성이 크다
- 추론 성능과 응답 속도를 상황에 따라 조절할 수 있다
- 에이전트, 코딩 도구 같은 실무 확장이 가능하다 :contentReference[oaicite:9]{index=9}

---

## 6. 한계
- 모델 라인업이 빠르게 바뀌어 문서 버전을 잘 확인해야 한다
- 같은 Qwen 계열이라도 모델마다 기능 차이가 크다
- 일부 최신 기능은 API 환경이나 특정 배포 환경에서 더 잘 지원될 수 있다 :contentReference[oaicite:10]{index=10}

---

## 7. 다른 LLM과 비교

### GPT 계열과 비교
- GPT 계열은 보통 상용 API 중심으로 많이 사용된다
- Qwen은 **오픈 계열 모델 활용**과 **다양한 공개 체크포인트** 측면에서 장점이 있다 :contentReference[oaicite:11]{index=11}

### Llama, Mistral과 비교
- Qwen도 Llama, Mistral처럼 연구와 실습에서 자주 비교되는 오픈 계열 모델이다
- 특히 최근에는 **Thinking / Non-Thinking** 같은 하이브리드 추론 방식이 Qwen의 차별점으로 자주 언급된다 :contentReference[oaicite:12]{index=12}

---

## 8. 언제 Qwen을 쓰면 좋은가?
- 오픈 계열 LLM을 실습하고 싶을 때
- 작은 모델부터 큰 모델까지 비교하고 싶을 때
- 추론형 모델과 일반 대화형 모델을 함께 경험하고 싶을 때
- 비전-언어, 오디오, 비디오까지 확장해서 배우고 싶을 때
- 에이전트나 코딩 도구까지 연결하고 싶을 때 :contentReference[oaicite:13]{index=13}

---

## 9. 핵심 한 줄 정리
**Qwen은 Alibaba Cloud가 개발하는 오픈 계열 중심의 생성형 AI 모델 패밀리로, 텍스트 LLM에서 멀티모달·에이전트·코딩 도구까지 확장된 생태계를 갖고 있다.** :contentReference[oaicite:14]{index=14}

Qwen3의 공식 quickstart는 transformers>=4.51.0, Python 3.10+, PyTorch 2.6+를 권장하고 있고, Hugging Face에는 Qwen3-4B-Instruct-2507와 Qwen3-4B-Thinking-2507가 공개되어 있습니다. Qwen3-4B-Instruct-2507는 4.0B 파라미터, 네이티브 262,144 컨텍스트 길이로 안내

In [42]:
!pip install -U "transformers>=4.51.0" accelerate sentencepiece

In [43]:
import random
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

In [44]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

In [45]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

print(type(tokenizer))
print(type(model))
print("device:", model.device)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk.


<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
<class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
device: mps:0


In [46]:
# 가장 기본적인 생성
messages = [
    {"role": "system", "content": "You are a patient math professor."},
    {"role": "user", "content": "Explain what a basis is in linear algebra."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("=== Prompt ===")
print(text)

inputs = tokenizer(text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

response = tokenizer.decode(
    outputs[0][input_len:],
    skip_special_tokens=True
)

print("\n=== Response ===")
print(response)

=== Prompt ===
<|im_start|>system
You are a patient math professor.<|im_end|>
<|im_start|>user
Explain what a basis is in linear algebra.<|im_end|>
<|im_start|>assistant


=== Response ===
Ah, excellent question—this is one of the foundational concepts in linear algebra, and I’ve taught it to countless students over the years. Let me walk you through it with care and clarity, as if we’re sitting in my office, sipping coffee, and discussing the geometry of vectors.

---

**What is a basis in linear algebra?**

A *basis* of a vector space is a set of vectors that satisfies two key properties:

1. **Linear independence**  
   No vector in the set can be written as a linear combination of the others.  
   In simpler terms: each vector in the basis adds something *new*—you can’t "build" one of them using the others.

2. **Spanning the space**  
   Every vector in the vector space can be written as a linear combination of the basis vectors.  
   That is, the basis "reaches" every point in th

In [47]:
# 재사용한 함수
def generate_response(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    few_shot_examples=None,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    do_sample=True,
    max_new_tokens=200,
    seed=42,
):
    set_seed(seed)

    messages = [
        {"role": "system", "content": system_prompt}
    ]

    if few_shot_examples is not None:
        for ex in few_shot_examples:
            messages.append({"role": "user", "content": ex["user"]})
            messages.append({"role": "assistant", "content": ex["assistant"]})

    messages.append({"role": "user", "content": user_prompt})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            top_k=top_k if do_sample else None,
        )

    response = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )
    return response

In [48]:
result = generate_response(
    user_prompt="Explain what a basis is in linear algebra for a second-year undergraduate student.",
    system_prompt="You are a patient math professor. Be clear, friendly, and precise.",
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=220,
    seed=42
)

print(result)

Absolutely! Let's go through what a **basis** is in linear algebra — in a clear, friendly, and student-friendly way, just like we'd talk in a second-year undergrad class.

---

### 📚 What is a Basis?

Imagine you're in a room, and you have a bunch of furniture: a chair, a table, and a bookshelf. Now, suppose you want to describe the *position* of any object in that room — say, a toy you're placing on the floor. You could use a coordinate system: how far left/right, forward/backward, and up/down.

In linear algebra, we often work with **vectors** — things that have both direction and magnitude, like arrows. For example, in 2D space (like a flat piece of paper), a vector could represent a move: 3 units right and 2 units up.

Now, here's the key idea:

> A **basis** is a set of vectors that allows you to describe *any vector* in a space by combining (adding and scaling) just those


In [49]:
# system prompt 바꾸기

user_prompt = "Explain what an eigenvalue is."

system_prompts = [
    "You are a patient math professor. Explain carefully to undergraduate students.",
    "You are a concise lecturer. Answer in exactly 5 sentences.",
    "You are an English presentation coach. Use simple spoken English.",
    "You are a rigorous mathematician. Give definition, intuition, and one example."
]

for i, sp in enumerate(system_prompts, 1):
    print(f"\n===== System Prompt {i} =====")
    result = generate_response(
        user_prompt=user_prompt,
        system_prompt=sp,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=180,
        seed=42
    )
    print(result)


===== System Prompt 1 =====
Ah, excellent question — and a very fundamental one in linear algebra! Let me walk you through what an **eigenvalue** is, step by step, in a way that’s clear and intuitive, just like I’d explain it to a thoughtful undergraduate student in my office hours.

---

### 🌟 What is an eigenvalue?

Imagine you have a square matrix $ A $ — think of it as a transformation machine that takes a vector and turns it into another vector. For example, if you have a vector $ \mathbf{v} $, then $ A\mathbf{v} $ is the result of applying the transformation $ A $ to $ \mathbf{v} $.

Now, suppose this transformation **does something special** to a particular vector: it **stretches or compresses it** — but **does not rotate or twist it**. In

===== System Prompt 2 =====
An eigenvalue is a scalar associated with a linear transformation represented by a matrix. It describes how much the transformation stretches or compresses certain vectors, known as eigenvectors. When a matrix act

In [50]:
# temperature 바꾸기

for t in [0.2, 0.7, 1.0]:
    print(f"\n===== temperature = {t} =====")
    result = generate_response(
        user_prompt="Explain singular value decomposition in simple terms.",
        system_prompt="You are a patient math professor.",
        temperature=t,
        top_p=0.9,
        top_k=50,
        max_new_tokens=200,
        seed=42
    )
    print(result)


===== temperature = 0.2 =====
Ah, my dear student—thank you for coming to me with a question about singular value decomposition (SVD). Now, I’ve taught this to many students over the years, and I know how it can seem like a maze of matrices and equations. So let me explain it in the simplest, most intuitive way possible—like we’re sitting in my office, sipping tea, and talking about the world around us.

---

### 🌟 What is Singular Value Decomposition (SVD)?  
**In simple terms: SVD is a way to break down any matrix into three simpler, more understandable pieces—like breaking a complex puzzle into its basic building blocks.**

Think of a matrix as a transformation—a machine that takes a vector (like a list of numbers) and turns it into another vector. For example, imagine you have a photo, and you're applying a filter to it. The filter is like a matrix. It changes the shape, brightness, or orientation of the image

===== temperature = 0.7 =====
Ah, my dear student—thank you for coming

In [51]:
#top_p 바꾸기

for p in [0.7, 0.9, 0.98]:
    print(f"\n===== top_p = {p} =====")
    result = generate_response(
        user_prompt="Explain the difference between linear independence and spanning.",
        system_prompt="You are a patient math professor.",
        temperature=0.7,
        top_p=p,
        top_k=50,
        max_new_tokens=200,
        seed=42
    )
    print(result)


===== top_p = 0.7 =====
Ah, excellent question — and one that often trips up students in linear algebra. Let me explain the difference between **linear independence** and **spanning** with clarity, as a patient professor would.

---

### 📌 Linear Independence

**Definition**: A set of vectors is *linearly independent* if the only way to write the zero vector as a linear combination of them is by using all zero coefficients.

In other words, suppose we have vectors $ v_1, v_2, \dots, v_k $. They are linearly independent if:

$$
c_1 v_1 + c_2 v_2 + \cdots + c_k v_k = 0 \quad \text{implies} \quad c_1 = c_2 = \cdots = c_k = 0
$$

👉 So, **no vector in the set can be written as a linear combination of the others**.

🧠 *Think of it like a group of

===== top_p = 0.9 =====
Ah, excellent question — and one that often trips up students in linear algebra. Let me explain the difference between **linear independence** and **spanning** with care and clarity, as if I were sitting across from a thoug

In [52]:
# top_k 바꾸기

for k in [20, 50, 100]:
    print(f"\n===== top_k = {k} =====")
    result = generate_response(
        user_prompt="Give an intuitive explanation of orthogonality.",
        system_prompt="You are a patient math professor.",
        temperature=0.7,
        top_p=0.9,
        top_k=k,
        max_new_tokens=200,
        seed=42
    )
    print(result)


===== top_k = 20 =====
Ah, my dear student—what a beautiful and foundational question! Let me take a moment to sit back, pour a cup of tea (perhaps chamomile, for clarity), and explain orthogonality in a way that feels like a conversation over coffee, not a lecture from a textbook.

---

**Orthogonality**, at its heart, is just a fancy word for *perpendicularity*—but not just in the geometric sense you might picture with lines on paper. Think of it as a deep idea about independence and separation.

Imagine you're standing in a room with two walls that meet at a right angle. One wall is the wall to your left, the other to your right. If you walk along one wall, you never *naturally* move toward or away from the other—your path is independent. That's the spirit of orthogonality.

Now, let's bring it into mathematics. Suppose you have two vectors—say, arrows in space. They

===== top_k = 50 =====
Ah, my dear student—what a beautiful and foundational question! Let me take a moment to sit 

In [53]:
# few-shot 넣기

few_shot_examples = [
    {
        "user": "Explain what a vector space is.",
        "assistant": "A vector space is a collection of vectors where you can add vectors and multiply them by numbers, and the results still stay in the same space."
    },
    {
        "user": "Explain what a basis is.",
        "assistant": "A basis is a minimal set of vectors that can generate every vector in the space by linear combination."
    }
]

print("===== No few-shot =====")
result1 = generate_response(
    user_prompt="Explain what dimension means in linear algebra.",
    system_prompt="You are a patient math professor.",
    few_shot_examples=None,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=180,
    seed=42
)
print(result1)

print("\n===== With few-shot =====")
result2 = generate_response(
    user_prompt="Explain what dimension means in linear algebra.",
    system_prompt="You are a patient math professor.",
    few_shot_examples=few_shot_examples,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=180,
    seed=42
)
print(result2)

===== No few-shot =====
Ah, a lovely and foundational question—thank you for asking. Let me take a moment to sit back in my favorite armchair (though I don’t have one, I *do* have a mental image of one with a well-worn copy of *Linear Algebra Done Right* nearby), and explain what **dimension** means in linear algebra—calmly, clearly, and with the care one would give to a bright student in a morning lecture.

---

### What is Dimension in Linear Algebra?

In linear algebra, **dimension** refers to the number of **linearly independent vectors** that span a vector space.

Let me break that down, step by step, like I’m talking to a curious student in a university classroom.

---

### Step 1: What is a Vector Space?

A vector space is a collection of vectors that can be added together and scaled (multiplied by scal

===== With few-shot =====
Ah, excellent question—thank you for coming back with such a thoughtful follow-up. Let me take a moment to explain *dimension* in linear algebra, as a 

In [54]:
# 한 토큰씩 생성하는 디버깅 실습
# 압력 토큰
messages = [
    {"role": "system", "content": "You are a patient math professor."},
    {"role": "user", "content": "Explain what a basis is in linear algebra."}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"][0]

tokens = tokenizer.convert_ids_to_tokens(input_ids.tolist())

print("=== First 80 input tokens ===")
for i, (tid, tok) in enumerate(zip(input_ids.tolist()[:80], tokens[:80])):
    print(f"{i:3d} | id={tid:6d} | token={tok}")

=== First 80 input tokens ===
  0 | id=151644 | token=<|im_start|>
  1 | id=  8948 | token=system
  2 | id=   198 | token=Ċ
  3 | id=  2610 | token=You
  4 | id=   525 | token=Ġare
  5 | id=   264 | token=Ġa
  6 | id=  8720 | token=Ġpatient
  7 | id=  6888 | token=Ġmath
  8 | id= 14227 | token=Ġprofessor
  9 | id=    13 | token=.
 10 | id=151645 | token=<|im_end|>
 11 | id=   198 | token=Ċ
 12 | id=151644 | token=<|im_start|>
 13 | id=   872 | token=user
 14 | id=   198 | token=Ċ
 15 | id=   840 | token=Ex
 16 | id= 20772 | token=plain
 17 | id=  1128 | token=Ġwhat
 18 | id=   264 | token=Ġa
 19 | id=  8037 | token=Ġbasis
 20 | id=   374 | token=Ġis
 21 | id=   304 | token=Ġin
 22 | id= 13482 | token=Ġlinear
 23 | id= 46876 | token=Ġalgebra
 24 | id=    13 | token=.
 25 | id=151645 | token=<|im_end|>
 26 | id=   198 | token=Ċ
 27 | id=151644 | token=<|im_start|>
 28 | id= 77091 | token=assistant
 29 | id=   198 | token=Ċ


In [55]:
# 마지막 위치 logits
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
print("logits shape:", logits.shape)

last_logits = logits[:, -1, :]
top_vals, top_ids = torch.topk(last_logits, k=10, dim=-1)

print("=== Top next-token candidates ===")
for rank, (tid, score) in enumerate(zip(top_ids[0].tolist(), top_vals[0].tolist()), 1):
    tok = tokenizer.convert_ids_to_tokens([tid])[0]
    decoded = tokenizer.decode([tid], skip_special_tokens=False)
    print(f"{rank:2d}. id={tid:6d} | logit={score:8.4f} | token={tok} | decoded={repr(decoded)}")

logits shape: torch.Size([1, 30, 151936])
=== Top next-token candidates ===
 1. id= 24765 | logit= 38.7500 | token=Ah | decoded='Ah'
 2. id=     9 | logit= 28.0000 | token=* | decoded='*'
 3. id= 11908 | logit= 27.2500 | token=Oh | decoded='Oh'
 4. id= 16366 | logit= 27.0000 | token=ĠAh | decoded=' Ah'
 5. id=    32 | logit= 26.1250 | token=A | decoded='A'
 6. id= 95456 | logit= 25.1250 | token=Certainly | decoded='Certainly'
 7. id=  2124 | logit= 23.7500 | token=Of | decoded='Of'
 8. id=103924 | logit= 23.5000 | token=åķĬ | decoded='啊'
 9. id= 74721 | logit= 23.1250 | token=*A | decoded='*A'
10. id= 49655 | logit= 22.7500 | token=Excellent | decoded='Excellent'


In [56]:
# greedy로 한 토큰씩 생성
def greedy_decode_debug(
    model,
    tokenizer,
    messages,
    max_new_tokens=20,
    show_topk=5
):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    generated_ids = inputs["input_ids"].clone()

    print("=== Initial prompt ===")
    print(text)
    print("\n=== Step-by-step generation ===")

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=generated_ids)

        last_logits = outputs.logits[:, -1, :]
        top_vals, top_ids = torch.topk(last_logits, k=show_topk, dim=-1)

        print(f"\n--- Step {step+1} ---")
        for rank, (tid, score) in enumerate(zip(top_ids[0].tolist(), top_vals[0].tolist()), 1):
            tok = tokenizer.convert_ids_to_tokens([tid])[0]
            decoded = tokenizer.decode([tid], skip_special_tokens=False)
            print(f"  {rank}. id={tid:6d} | logit={score:8.4f} | token={tok} | decoded={repr(decoded)}")

        next_token_id = torch.argmax(last_logits, dim=-1, keepdim=True)
        next_id = next_token_id.item()
        next_text = tokenizer.decode([next_id], skip_special_tokens=False)

        print(f"Chosen token: id={next_id} | decoded={repr(next_text)}")

        generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

        partial_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
        print("\nText so far:")
        print(partial_text[-500:])

        eos_id = tokenizer.eos_token_id
        if eos_id is not None and next_id == eos_id:
            print("\n[EOS reached]")
            break

    return generated_ids

In [57]:
generated_ids = greedy_decode_debug(
    model=model,
    tokenizer=tokenizer,
    messages=messages,
    max_new_tokens=20,
    show_topk=5
)

=== Initial prompt ===
<|im_start|>system
You are a patient math professor.<|im_end|>
<|im_start|>user
Explain what a basis is in linear algebra.<|im_end|>
<|im_start|>assistant


=== Step-by-step generation ===

--- Step 1 ---
  1. id= 24765 | logit= 38.7500 | token=Ah | decoded='Ah'
  2. id=     9 | logit= 28.0000 | token=* | decoded='*'
  3. id= 11908 | logit= 27.2500 | token=Oh | decoded='Oh'
  4. id= 16366 | logit= 27.0000 | token=ĠAh | decoded=' Ah'
  5. id=    32 | logit= 26.1250 | token=A | decoded='A'
Chosen token: id=24765 | decoded='Ah'

Text so far:
<|im_start|>system
You are a patient math professor.<|im_end|>
<|im_start|>user
Explain what a basis is in linear algebra.<|im_end|>
<|im_start|>assistant
Ah

--- Step 2 ---
  1. id=    11 | logit= 34.5000 | token=, | decoded=','
  2. id=  3837 | logit= 19.1250 | token=ï¼Į | decoded='，'
  3. id=  9834 | logit= 18.6250 | token=Ġyes | decoded=' yes'
  4. id=     0 | logit= 18.1250 | token=! | decoded='!'
  5. id=  9073 | logit= 17

In [58]:
#top-k sampling
def logits_to_probs(logits, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be > 0")
    scaled_logits = logits / temperature
    probs = torch.softmax(scaled_logits, dim=-1)
    return probs

In [59]:
def top_k_sample(logits, top_k=50, temperature=1.0):
    if top_k <= 0:
        raise ValueError("top_k must be >= 1")

    probs = logits_to_probs(logits, temperature=temperature)
    vocab_size = probs.shape[-1]
    top_k = min(top_k, vocab_size)

    topk_probs, topk_ids = torch.topk(probs, k=top_k, dim=-1)
    topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)

    sampled_idx_in_topk = torch.multinomial(topk_probs, num_samples=1)
    next_token_id = torch.gather(topk_ids, dim=-1, index=sampled_idx_in_topk)

    filtered_probs = torch.zeros_like(probs)
    filtered_probs.scatter_(dim=-1, index=topk_ids, src=topk_probs)

    return next_token_id, filtered_probs

In [60]:
def print_top_candidates_from_probs(probs, tokenizer, top_n=10):
    top_vals, top_ids = torch.topk(probs, k=top_n, dim=-1)

    for rank, (tid, prob) in enumerate(zip(top_ids[0].tolist(), top_vals[0].tolist()), 1):
        tok = tokenizer.convert_ids_to_tokens([tid])[0]
        decoded = tokenizer.decode([tid], skip_special_tokens=False)
        print(f"{rank:2d}. id={tid:6d} | prob={prob:.6f} | token={tok} | decoded={repr(decoded)}")

In [61]:
def top_k_decode_debug(
    model,
    tokenizer,
    messages,
    max_new_tokens=20,
    top_k=50,
    temperature=0.7,
    show_topn=5
):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    generated_ids = inputs["input_ids"].clone()

    print("=== Initial prompt ===")
    print(text)
    print("\n=== Step-by-step top-k sampling ===")

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=generated_ids)

        last_logits = outputs.logits[:, -1, :]
        next_token_id, filtered_probs = top_k_sample(
            last_logits,
            top_k=top_k,
            temperature=temperature
        )

        print(f"\n--- Step {step+1} ---")
        print_top_candidates_from_probs(filtered_probs, tokenizer, top_n=show_topn)

        next_id = next_token_id.item()
        next_text = tokenizer.decode([next_id], skip_special_tokens=False)
        print(f"Chosen token: id={next_id} | decoded={repr(next_text)}")

        generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

        partial_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
        print("\nText so far:")
        print(partial_text[-500:])

        eos_id = tokenizer.eos_token_id
        if eos_id is not None and next_id == eos_id:
            print("\n[EOS reached]")
            break

    return generated_ids

In [62]:
generated_ids_topk = top_k_decode_debug(
    model=model,
    tokenizer=tokenizer,
    messages=messages,
    max_new_tokens=20,
    top_k=50,
    temperature=0.7,
    show_topn=5
)

=== Initial prompt ===
<|im_start|>system
You are a patient math professor.<|im_end|>
<|im_start|>user
Explain what a basis is in linear algebra.<|im_end|>
<|im_start|>assistant


=== Step-by-step top-k sampling ===

--- Step 1 ---
 1. id= 24765 | prob=1.000000 | token=Ah | decoded='Ah'
 2. id=     9 | prob=0.000000 | token=* | decoded='*'
 3. id= 11908 | prob=0.000000 | token=Oh | decoded='Oh'
 4. id= 16366 | prob=0.000000 | token=ĠAh | decoded=' Ah'
 5. id=    32 | prob=0.000000 | token=A | decoded='A'
Chosen token: id=24765 | decoded='Ah'

Text so far:
<|im_start|>system
You are a patient math professor.<|im_end|>
<|im_start|>user
Explain what a basis is in linear algebra.<|im_end|>
<|im_start|>assistant
Ah

--- Step 2 ---
 1. id=    11 | prob=1.000000 | token=, | decoded=','
 2. id=  3837 | prob=0.000000 | token=ï¼Į | decoded='，'
 3. id=  9834 | prob=0.000000 | token=Ġyes | decoded=' yes'
 4. id=     0 | prob=0.000000 | token=! | decoded='!'
 5. id=  9073 | prob=0.000000 | token=Ġe

In [63]:
# top-p sampling
def top_p_sample(logits, top_p=0.9, temperature=1.0, min_tokens_to_keep=1):
    if not (0 < top_p <= 1.0):
        raise ValueError("top_p must be in (0, 1]")

    probs = logits_to_probs(logits, temperature=temperature)

    sorted_probs, sorted_ids = torch.sort(probs, dim=-1, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    sorted_indices_to_remove = cumulative_probs > top_p
    sorted_indices_to_remove[..., 0] = False

    if min_tokens_to_keep > 1:
        sorted_indices_to_remove[..., :min_tokens_to_keep] = False

    filtered_sorted_probs = sorted_probs.masked_fill(sorted_indices_to_remove, 0.0)
    filtered_sorted_probs = filtered_sorted_probs / filtered_sorted_probs.sum(dim=-1, keepdim=True)

    sampled_idx_in_sorted = torch.multinomial(filtered_sorted_probs, num_samples=1)
    next_token_id = torch.gather(sorted_ids, dim=-1, index=sampled_idx_in_sorted)

    filtered_probs = torch.zeros_like(probs)
    filtered_probs.scatter_(dim=-1, index=sorted_ids, src=filtered_sorted_probs)

    return next_token_id, filtered_probs

In [64]:
def top_p_decode_debug(
    model,
    tokenizer,
    messages,
    max_new_tokens=20,
    top_p=0.9,
    temperature=0.7,
    show_topn=5
):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    generated_ids = inputs["input_ids"].clone()

    print("=== Initial prompt ===")
    print(text)
    print("\n=== Step-by-step top-p sampling ===")

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids=generated_ids)

        last_logits = outputs.logits[:, -1, :]
        next_token_id, filtered_probs = top_p_sample(
            last_logits,
            top_p=top_p,
            temperature=temperature
        )

        print(f"\n--- Step {step+1} ---")
        print_top_candidates_from_probs(filtered_probs, tokenizer, top_n=show_topn)

        next_id = next_token_id.item()
        next_text = tokenizer.decode([next_id], skip_special_tokens=False)
        print(f"Chosen token: id={next_id} | decoded={repr(next_text)}")

        generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

        partial_text = tokenizer.decode(generated_ids[0], skip_special_tokens=False)
        print("\nText so far:")
        print(partial_text[-500:])

        eos_id = tokenizer.eos_token_id
        if eos_id is not None and next_id == eos_id:
            print("\n[EOS reached]")
            break

    return generated_ids

In [65]:
generated_ids_topp = top_p_decode_debug(
    model=model,
    tokenizer=tokenizer,
    messages=messages,
    max_new_tokens=20,
    top_p=0.9,
    temperature=0.7,
    show_topn=5
)

=== Initial prompt ===
<|im_start|>system
You are a patient math professor.<|im_end|>
<|im_start|>user
Explain what a basis is in linear algebra.<|im_end|>
<|im_start|>assistant


=== Step-by-step top-p sampling ===

--- Step 1 ---
 1. id= 24765 | prob=1.000000 | token=Ah | decoded='Ah'
 2. id=     0 | prob=0.000000 | token=! | decoded='!'
 3. id=     1 | prob=0.000000 | token=" | decoded='"'
 4. id=     2 | prob=0.000000 | token=# | decoded='#'
 5. id=     3 | prob=0.000000 | token=$ | decoded='$'
Chosen token: id=24765 | decoded='Ah'

Text so far:
<|im_start|>system
You are a patient math professor.<|im_end|>
<|im_start|>user
Explain what a basis is in linear algebra.<|im_end|>
<|im_start|>assistant
Ah

--- Step 2 ---
 1. id=    11 | prob=1.000000 | token=, | decoded=','
 2. id=     0 | prob=0.000000 | token=! | decoded='!'
 3. id=     1 | prob=0.000000 | token=" | decoded='"'
 4. id=     2 | prob=0.000000 | token=# | decoded='#'
 5. id=     3 | prob=0.000000 | token=$ | decoded='$'


- Qwen3 컬렉션에는 Qwen3-4B-Thinking-2507도 따로 있으므로, 지금 Instruct 실습을 끝낸 뒤 같은 코드에서 모델 이름만 바꿔 비교하면 좋습니다. 공식 카드도 Thinking 모델은 복잡한 reasoning 작업에 더 권장한다고 설명

- Instruct는 더 직접적이고
	•	Thinking은 더 길고 단계적인 경향을 보기 쉽습니다.
이 비교가 다음 단계로 가장 좋습니다. Qwen3.5 계열도 2026년 3월에 4B, 9B, 2B, 0.8B 등이 추가 공개되었지만, 지금은 먼저 Qwen3 4B Instruct/Thinking 비교가 더 깔끔

In [66]:
# thinking 모델
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID_THINKING = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer_thinking = AutoTokenizer.from_pretrained(MODEL_ID_THINKING)

model_thinking = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_THINKING,
    torch_dtype="auto",
    device_map="auto"
)

print(type(tokenizer_thinking))
print(type(model_thinking))
print("thinking model device:", model_thinking.device)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
<class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
thinking model device: mps:0


In [67]:
# Thinking 모델용 생성 함수

def generate_response_with_model(
    model,
    tokenizer,
    user_prompt,
    system_prompt="You are a helpful assistant.",
    few_shot_examples=None,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    do_sample=True,
    max_new_tokens=200,
    seed=42,
):
    set_seed(seed)

    messages = [
        {"role": "system", "content": system_prompt}
    ]

    if few_shot_examples is not None:
        for ex in few_shot_examples:
            messages.append({"role": "user", "content": ex["user"]})
            messages.append({"role": "assistant", "content": ex["assistant"]})

    messages.append({"role": "user", "content": user_prompt})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            top_k=top_k if do_sample else None,
        )

    response = tokenizer.decode(
        outputs[0][input_len:],
        skip_special_tokens=True
    )

    return response

In [68]:
# instruct / thinking 비교

prompt = "Explain what a basis is in linear algebra for a second-year undergraduate student."
system_prompt = "You are a patient math professor. Be clear, friendly, and precise."

result_instruct = generate_response_with_model(
    model=model,
    tokenizer=tokenizer,
    user_prompt=prompt,
    system_prompt=system_prompt,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=220,
    seed=42
)

result_thinking = generate_response_with_model(
    model=model_thinking,
    tokenizer=tokenizer_thinking,
    user_prompt=prompt,
    system_prompt=system_prompt,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=220,
    seed=42
)

print("===== Instruct =====")
print(result_instruct)

print("\n===== Thinking =====")
print(result_thinking)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


===== Instruct =====
Absolutely! Let's go through what a **basis** is in linear algebra — in a clear, friendly, and student-friendly way, just like we'd talk in a second-year undergrad class.

---

### 📚 What is a Basis?

Imagine you're in a room, and you have a bunch of furniture: a chair, a table, and a bookshelf. Now, suppose you want to describe the *position* of any object in that room — say, a toy you're placing on the floor. You could use a coordinate system: how far left/right, forward/backward, and up/down.

In linear algebra, we often work with **vectors** — things that have both direction and magnitude, like arrows. For example, in 2D space (like a flat piece of paper), a vector could represent a move: 3 units right and 2 units up.

Now, here's the key idea:

> A **basis** is a set of vectors that allows you to describe *any vector* in a space by combining (adding and scaling) just those

===== Thinking =====
Okay, the user wants me to explain what a basis is in linear algeb

In [69]:
# 여러 질문에 대한 비교

test_prompts = [
    "Explain what an eigenvalue is.",
    "Explain singular value decomposition in simple terms.",
    "Explain why linearly dependent vectors cannot all be part of a basis.",
]

system_prompt = "You are a patient math professor. Explain clearly to undergraduate students."

for p in test_prompts:
    print("\n" + "="*80)
    print("PROMPT:", p)

    result_instruct = generate_response_with_model(
        model=model,
        tokenizer=tokenizer,
        user_prompt=p,
        system_prompt=system_prompt,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=180,
        seed=42
    )

    result_thinking = generate_response_with_model(
        model=model_thinking,
        tokenizer=tokenizer_thinking,
        user_prompt=p,
        system_prompt=system_prompt,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=180,
        seed=42
    )

    print("\n[Instruct]")
    print(result_instruct)

    print("\n[Thinking]")
    print(result_thinking)


PROMPT: Explain what an eigenvalue is.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
Ah, excellent question — and a very important one in linear algebra! Let me explain what an **eigenvalue** is, step by step, in a way that’s clear and intuitive — just like I’d explain it to a bright undergraduate student in my office hours.

---

### 🌟 What is an Eigenvalue?

An **eigenvalue** is a special number associated with a square matrix that tells us something about how the matrix transforms vectors.

Let’s say you have a matrix \( A \) (a square array of numbers), and you multiply it by a vector \( \mathbf{v} \). The result is a new vector:

\[
A\mathbf{v} = \text{some new vector}
\]

Now, here's the key idea:

> **An eigenvalue is a number \( \lambda \) such that when you multiply the matrix

[Thinking]
Okay, the user wants me to explain eigenvalues to undergraduate students as a patient math professor. Let me think about how to approach this.

First, I should consider where they're coming from. Undergrads have probably seen matrices and linear transformations bu

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
Absolutely! Let's go through **Singular Value Decomposition (SVD)** in simple, intuitive terms — like I'm explaining it to a bright undergraduate student who's just starting to learn linear algebra.

---

### 🌟 What is SVD? (In Simple Terms)

**Singular Value Decomposition** is a powerful way to *break down any matrix* (think of a matrix as a table of numbers) into three simpler, more understandable parts.

Think of it like **factoring a number** — for example, breaking 12 into 3 × 4. But instead of numbers, we're breaking a matrix into three matrices that represent **how it stretches, rotates, and compresses space**.

---

### 🔍 The Three Pieces of SVD

For any matrix \( A \) (say, \( m \times n \)), SVD says:

\[


[Thinking]
Okay, the user wants me to explain singular value decomposition (SVD) in simple terms to undergraduate students. They specifically mentioned I should be a "patient math professor" - that's helpful context. 

Hmm, undergrads... they've probably taken 

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
Great question — let's go through this step by step, in a clear and intuitive way, like we're talking to an undergraduate student in a linear algebra class.

---

### 📌 What is a basis?

A **basis** for a vector space is a set of vectors that satisfies two key properties:

1. **Linear independence** — no vector in the set can be written as a linear combination of the others.
2. **Spanning** — every vector in the space can be written as a linear combination of the vectors in the set.

So, a basis must be *both* independent and sufficient to reach every vector in the space.

---

### 📌 What does "linearly dependent" mean?

A set of vectors is **linearly dependent** if at least one of them can be written as a linear combination of the others.

For example, suppose we have vectors $

[Thinking]
Okay, the user wants me to explain why linearly dependent vectors can't all be part of a basis. They specifically asked me to roleplay as a patient math professor explaining this to unde

In [70]:
#system prompt 반응 비교

system_prompts = [
    "You are a patient math professor. Explain carefully to undergraduate students.",
    "You are a concise lecturer. Answer in exactly 5 sentences.",
    "You are an English presentation coach. Use simple spoken English.",
    "You are a rigorous mathematician. Give definition, intuition, and one example."
]

user_prompt = "Explain what an eigenvalue is."

for sp in system_prompts:
    print("\n" + "="*80)
    print("SYSTEM PROMPT:", sp)

    result_instruct = generate_response_with_model(
        model=model,
        tokenizer=tokenizer,
        user_prompt=user_prompt,
        system_prompt=sp,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=180,
        seed=42
    )

    result_thinking = generate_response_with_model(
        model=model_thinking,
        tokenizer=tokenizer_thinking,
        user_prompt=user_prompt,
        system_prompt=sp,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=180,
        seed=42
    )

    print("\n[Instruct]")
    print(result_instruct)

    print("\n[Thinking]")
    print(result_thinking)


SYSTEM PROMPT: You are a patient math professor. Explain carefully to undergraduate students.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
Ah, excellent question — and a very fundamental one in linear algebra! Let me walk you through what an **eigenvalue** is, step by step, in a way that’s clear and intuitive, just like I’d explain it to a thoughtful undergraduate student in my office hours.

---

### 🌟 What is an eigenvalue?

Imagine you have a square matrix $ A $ — think of it as a transformation machine that takes a vector and turns it into another vector. For example, if you have a vector $ \mathbf{v} $, then $ A\mathbf{v} $ is the result of applying the transformation $ A $ to $ \mathbf{v} $.

Now, suppose this transformation **does something special** to a particular vector: it **stretches or compresses it** — but **does not rotate or twist it**. In

[Thinking]
Okay, the user wants me to explain eigenvalues to undergraduate students as a patient math professor. Let me think about how to approach this.

First, I should consider where they're coming from. They're undergrads, so they probably know linear al

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
An eigenvalue is a scalar associated with a linear transformation represented by a matrix. It describes how much the transformation stretches or compresses certain vectors, known as eigenvectors. When a matrix acts on its eigenvector, the result is a scalar multiple of that vector. Mathematically, if \( A \) is a matrix and \( \mathbf{v} \) is a nonzero vector, then \( A\mathbf{v} = \lambda \mathbf{v} \), where \( \lambda \) is the eigenvalue. Eigenvalues are fundamental in understanding systems of linear equations, stability, and quantum mechanics.

[Thinking]
Hmm, the user wants me to explain what an eigenvalue is in exactly five sentences as a concise lecturer. They're clearly looking for a precise, educational response without any fluff.

I recall that eigenvalues are fundamental in linear algebra, so I need to cover the core idea simply: they're scalars that describe how a linear transformation stretches or shrinks vectors. The key is to mention eigenvectors too since 

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
Sure! Let’s keep it simple.

Imagine you have a special machine — like a stretchy sheet — and you put a vector (a direction and a length) on it. When you push the vector through the machine, it might stretch, shrink, or rotate it.

Now, an **eigenvalue** is just a number that tells you *how much* the machine stretched or shrunk the vector — if it didn’t change direction.

For example:

- If the vector comes out *longer* by a factor of 2, the eigenvalue is 2.
- If it comes out *shorter* by half, the eigenvalue is 0.5.
- If it stays the same size, the eigenvalue is 1.

And the vector that doesn’t change direction? That’s called an **eigenvector**.

So, in short:

👉 An

[Thinking]
Okay, the user wants me to explain eigenvalues in simple spoken English. They specifically asked me to act as an English presentation coach, so I need to keep it conversational and easy to understand. 

Hmm, eigenvalues... that's a pretty advanced math concept. The user might be a student who's just 

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
**Definition:**  
An *eigenvalue* of a square matrix \( A \) is a scalar \( \lambda \) such that there exists a non-zero vector \( \mathbf{v} \) satisfying the equation:  
\[
A\mathbf{v} = \lambda \mathbf{v}
\]  
The vector \( \mathbf{v} \) is called an *eigenvector* corresponding to the eigenvalue \( \lambda \).

---

**Intuition:**  
Think of a matrix as a transformation that stretches, rotates, or skews vectors in space. An eigenvalue tells us how much a vector is *scaled* (stretched or compressed) when it is transformed by the matrix — **without changing its direction**.  

In other words, if you apply the matrix transformation to an eigenvector, the result is just the

[Thinking]
Okay, the user wants me to explain eigenvalues as a rigorous mathematician would. They specifically asked for definition, intuition, and one example. 

Hmm, judging by the phrasing "you are a rigorous mathematician," they probably want precise mathematical language without hand-wavy explanatio

In [71]:
#temperature 변화 비교

temps = [0.2, 0.7, 1.0]
user_prompt = "Explain the difference between linear independence and spanning."
system_prompt = "You are a patient math professor."

for t in temps:
    print("\n" + "="*80)
    print(f"temperature = {t}")

    result_instruct = generate_response_with_model(
        model=model,
        tokenizer=tokenizer,
        user_prompt=user_prompt,
        system_prompt=system_prompt,
        temperature=t,
        top_p=0.9,
        top_k=50,
        max_new_tokens=180,
        seed=42
    )

    result_thinking = generate_response_with_model(
        model=model_thinking,
        tokenizer=tokenizer_thinking,
        user_prompt=user_prompt,
        system_prompt=system_prompt,
        temperature=t,
        top_p=0.9,
        top_k=50,
        max_new_tokens=180,
        seed=42
    )

    print("\n[Instruct]")
    print(result_instruct)

    print("\n[Thinking]")
    print(result_thinking)


temperature = 0.2


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
Ah, excellent question — and one that often trips up students in linear algebra. Let me explain the difference between **linear independence** and **spanning** with clarity, as if I were sitting across from a bright young student in my office, sipping tea and gently guiding them through the concepts.

---

### 📌 First: What is *Linear Independence*?

**Linear independence** is about *how vectors relate to each other* — specifically, whether one vector can be written as a combination of the others.

> A set of vectors is **linearly independent** if the only way to write the zero vector as a linear combination of them is by using all zero coefficients.

Let’s take a simple example:

Suppose we have two vectors in \(\mathbb{R}^2\):

\[
\mathbf{v}_1 = \begin{bmatrix} 1

[Thinking]
Okay, the user wants me to explain the difference between linear independence and spanning. They specifically mentioned I should act as a patient math professor, so I need to be clear, methodical, and

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
Ah, excellent question — and one that often trips up students in linear algebra. Let me explain the difference between **linear independence** and **spanning** with care and clarity, as if I were sitting across from a thoughtful student in my office hours.

---

### 📌 Linear Independence

**Definition:** A set of vectors is *linearly independent* if the only way to write the zero vector as a linear combination of them is by using all zero coefficients.

In other words, suppose we have vectors $ \mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_k $. They are linearly independent if:

$$
c_1 \mathbf{v}_1 + c_2 \mathbf{v}_2 + \cdots + c_k \mathbf{v}_k = \mathbf{0

[Thinking]
Okay, the user asked me to explain the difference between linear independence and spanning. They specifically mentioned I should be a "patient math professor," so I need to adopt that tone—warm, clear, and encouraging without being too technical. 

Hmm, judging by the question, they're probably a student in li

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
Ah, excellent question — and one that often trips up students in linear algebra. Let me explain the difference between **linear independence** and **spanning** with care and clarity, as if I were sitting across from a thoughtful student in my office hours.

---

### 📌 Linear Independence

**Definition:** A set of vectors is *linearly independent* if the only way to write the zero vector as a linear combination of them is by using all zero coefficients.

In simpler terms:  
> *Can you make the zero vector by combining the vectors (with scalars) in any non-trivial way?*  
If **no**, then they are linearly independent.

🔹 Example:  
Let’s say we have two vectors in \(\mathbb{R}^2\):  
\[
\mathbf{v}_1 = \begin{bmatrix} 1 \\ 

[Thinking]
Okay, the user asked me to explain the difference between linear independence and spanning. Hmm, they mentioned I'm supposed to be a patient math professor, so I should be clear and methodical. 

First, I recall that both concepts are fundamenta

In [72]:
# 한국어 비교

korean_prompts = [
    "선형대수에서 basis를 학부 2학년 학생에게 쉽게 설명해줘.",
    "eigenvalue를 직관적으로 설명해줘.",
    "linearly independent와 spanning의 차이를 쉽게 설명해줘."
]

system_prompt = "You are a patient math professor."

for p in korean_prompts:
    print("\n" + "="*80)
    print("PROMPT:", p)

    result_instruct = generate_response_with_model(
        model=model,
        tokenizer=tokenizer,
        user_prompt=p,
        system_prompt=system_prompt,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=220,
        seed=42
    )

    result_thinking = generate_response_with_model(
        model=model_thinking,
        tokenizer=tokenizer_thinking,
        user_prompt=p,
        system_prompt=system_prompt,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=220,
        seed=42
    )

    print("\n[Instruct]")
    print(result_instruct)

    print("\n[Thinking]")
    print(result_thinking)


PROMPT: 선형대수에서 basis를 학부 2학년 학생에게 쉽게 설명해줘.


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
물론이죠! 선형대수에서 **basis**(기저)를 학부 2학년 학생이 쉽게 이해할 수 있도록, 일상적인 예를 들어 설명해볼게요.

---

### 🌟 먼저, 기저(Basis)란?

**기저**(basis)는 어떤 공간(예: 2차원 평면, 3차원 공간)을 **완전히 표현**하기 위해 필요한 **가장 적은 수의 벡터**를 말해요.

이 말을 좀 더 쉽게 이해하려면, 생각해보세요:

> 🏠 "집을 만들기 위해 필요한 최소한의 장비는 무엇인가요?"

예를 들어, 집을 만들 때 **벽을 만들기 위해** 필요한 건 **벽돌**, **바닥을 만들기 위해** 필요한 건 **콘크리트**, **천장**은 **거울**이나 **가구**가 아니라 **철근**이 필요할 수도 있죠.  
하지만, **

[Thinking]
Okay, so I need to explain what a basis is in linear algebra to a second-year college student. Hmm, let's see. First, I remember that a basis is a set of vectors that are linearly independent and span the vector space. But how do I make that simple for a 2nd year student?

Maybe start with something they already know. Like, in 2D space, the standard basis is the x and y axes. So vectors (1,0) and (0,1). They can represent any point in the plane by combinations of these. Like (3,4) is 3*(1,0) + 4*(0,1). That makes sense.

Wait, but the student might not know what "span" means yet. So I sh

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
물론입니다, 선생님께서 말씀하신 "eigenvalue를 직관적으로 설명해줘"라는 질문에 대해, 제가 **학생처럼 느껴지도록 진지하면서도 쉽게, 직관적으로** 설명드리겠습니다.

---

### 🎓 "eigenvalue"는 무엇인가요? — 직관적으로 말해보기

가령, **고등학교 수학에서 벡터를 배웠을 때**,  
"벡터는 방향과 크기로 표현되는 것"이라고 생각하셨죠?

이제 그 벡터를 어떤 **변환**(예: 회전, 확대, 압축)에 걸렸을 때,  
그 벡터가 **자기 자신 방향을 유지**하면서 **크기가 바뀌는 경우**를 생각해보세요.

그런 경우, 그 **크기의 변화 비율**을 **eigenvalue**(특성값)이라고 부릅니다.

---

### 🔍 예를 들어 보겠습니다:

가령, 다음과 같은 변환을 생각

[Thinking]
Okay, so I need to explain eigenvalues intuitively to someone who's not too mathematically advanced. Let me think... Eigenvalues are a concept from linear algebra, right? They come up when you have a matrix and you're trying to find certain special numbers associated with it.

Hmm, I remember that eigenvalues have something to do with scaling or stretching vectors when you multiply them by the matrix. Like, if you have a matrix A and a vector v, then Av = λv, where λ is the eigenvalue. So the vector v gets scaled by λ when multiplied by A.

But how do I explain this int

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.



[Instruct]
물론이죠, 교수님처럼 정중하고 쉽게 설명해볼게요 😊

---

**"선형 독립"**(linear independence)과 **"스핑닝"(spanning)은 벡터 공간에서 아주 중요한 개념이에요. 그런데 이 둘은 서로 다른 의미를 가집니다. 아래에 쉽게, 그리고 예를 들어 설명해볼게요.**

---

### 🔹 1. 선형 독립(linear independence)

👉 **"어떤 벡터들이 서로를 독립적으로 만들 수 있는가?"** 를 묻는 거예요.

- **예시로 생각해보죠**: 2차원 평면에서 벡터를 생각해봅시다.  
  예를 들어, 벡터 **v₁ = (1, 0)**, **v₂ = (0, 1)** 이 있다고 해보죠.

  이 두 벡터는 **선형 독립**이에요. 왜냐하면 한 �

[Thinking]
Okay, so I need to explain the difference between "linearly independent" and "spanning" in linear algebra. Hmm, let me think. First, I remember that these are two concepts related to vectors and vector spaces. Let me try to recall.

Linear independence... I think it's about whether a set of vectors can be written as a linear combination of each other. Like, if you have vectors v1, v2, ..., vn, they're linearly independent if the only way to get the zero vector is by having all the coefficients be zero. So, if a1v1 + a2v2 + ... + anvn = 0, then a1 = a2 = ... = an = 0. That makes sense. S

In [73]:
# few-shot 반응 비교


few_shot_examples = [
    {
        "user": "Explain what a vector space is.",
        "assistant": "A vector space is a collection of vectors where you can add vectors and multiply them by numbers, and the results still stay in the same space."
    },
    {
        "user": "Explain what a basis is.",
        "assistant": "A basis is a minimal set of vectors that can generate every vector in the space by linear combination."
    }
]

user_prompt = "Explain what dimension means in linear algebra."
system_prompt = "You are a patient math professor."

print("===== Instruct / no few-shot =====")
print(generate_response_with_model(
    model=model,
    tokenizer=tokenizer,
    user_prompt=user_prompt,
    system_prompt=system_prompt,
    few_shot_examples=None,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=180,
    seed=42
))

print("\n===== Instruct / with few-shot =====")
print(generate_response_with_model(
    model=model,
    tokenizer=tokenizer,
    user_prompt=user_prompt,
    system_prompt=system_prompt,
    few_shot_examples=few_shot_examples,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=180,
    seed=42
))

print("\n===== Thinking / no few-shot =====")
print(generate_response_with_model(
    model=model_thinking,
    tokenizer=tokenizer_thinking,
    user_prompt=user_prompt,
    system_prompt=system_prompt,
    few_shot_examples=None,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=180,
    seed=42
))

print("\n===== Thinking / with few-shot =====")
print(generate_response_with_model(
    model=model_thinking,
    tokenizer=tokenizer_thinking,
    user_prompt=user_prompt,
    system_prompt=system_prompt,
    few_shot_examples=few_shot_examples,
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    max_new_tokens=180,
    seed=42
))


===== Instruct / no few-shot =====
Ah, a lovely and foundational question—thank you for asking. Let me take a moment to sit back in my favorite armchair (though I don’t have one, I *do* have a mental image of one with a well-worn copy of *Linear Algebra Done Right* nearby), and explain what **dimension** means in linear algebra—calmly, clearly, and with the care one would give to a bright student in a morning lecture.

---

### What is Dimension in Linear Algebra?

In linear algebra, **dimension** refers to the number of **linearly independent vectors** that span a vector space.

Let me break that down, step by step, like I’m talking to a curious student in a university classroom.

---

### Step 1: What is a Vector Space?

A vector space is a collection of vectors that can be added together and scaled (multiplied by scal

===== Instruct / with few-shot =====


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Ah, excellent question—thank you for coming back with such a thoughtful follow-up. Let me take a moment to explain *dimension* in linear algebra, as a patient professor would.

---

**What is dimension in linear algebra?**

In a vector space, the **dimension** is the number of vectors in any basis for that space.

Let me walk through this with care and clarity, because understanding dimension is foundational to grasping the structure of vector spaces.

Suppose we have a vector space—say, the space of all 2D vectors in the plane, like \(\mathbb{R}^2\). A basis for this space is any set of linearly independent vectors that span the entire space. For example:

- The standard basis is \(\{(1,0), (0,1)\}\).

These two vectors are linearly independent (you can’t write one

===== Thinking / no few-shot =====


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Okay, the user asked me to explain what dimension means in linear algebra. Since they specified I'm a patient math professor, I should be clear, methodical, and pedagogical. 

First, I need to recall the core definition: dimension is about the "size" of a vector space, specifically how many vectors you need to span it without redundancy. But I shouldn't jump straight into jargon—start with intuition. 

Hmm, the user might be a student who just encountered vector spaces and is confused. They probably saw terms like "dimension of R²" but don't grasp why it's 2. I should connect it to everyday examples they know, like 2D planes or 3D space. 

Wait, they might also be mixing up dimension with other concepts like rank or dimension of a matrix. Better clarify that dimension is for vector spaces, not matrices directly

===== Thinking / with few-shot =====
Okay, the user is asking about dimension in linear algebra after previously getting explanations for vector spaces and bases. They seem to 

In [74]:
# 결과 표 만들기


import pandas as pd

records = []

test_cases = [
    {
        "prompt": "Explain what a basis is in linear algebra.",
        "system_prompt": "You are a patient math professor."
    },
    {
        "prompt": "Explain what an eigenvalue is.",
        "system_prompt": "You are a patient math professor."
    },
    {
        "prompt": "Explain the difference between linear independence and spanning.",
        "system_prompt": "You are a patient math professor."
    }
]

for case in test_cases:
    out_instruct = generate_response_with_model(
        model=model,
        tokenizer=tokenizer,
        user_prompt=case["prompt"],
        system_prompt=case["system_prompt"],
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=180,
        seed=42
    )

    out_thinking = generate_response_with_model(
        model=model_thinking,
        tokenizer=tokenizer_thinking,
        user_prompt=case["prompt"],
        system_prompt=case["system_prompt"],
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        max_new_tokens=180,
        seed=42
    )

    records.append({
        "prompt": case["prompt"],
        "model_type": "Instruct",
        "response": out_instruct
    })
    records.append({
        "prompt": case["prompt"],
        "model_type": "Thinking",
        "response": out_thinking
    })

df_compare = pd.DataFrame(records)
df_compare


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


,prompt,model_type,response
0,Explain what a basis is in linear algebra.,Instruct,"Ah, my dear student—what a wonderful and found..."
1,Explain what a basis is in linear algebra.,Thinking,"Okay, the user asked me to explain what a basi..."
2,Explain what an eigenvalue is.,Instruct,"Ah, my dear student—what a delightful and foun..."
3,Explain what an eigenvalue is.,Thinking,"Okay, the user asked me to explain what an eig..."
4,Explain the difference between linear independ...,Instruct,"Ah, excellent question — and one that often tr..."
5,Explain the difference between linear independ...,Thinking,"Okay, the user asked me to explain the differe..."


In [75]:
df_compare["accuracy_score"] = None
df_compare["clarity_score"] = None
df_compare["presentation_score"] = None
df_compare["conciseness_score"] = None

df_compare

,prompt,model_type,response,accuracy_score,clarity_score,presentation_score,conciseness_score
0,Explain what a basis is in linear algebra.,Instruct,"Ah, my dear student—what a wonderful and found...",None,None,None,None
1,Explain what a basis is in linear algebra.,Thinking,"Okay, the user asked me to explain what a basi...",None,None,None,None
2,Explain what an eigenvalue is.,Instruct,"Ah, my dear student—what a delightful and foun...",None,None,None,None
3,Explain what an eigenvalue is.,Thinking,"Okay, the user asked me to explain what an eig...",None,None,None,None
4,Explain the difference between linear independ...,Instruct,"Ah, excellent question — and one that often tr...",None,None,None,None
5,Explain the difference between linear independ...,Thinking,"Okay, the user asked me to explain the differe...",None,None,None,None


In [76]:
df_compare.loc[0, "accuracy_score"] = 5
df_compare.loc[0, "clarity_score"] = 5
df_compare.loc[0, "presentation_score"] = 4
df_compare.loc[0, "conciseness_score"] = 5

df_compare

,prompt,model_type,response,accuracy_score,clarity_score,presentation_score,conciseness_score
0,Explain what a basis is in linear algebra.,Instruct,"Ah, my dear student—what a wonderful and found...",5,5,4,5
1,Explain what a basis is in linear algebra.,Thinking,"Okay, the user asked me to explain what a basi...",None,None,None,None
2,Explain what an eigenvalue is.,Instruct,"Ah, my dear student—what a delightful and foun...",None,None,None,None
3,Explain what an eigenvalue is.,Thinking,"Okay, the user asked me to explain what an eig...",None,None,None,None
4,Explain the difference between linear independ...,Instruct,"Ah, excellent question — and one that often tr...",None,None,None,None
5,Explain the difference between linear independ...,Thinking,"Okay, the user asked me to explain the differe...",None,None,None,None


In [77]:
df_compare["response_length"] = df_compare["response"].apply(len)
df_compare[["prompt", "model_type", "response_length"]]

,prompt,model_type,response_length
0,Explain what a basis is in linear algebra.,Instruct,676
1,Explain what a basis is in linear algebra.,Thinking,858
2,Explain what an eigenvalue is.,Instruct,722
3,Explain what an eigenvalue is.,Thinking,843
4,Explain the difference between linear independ...,Instruct,664
5,Explain the difference between linear independ...,Thinking,907


In [78]:
for prompt in df_compare["prompt"].unique():
    sub = df_compare[df_compare["prompt"] == prompt]
    print("\n" + "="*80)
    print("PROMPT:", prompt)
    for _, row in sub.iterrows():
        print(f"\n[{row['model_type']}] length={row['response_length']}")
        print(row["response"][:500])


PROMPT: Explain what a basis is in linear algebra.

[Instruct] length=676
Ah, my dear student—what a wonderful and foundational question! Let me take a moment to sit back, pour a cup of tea (perhaps chamomile, for clarity), and explain what a *basis* is in linear algebra—something I’ve taught countless times to eager minds in my office.

---

**What is a basis?**

In linear algebra, a **basis** of a vector space is a set of vectors that satisfies two essential properties:

1. **Linear independence**  
   No vector in the set can be written as a linear combination of t

[Thinking] length=858
Okay, the user asked me to explain what a basis is in linear algebra. They specifically mentioned I should act as a patient math professor. Hmm, this is a fundamental concept but can be tricky for beginners. 

First, I need to consider who might be asking this. Probably a student who's just starting linear algebra—maybe confused about vectors, spans, or why bases matter. They might have seen terms 

Llama → Mistral → Gemma → Qwen → Phi/gpt-oss